<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/06_CTGAN_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
# ==================================================================================================
# NOTEBOOK 06 — CTGAN BASELINE
# ==================================================================================================
#
# Purpose:
#   Establish a reproducible non-private CTGAN baseline for comparison with:
#       - Statistical baseline
#       - TVAE baseline
#       - DP-CTGAN baseline
#       - SPP-GAN
#
# Research role:
#   CTGAN provides a neural generative baseline without:
#       - differential privacy
#       - statistical guidance
#       - SPP-GAN components
#
# Data policy:
#   - TRAIN split only
#   - Validation split excluded
#   - Test split excluded
#   - Notebook 02 preprocessing is NOT refitted
#   - Native generative schema from Notebook 02 is authoritative
#
# Output policy:
#   Every trained model, checkpoint, synthetic dataset, metadata record,
#   training history, manifest, hash and validation result is persisted.
#
# RAM policy:
#   - Process one dataset at a time
#   - Do not load validation/test data
#   - Do not retain unnecessary model/data objects
#   - Persist artifacts immediately
#   - Explicitly release memory after each dataset
#
# ==================================================================================================

NOTEBOOK_ID = "06"
NOTEBOOK_NAME = "CTGAN Baseline"
NOTEBOOK_VERSION = "1.0"

print("=" * 100)
print("NOTEBOOK 06 — CTGAN BASELINE")
print("=" * 100)
print(f"Notebook ID      : {NOTEBOOK_ID}")
print(f"Notebook version : {NOTEBOOK_VERSION}")
print(f"Notebook name    : {NOTEBOOK_NAME}")
print("=" * 100)

NOTEBOOK 06 — CTGAN BASELINE
Notebook ID      : 06
Notebook version : 1.0
Notebook name    : CTGAN Baseline


In [24]:
!pip install -q sdv==1.38.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.4/215.4 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.7/213.7 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 10.0 MB/s eta 0:00:00


In [25]:
# ==================================================================================================
# SECTION 2 — LOAD CONFIGURATION
# ==================================================================================================

from pathlib import Path
import sdv
import os
import gc
import json
import time
import random
import hashlib
import traceback
import warnings
import sys
import subprocess
from datetime import datetime, timezone

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------------------------------
# 1. Mount Google Drive
# --------------------------------------------------------------------------------------------------

from google.colab import drive

drive.mount("/content/drive")

# --------------------------------------------------------------------------------------------------
# 2. Canonical project paths
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/SPP_GAN_Research")

NB02_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_02"
)

NB02_NATIVE_ROOT = NB02_ROOT / "native"

NB02_NATIVE_MANIFEST = (
    NB02_NATIVE_ROOT
    / "native_dataset_manifest.csv"
)

NB06_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_06"
)

NB06_MODEL_ROOT = NB06_ROOT / "models"
NB06_CHECKPOINT_ROOT = NB06_ROOT / "checkpoints"
NB06_SYNTHETIC_ROOT = NB06_ROOT / "synthetic"
NB06_METADATA_ROOT = NB06_ROOT / "metadata"
NB06_HISTORY_ROOT = NB06_ROOT / "history"
NB06_MANIFEST_ROOT = NB06_ROOT / "manifest"
NB06_VALIDATION_ROOT = NB06_ROOT / "validation"
NB06_CONFIG_ROOT = NB06_ROOT / "config"

for root in [
    NB06_ROOT,
    NB06_MODEL_ROOT,
    NB06_CHECKPOINT_ROOT,
    NB06_SYNTHETIC_ROOT,
    NB06_METADATA_ROOT,
    NB06_HISTORY_ROOT,
    NB06_MANIFEST_ROOT,
    NB06_VALIDATION_ROOT,
    NB06_CONFIG_ROOT,
]:
    root.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------------------------------------------
# 3. Dataset registry — aligned with Notebook 00
# --------------------------------------------------------------------------------------------------

DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

TARGET_COLUMNS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

IDENTIFIER_COLUMNS = {
    "adult_income": [],
    "bank_marketing": [],
    "diabetes_130us": [
        "encounter_id",
        "patient_nbr",
    ],
}

PROVENANCE_COLUMN = "__original_row_id__"

# --------------------------------------------------------------------------------------------------
# 4. Required dependency check
# --------------------------------------------------------------------------------------------------

print("=" * 100)
print("DEPENDENCY CHECK")
print("=" * 100)

try:
    import torch
except ImportError as exc:
    raise RuntimeError(
        "PyTorch is required for CTGAN but is not installed."
    ) from exc

try:
    import sdv
except ImportError as exc:
    raise RuntimeError(
        "SDV is required for Notebook 06 CTGAN Baseline but is not installed.\n\n"
        "Install the same validated SDV version used by Notebook 05 before "
        "re-running Section 2."
    ) from exc

print(f"✓ PyTorch available : {torch.__version__}")
print(f"✓ SDV available     : {sdv.__version__}")

# --------------------------------------------------------------------------------------------------
# 5. Canonical dependency compatibility
# --------------------------------------------------------------------------------------------------
#
# IMPORTANT:
# Notebook 05 established the validated SDV environment.
# Do not silently install or upgrade SDV inside the experiment.
#
# If the runtime does not contain SDV, install the validated version explicitly
# in a separate dependency cell and restart the runtime if required.
# --------------------------------------------------------------------------------------------------

VALIDATED_SDV_VERSION = "1.38.3"

if sdv.__version__ != VALIDATED_SDV_VERSION:
    raise RuntimeError(
        "SDV version mismatch.\n"
        f"Expected validated version : {VALIDATED_SDV_VERSION}\n"
        f"Detected version           : {sdv.__version__}\n\n"
        "Do not continue the CTGAN experiment with an unvalidated SDV version."
    )

print(
    f"✓ SDV version validated : {VALIDATED_SDV_VERSION}"
)

# --------------------------------------------------------------------------------------------------
# 6. Notebook 02 canonical native TRAIN files
# --------------------------------------------------------------------------------------------------

NATIVE_DATA_FILES = {
    dataset_id: NB02_NATIVE_ROOT / dataset_id / "train.csv"
    for dataset_id in DATASET_IDS
}

# Fallback discovery for robustness.
for dataset_id in DATASET_IDS:

    if not NATIVE_DATA_FILES[dataset_id].exists():

        candidates = sorted(
            NB02_NATIVE_ROOT.glob(
                f"**/{dataset_id}*train*.csv"
            )
        )

        if candidates:
            NATIVE_DATA_FILES[dataset_id] = candidates[0]

# --------------------------------------------------------------------------------------------------
# 7. Validate Notebook 02 native manifest
# --------------------------------------------------------------------------------------------------

if not NB02_NATIVE_MANIFEST.exists():
    raise FileNotFoundError(
        "Notebook 02 native manifest not found:\n"
        f"{NB02_NATIVE_MANIFEST}"
    )

NB02_NATIVE_MANIFEST_DF = pd.read_csv(
    NB02_NATIVE_MANIFEST
)

required_manifest_columns = {
    "dataset_id",
    "preprocessing_feature_columns",
    "generative_columns",
    "target_column",
    "provenance_column",
    "identifier_columns",
    "provenance_present",
    "target_present",
    "identifiers_excluded",
    "file_exists",
    "reload_validation",
    "status",
}

missing_manifest_columns = (
    required_manifest_columns
    - set(NB02_NATIVE_MANIFEST_DF.columns)
)

if missing_manifest_columns:
    raise RuntimeError(
        "Notebook 02 manifest is missing required columns: "
        f"{sorted(missing_manifest_columns)}"
    )

# --------------------------------------------------------------------------------------------------
# 8. Validate dataset registry against Notebook 02 manifest
# --------------------------------------------------------------------------------------------------

manifest_dataset_ids = set(
    NB02_NATIVE_MANIFEST_DF["dataset_id"]
    .astype(str)
    .str.strip()
)

missing_datasets = (
    set(DATASET_IDS)
    - manifest_dataset_ids
)

if missing_datasets:
    raise RuntimeError(
        "Notebook 02 manifest does not contain the required datasets: "
        f"{sorted(missing_datasets)}"
    )

# --------------------------------------------------------------------------------------------------
# 9. Resolve authoritative Notebook 02 manifest records
# --------------------------------------------------------------------------------------------------

AUTHORITATIVE_MANIFEST_ROWS = {}

for dataset_id in DATASET_IDS:

    dataset_rows = NB02_NATIVE_MANIFEST_DF[
        NB02_NATIVE_MANIFEST_DF["dataset_id"].astype(str).str.strip()
        == dataset_id
    ].copy()

    if dataset_rows.empty:
        raise RuntimeError(
            f"No Notebook 02 manifest records found for {dataset_id}."
        )

    dataset_rows["_status_norm"] = (
        dataset_rows["status"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    dataset_rows["_target_norm"] = (
        dataset_rows["target_column"]
        .astype(str)
        .str.strip()
    )

    dataset_rows["_provenance_norm"] = (
        dataset_rows["provenance_column"]
        .astype(str)
        .str.strip()
    )

    candidates = dataset_rows[
        (dataset_rows["_status_norm"] == "PASS")
        & (
            dataset_rows["_target_norm"]
            == TARGET_COLUMNS[dataset_id]
        )
        & (
            dataset_rows["_provenance_norm"]
            == PROVENANCE_COLUMN
        )
    ].copy()

    if len(candidates) == 0:
        raise RuntimeError(
            f"{dataset_id}: no authoritative PASS native manifest "
            "record could be resolved."
        )

    comparison_columns = [
        "preprocessing_feature_columns",
        "generative_columns",
        "target_column",
        "provenance_column",
        "identifier_columns",
        "provenance_present",
        "target_present",
        "identifiers_excluded",
        "file_exists",
        "reload_validation",
        "status",
    ]

    signatures = (
        candidates[comparison_columns]
        .astype(str)
        .apply(
            lambda row: "||".join(row.tolist()),
            axis=1,
        )
        .unique()
    )

    if len(signatures) != 1:
        raise RuntimeError(
            f"{dataset_id}: conflicting authoritative Notebook 02 "
            "manifest records were found."
        )

    AUTHORITATIVE_MANIFEST_ROWS[dataset_id] = candidates.iloc[0]

# --------------------------------------------------------------------------------------------------
# 10. Derive authoritative generative schemas
# --------------------------------------------------------------------------------------------------

GENERATIVE_COLUMNS = {}
FEATURE_COLUMNS = {}

for dataset_id in DATASET_IDS:

    row = AUTHORITATIVE_MANIFEST_ROWS[dataset_id]

    path = NATIVE_DATA_FILES[dataset_id]

    if not path.exists():
        raise FileNotFoundError(
            f"Native TRAIN file not found for {dataset_id}:\n{path}"
        )

    header = pd.read_csv(
        path,
        nrows=0,
    )

    native_columns = list(header.columns)

    target = TARGET_COLUMNS[dataset_id]
    identifiers = IDENTIFIER_COLUMNS[dataset_id]

    # ----------------------------------------------------------------------------------------------
    # Provenance validation
    # ----------------------------------------------------------------------------------------------

    if PROVENANCE_COLUMN not in native_columns:
        raise RuntimeError(
            f"{dataset_id}: required provenance column "
            f"'{PROVENANCE_COLUMN}' is missing."
        )

    # ----------------------------------------------------------------------------------------------
    # Target validation
    # ----------------------------------------------------------------------------------------------

    if target not in native_columns:
        raise RuntimeError(
            f"{dataset_id}: target column '{target}' is missing "
            "from native TRAIN data."
        )

    # ----------------------------------------------------------------------------------------------
    # Identifier exclusion validation
    # ----------------------------------------------------------------------------------------------

    forbidden_identifiers = [
        column
        for column in identifiers
        if column in native_columns
    ]

    if forbidden_identifiers:
        raise RuntimeError(
            f"{dataset_id}: identifier columns remain in native "
            f"TRAIN data: {forbidden_identifiers}"
        )

    # ----------------------------------------------------------------------------------------------
    # Generative schema
    # ----------------------------------------------------------------------------------------------

    generative_columns = [
        column
        for column in native_columns
        if column != PROVENANCE_COLUMN
        and column not in identifiers
    ]

    # ----------------------------------------------------------------------------------------------
    # Target must be generated jointly with the table.
    # ----------------------------------------------------------------------------------------------

    if target not in generative_columns:
        raise RuntimeError(
            f"{dataset_id}: target column '{target}' is not present "
            "in the generative schema."
        )

    # ----------------------------------------------------------------------------------------------
    # Feature schema for downstream policy bookkeeping.
    # ----------------------------------------------------------------------------------------------

    feature_columns = [
        column
        for column in generative_columns
        if column != target
    ]

    GENERATIVE_COLUMNS[dataset_id] = generative_columns
    FEATURE_COLUMNS[dataset_id] = feature_columns

    # ----------------------------------------------------------------------------------------------
    # Manifest count validation
    # ----------------------------------------------------------------------------------------------

    manifest_generatives = int(
        row["generative_columns"]
    )

    manifest_features = int(
        row["preprocessing_feature_columns"]
    )

    if len(generative_columns) != manifest_generatives:
        raise RuntimeError(
            f"{dataset_id}: generative-column count mismatch. "
            f"Manifest={manifest_generatives}, "
            f"derived={len(generative_columns)}."
        )

    if len(feature_columns) != manifest_features:
        raise RuntimeError(
            f"{dataset_id}: feature-column count mismatch. "
            f"Manifest={manifest_features}, "
            f"derived={len(feature_columns)}."
        )

    # ----------------------------------------------------------------------------------------------
    # Manifest policy validation
    # ----------------------------------------------------------------------------------------------

    if str(row["target_column"]).strip() != target:
        raise RuntimeError(
            f"{dataset_id}: manifest target mismatch."
        )

    if str(row["provenance_column"]).strip() != PROVENANCE_COLUMN:
        raise RuntimeError(
            f"{dataset_id}: manifest provenance mismatch."
        )

    if str(row["provenance_present"]).strip().upper() not in {
        "TRUE",
        "1",
        "YES",
    }:
        raise RuntimeError(
            f"{dataset_id}: manifest does not confirm provenance presence."
        )

    if str(row["target_present"]).strip().upper() not in {
        "TRUE",
        "1",
        "YES",
    }:
        raise RuntimeError(
            f"{dataset_id}: manifest does not confirm target presence."
        )

    # ----------------------------------------------------------------------------------------------
    # Summary
    # ----------------------------------------------------------------------------------------------

    print(
        f"✓ {dataset_id:<18} | "
        f"Native columns: {len(native_columns):>3} | "
        f"Generative: {len(generative_columns):>3} | "
        f"Features: {len(feature_columns):>3} | "
        f"Target: {target}"
    )

# --------------------------------------------------------------------------------------------------
# 11. Runtime information
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("RUNTIME INFORMATION")
print("=" * 100)

print(
    f"Python version     : {sys.version.split()[0]}"
)

print(
    f"SDV version        : {sdv.__version__}"
)

print(
    f"PyTorch version    : {torch.__version__}"
)

print(
    f"CUDA available     : {torch.cuda.is_available()}"
)

if torch.cuda.is_available():
    print(
        f"GPU                : "
        f"{torch.cuda.get_device_name(0)}"
    )

print(
    f"Project root       : {PROJECT_ROOT}"
)

print(
    f"Notebook 06 root   : {NB06_ROOT}"
)

# --------------------------------------------------------------------------------------------------
# 12. Final configuration summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("NOTEBOOK 06 CONFIGURATION SUMMARY")
print("=" * 100)

print(
    f"Datasets           : {len(DATASET_IDS)}"
)

print(
    f"Manifest records   : {len(NB02_NATIVE_MANIFEST_DF)}"
)

print(
    f"Authoritative sets : {len(AUTHORITATIVE_MANIFEST_ROWS)}"
)

print(
    "Train-only policy  : ENABLED"
)

print(
    "Validation used    : NO"
)

print(
    "Test used          : NO"
)

print(
    "Differential DP    : NO"
)

print(
    "Statistical guide  : NO"
)

print(
    "SPP-GAN components : NO"
)

print()
print("✓ Section 2 configuration loaded successfully.")
print("✓ Notebook 02 authoritative schemas resolved.")
print("✓ Generative-column counts validated.")
print("✓ Feature-column counts validated.")
print("✓ Target/provenance policy validated.")
print("✓ Required dependencies validated.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DEPENDENCY CHECK
✓ PyTorch available : 2.11.0+cu128
✓ SDV available     : 1.38.3
✓ SDV version validated : 1.38.3
✓ adult_income       | Native columns:  16 | Generative:  15 | Features:  14 | Target: income
✓ bank_marketing     | Native columns:  18 | Generative:  17 | Features:  16 | Target: y
✓ diabetes_130us     | Native columns:  49 | Generative:  48 | Features:  47 | Target: readmitted

RUNTIME INFORMATION
Python version     : 3.13.15
SDV version        : 1.38.3
PyTorch version    : 2.11.0+cu128
CUDA available     : True
GPU                : Tesla T4
Project root       : /content/drive/MyDrive/SPP_GAN_Research
Notebook 06 root   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_06

NOTEBOOK 06 CONFIGURATION SUMMARY
Datasets           : 3
Manifest records   : 9
Authoritative sets : 3
Train-only policy  : ENABLED
Validation used    : NO

In [27]:
# ==================================================================================================
# SECTION 3 — LOAD TRAINING DATA
# ==================================================================================================

print("=" * 100)
print("SECTION 3 — LOAD TRAINING DATA")
print("=" * 100)

TRAINING_DATA = {}
TRAINING_SHAPES = {}
TRAINING_SCHEMA_HASHES = {}

# --------------------------------------------------------------------------------------------------
# Helper — deterministic schema hash
# --------------------------------------------------------------------------------------------------

def compute_schema_hash(columns):
    """
    Compute a deterministic SHA-256 hash for an ordered column schema.
    """

    schema_payload = json.dumps(
        list(columns),
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")

    return hashlib.sha256(schema_payload).hexdigest()


# --------------------------------------------------------------------------------------------------
# Load and validate each canonical Notebook 02 TRAIN dataset
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"Loading dataset : {dataset_id}")

    path = NATIVE_DATA_FILES[dataset_id]

    if not path.exists():
        raise FileNotFoundError(
            f"{dataset_id}: canonical native TRAIN file not found:\n{path}"
        )

    target = TARGET_COLUMNS[dataset_id]
    identifiers = IDENTIFIER_COLUMNS[dataset_id]
    generative_columns = GENERATIVE_COLUMNS[dataset_id]

    # ----------------------------------------------------------------------------------------------
    # 1. Load canonical native TRAIN data
    # ----------------------------------------------------------------------------------------------

    df = pd.read_csv(path)

    if df.empty:
        raise RuntimeError(
            f"{dataset_id}: training dataset is empty."
        )

    print(f"✓ Native rows    : {len(df):,}")
    print(f"✓ Native columns : {len(df.columns)}")

    # ----------------------------------------------------------------------------------------------
    # 2. Reject duplicate column names
    # ----------------------------------------------------------------------------------------------

    duplicated_columns = (
        df.columns[df.columns.duplicated()]
        .tolist()
    )

    if duplicated_columns:
        raise RuntimeError(
            f"{dataset_id}: duplicate column names detected: "
            f"{duplicated_columns}"
        )

    # ----------------------------------------------------------------------------------------------
    # 3. Validate target
    # ----------------------------------------------------------------------------------------------

    if target not in df.columns:
        raise RuntimeError(
            f"{dataset_id}: target column '{target}' is missing."
        )

    # ----------------------------------------------------------------------------------------------
    # 4. Validate provenance
    # ----------------------------------------------------------------------------------------------

    if PROVENANCE_COLUMN not in df.columns:
        raise RuntimeError(
            f"{dataset_id}: provenance column "
            f"'{PROVENANCE_COLUMN}' is missing."
        )

    # ----------------------------------------------------------------------------------------------
    # 5. Validate provenance uniqueness
    #
    # Provenance is retained only for integrity validation and is NOT supplied
    # to CTGAN.
    # ----------------------------------------------------------------------------------------------

    provenance_missing = df[PROVENANCE_COLUMN].isna().sum()

    if provenance_missing > 0:
        raise RuntimeError(
            f"{dataset_id}: provenance column contains "
            f"{provenance_missing:,} missing values."
        )

    provenance_duplicates = (
        df[PROVENANCE_COLUMN]
        .duplicated()
        .sum()
    )

    if provenance_duplicates > 0:
        raise RuntimeError(
            f"{dataset_id}: provenance column contains "
            f"{provenance_duplicates:,} duplicate values."
        )

    # ----------------------------------------------------------------------------------------------
    # 6. Validate explicit identifier exclusion
    # ----------------------------------------------------------------------------------------------

    forbidden_native = [
        column
        for column in identifiers
        if column in df.columns
    ]

    if forbidden_native:
        raise RuntimeError(
            f"{dataset_id}: identifier columns remain in native TRAIN data: "
            f"{forbidden_native}"
        )

    # ----------------------------------------------------------------------------------------------
    # 7. Validate authoritative generative schema
    # ----------------------------------------------------------------------------------------------

    actual_generative_columns = [
        column
        for column in df.columns
        if column != PROVENANCE_COLUMN
        and column not in identifiers
    ]

    # Exact column-set validation.
    expected_set = set(generative_columns)
    actual_set = set(actual_generative_columns)

    missing_columns = sorted(
        expected_set - actual_set
    )

    unexpected_columns = sorted(
        actual_set - expected_set
    )

    if missing_columns:
        raise RuntimeError(
            f"{dataset_id}: missing authoritative generative columns: "
            f"{missing_columns}"
        )

    if unexpected_columns:
        raise RuntimeError(
            f"{dataset_id}: unexpected columns found in generative data: "
            f"{unexpected_columns}"
        )

    # Exact order validation.
    if actual_generative_columns != generative_columns:
        raise RuntimeError(
            f"{dataset_id}: generative column order does not match "
            "the authoritative Notebook 02 schema.\n"
            f"Expected: {generative_columns}\n"
            f"Actual  : {actual_generative_columns}"
        )

    # ----------------------------------------------------------------------------------------------
    # 8. Extract CTGAN generative training data
    # ----------------------------------------------------------------------------------------------

    train_df = df[generative_columns].copy()

    # ----------------------------------------------------------------------------------------------
    # 9. Defensive identifier validation after extraction
    # ----------------------------------------------------------------------------------------------

    forbidden = [
        column
        for column in identifiers
        if column in train_df.columns
    ]

    if forbidden:
        raise RuntimeError(
            f"{dataset_id}: identifier leakage detected in CTGAN input: "
            f"{forbidden}"
        )

    # ----------------------------------------------------------------------------------------------
    # 10. Defensive provenance validation
    # ----------------------------------------------------------------------------------------------

    if PROVENANCE_COLUMN in train_df.columns:
        raise RuntimeError(
            f"{dataset_id}: provenance column "
            f"'{PROVENANCE_COLUMN}' leaked into CTGAN input."
        )

    # ----------------------------------------------------------------------------------------------
    # 11. Target validation inside CTGAN input
    # ----------------------------------------------------------------------------------------------

    if target not in train_df.columns:
        raise RuntimeError(
            f"{dataset_id}: target '{target}' is absent from "
            "the CTGAN generative input."
        )

    # ----------------------------------------------------------------------------------------------
    # 12. Validate row count
    # ----------------------------------------------------------------------------------------------

    if len(train_df) != len(df):
        raise RuntimeError(
            f"{dataset_id}: row count changed during generative-column "
            "extraction."
        )

    if len(train_df) == 0:
        raise RuntimeError(
            f"{dataset_id}: CTGAN training dataframe is empty."
        )

    # ----------------------------------------------------------------------------------------------
    # 13. Validate authoritative feature count
    # ----------------------------------------------------------------------------------------------

    expected_feature_count = len(
        FEATURE_COLUMNS[dataset_id]
    )

    actual_feature_count = (
        len(train_df.columns) - 1
    )

    if actual_feature_count != expected_feature_count:
        raise RuntimeError(
            f"{dataset_id}: feature-count mismatch. "
            f"Expected={expected_feature_count}, "
            f"Actual={actual_feature_count}."
        )

    # ----------------------------------------------------------------------------------------------
    # 14. Compute deterministic schema hash
    # ----------------------------------------------------------------------------------------------

    schema_hash = compute_schema_hash(
        train_df.columns.tolist()
    )

    TRAINING_SCHEMA_HASHES[dataset_id] = schema_hash

    # ----------------------------------------------------------------------------------------------
    # 15. Store only the generative TRAIN dataframe
    # ----------------------------------------------------------------------------------------------

    TRAINING_DATA[dataset_id] = train_df
    TRAINING_SHAPES[dataset_id] = train_df.shape

    # Release the native dataframe immediately.
    del df
    gc.collect()

    # ----------------------------------------------------------------------------------------------
    # 16. Dataset summary
    # ----------------------------------------------------------------------------------------------

    print(
        f"✓ Rows             : {train_df.shape[0]:,}"
    )

    print(
        f"✓ Generative cols   : {train_df.shape[1]}"
    )

    print(
        f"✓ Feature cols      : {actual_feature_count}"
    )

    print(
        f"✓ Target            : {target}"
    )

    print(
        f"✓ Provenance input  : EXCLUDED"
    )

    print(
        f"✓ Identifier input  : EXCLUDED"
    )

    print(
        f"✓ TRAIN split only  : YES"
    )

    print(
        f"✓ Schema hash       : {schema_hash}"
    )

# --------------------------------------------------------------------------------------------------
# Final cross-dataset validation
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("TRAINING DATA VALIDATION SUMMARY")
print("=" * 100)

if set(TRAINING_DATA.keys()) != set(DATASET_IDS):
    raise RuntimeError(
        "Training-data registry does not contain exactly the expected datasets."
    )

for dataset_id in DATASET_IDS:

    train_df = TRAINING_DATA[dataset_id]

    if train_df.shape != TRAINING_SHAPES[dataset_id]:
        raise RuntimeError(
            f"{dataset_id}: stored training shape does not match registry."
        )

    if list(train_df.columns) != GENERATIVE_COLUMNS[dataset_id]:
        raise RuntimeError(
            f"{dataset_id}: stored training schema does not match "
            "authoritative Notebook 02 schema."
        )

    if TARGET_COLUMNS[dataset_id] not in train_df.columns:
        raise RuntimeError(
            f"{dataset_id}: target missing from stored CTGAN training data."
        )

    if PROVENANCE_COLUMN in train_df.columns:
        raise RuntimeError(
            f"{dataset_id}: provenance column present in CTGAN training data."
        )

    forbidden = [
        column
        for column in IDENTIFIER_COLUMNS[dataset_id]
        if column in train_df.columns
    ]

    if forbidden:
        raise RuntimeError(
            f"{dataset_id}: identifier leakage in stored CTGAN data: "
            f"{forbidden}"
        )

    print(
        f"✓ {dataset_id:<18} | "
        f"Shape={train_df.shape} | "
        f"Schema=PASS"
    )

print("\n✓ All CTGAN training datasets loaded and validated successfully.")

SECTION 3 — LOAD TRAINING DATA

----------------------------------------------------------------------------------------------------
Loading dataset : adult_income
✓ Native rows    : 34,189
✓ Native columns : 16
✓ Rows             : 34,189
✓ Generative cols   : 15
✓ Feature cols      : 14
✓ Target            : income
✓ Provenance input  : EXCLUDED
✓ Identifier input  : EXCLUDED
✓ TRAIN split only  : YES
✓ Schema hash       : e331e1d0b87d393f28e1739b391d3b48dd10c2c375f2531b398e337fe74ab308

----------------------------------------------------------------------------------------------------
Loading dataset : bank_marketing
✓ Native rows    : 31,647
✓ Native columns : 18
✓ Rows             : 31,647
✓ Generative cols   : 17
✓ Feature cols      : 16
✓ Target            : y
✓ Provenance input  : EXCLUDED
✓ Identifier input  : EXCLUDED
✓ TRAIN split only  : YES
✓ Schema hash       : 2ca9848ead4ffff660c82678f736c384f01b5764f1df3b1fe3a30d8e3fd461aa

---------------------------------------------

In [28]:
# ==================================================================================================
# SECTION 4 — VALIDATE SCHEMA
# ==================================================================================================

print("=" * 100)
print("SECTION 4 — VALIDATE SCHEMA")
print("=" * 100)

SCHEMA_VALIDATION_RECORDS = []

for dataset_id in DATASET_IDS:

    df = TRAINING_DATA[dataset_id]

    target = TARGET_COLUMNS[dataset_id]
    identifiers = IDENTIFIER_COLUMNS[dataset_id]
    expected_columns = GENERATIVE_COLUMNS[dataset_id]

    actual_columns = list(df.columns)

    exact_schema = actual_columns == expected_columns

    duplicate_columns = int(df.columns.duplicated().sum())

    identifier_leakage = [
        col for col in identifiers
        if col in actual_columns
    ]

    provenance_present = "__original_row_id__" in actual_columns

    target_present = target in actual_columns

    record = {
        "dataset_id": dataset_id,
        "rows": int(len(df)),
        "columns": int(df.shape[1]),
        "expected_columns": int(len(expected_columns)),
        "exact_schema": bool(exact_schema),
        "duplicate_columns": duplicate_columns,
        "provenance_present": bool(provenance_present),
        "target_present": bool(target_present),
        "identifier_leakage": bool(identifier_leakage),
        "target": target,
        "status": "PASS",
    }

    if not exact_schema:
        record["status"] = "FAIL"

    if duplicate_columns != 0:
        record["status"] = "FAIL"

    if provenance_present:
        record["status"] = "FAIL"

    if identifier_leakage:
        record["status"] = "FAIL"

    if not target_present:
        record["status"] = "FAIL"

    SCHEMA_VALIDATION_RECORDS.append(record)

    print(
        f"✓ {dataset_id:<20} | "
        f"rows={len(df):,} | "
        f"columns={df.shape[1]} | "
        f"schema={record['status']}"
    )

SCHEMA_VALIDATION_DF = pd.DataFrame(SCHEMA_VALIDATION_RECORDS)

if not (SCHEMA_VALIDATION_DF["status"] == "PASS").all():
    raise RuntimeError("Notebook 06 schema validation failed.")

print("\n✓ SECTION 4 — SCHEMA VALIDATION : PASS")

SECTION 4 — VALIDATE SCHEMA
✓ adult_income         | rows=34,189 | columns=15 | schema=PASS
✓ bank_marketing       | rows=31,647 | columns=17 | schema=PASS
✓ diabetes_130us       | rows=71,236 | columns=48 | schema=PASS

✓ SECTION 4 — SCHEMA VALIDATION : PASS


In [31]:
# ==================================================================================================
# SECTION 5 — CONFIGURE CTGAN
# ==================================================================================================

print("=" * 100)
print("SECTION 5 — CONFIGURE CTGAN")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. CTGAN configuration
# --------------------------------------------------------------------------------------------------

CTGAN_CONFIG = {
    "model": "CTGANSynthesizer",

    # ----------------------------------------------------------------------------------------------
    # Architecture
    # ----------------------------------------------------------------------------------------------

    "embedding_dim": 128,
    "generator_dim": (256, 256),
    "discriminator_dim": (256, 256),

    # ----------------------------------------------------------------------------------------------
    # Training
    # ----------------------------------------------------------------------------------------------

    "epochs": 300,
    "batch_size": 500,
    "pac": 10,

    # ----------------------------------------------------------------------------------------------
    # Optimisation
    # ----------------------------------------------------------------------------------------------

    "generator_lr": 2e-4,
    "generator_decay": 1e-6,
    "discriminator_lr": 2e-4,
    "discriminator_decay": 1e-6,

    # ----------------------------------------------------------------------------------------------
    # Data constraints
    # ----------------------------------------------------------------------------------------------

    "enforce_min_max_values": True,
    "enforce_rounding": True,

    # ----------------------------------------------------------------------------------------------
    # Runtime
    # ----------------------------------------------------------------------------------------------

    "verbose": False,
    "requested_gpu": True,

    # ----------------------------------------------------------------------------------------------
    # Sampling
    # ----------------------------------------------------------------------------------------------

    "sample_size_policy": "training_rows",

    # ----------------------------------------------------------------------------------------------
    # Data policy
    # ----------------------------------------------------------------------------------------------

    "fit_data_policy": "native_train_only",
    "validation_used": False,
    "test_used": False,

    # ----------------------------------------------------------------------------------------------
    # Research separation
    # ----------------------------------------------------------------------------------------------

    "differential_privacy": False,
    "statistical_guidance": False,
    "sppgan_components": False,

    # ----------------------------------------------------------------------------------------------
    # Reproducibility
    # ----------------------------------------------------------------------------------------------

    "seed_policy": "Notebook_00_global_seed_policy",
    "random_state_parameter_checked_at_runtime": True,

    # ----------------------------------------------------------------------------------------------
    # SDV metadata
    # ----------------------------------------------------------------------------------------------

    "metadata_source": "Notebook_06_detected_from_native_train_only",
}

# --------------------------------------------------------------------------------------------------
# 2. Validate CTGAN configuration
# --------------------------------------------------------------------------------------------------

if CTGAN_CONFIG["epochs"] <= 0:
    raise ValueError(
        "CTGAN epochs must be greater than zero."
    )

if CTGAN_CONFIG["batch_size"] <= 0:
    raise ValueError(
        "CTGAN batch_size must be greater than zero."
    )

if CTGAN_CONFIG["pac"] <= 0:
    raise ValueError(
        "CTGAN pac must be greater than zero."
    )

# Critical CTGAN requirement:
# batch_size must be exactly divisible by pac.
if CTGAN_CONFIG["batch_size"] % CTGAN_CONFIG["pac"] != 0:
    raise ValueError(
        "Invalid CTGAN configuration: "
        f"batch_size={CTGAN_CONFIG['batch_size']} is not divisible by "
        f"pac={CTGAN_CONFIG['pac']}."
    )

if CTGAN_CONFIG["embedding_dim"] <= 0:
    raise ValueError(
        "CTGAN embedding_dim must be greater than zero."
    )

if not CTGAN_CONFIG["generator_dim"]:
    raise ValueError(
        "CTGAN generator_dim cannot be empty."
    )

if not CTGAN_CONFIG["discriminator_dim"]:
    raise ValueError(
        "CTGAN discriminator_dim cannot be empty."
    )

if not CTGAN_CONFIG["fit_data_policy"] == "native_train_only":
    raise RuntimeError(
        "CTGAN must be trained on native TRAIN data only."
    )

if CTGAN_CONFIG["validation_used"]:
    raise RuntimeError(
        "Validation data must not be used during CTGAN baseline training."
    )

if CTGAN_CONFIG["test_used"]:
    raise RuntimeError(
        "Test data must not be used during CTGAN baseline training."
    )

if CTGAN_CONFIG["differential_privacy"]:
    raise RuntimeError(
        "Differential privacy must remain disabled for the CTGAN baseline."
    )

if CTGAN_CONFIG["statistical_guidance"]:
    raise RuntimeError(
        "Statistical guidance must remain disabled for the CTGAN baseline."
    )

if CTGAN_CONFIG["sppgan_components"]:
    raise RuntimeError(
        "SPP-GAN components must remain disabled for the CTGAN baseline."
    )

# --------------------------------------------------------------------------------------------------
# 3. Runtime API inspection
# --------------------------------------------------------------------------------------------------

from inspect import signature

try:
    from sdv.single_table import CTGANSynthesizer
except ImportError as exc:
    raise RuntimeError(
        "CTGANSynthesizer could not be imported from the validated SDV installation."
    ) from exc

CTGAN_SIGNATURE = signature(
    CTGANSynthesizer.__init__
)

CTGAN_PARAMETER_NAMES = set(
    CTGAN_SIGNATURE.parameters.keys()
)

RANDOM_STATE_SUPPORTED = (
    "random_state" in CTGAN_PARAMETER_NAMES
)

CTGAN_CONFIG["random_state_supported"] = bool(
    RANDOM_STATE_SUPPORTED
)

print(
    f"✓ CTGAN random_state parameter supported : "
    f"{RANDOM_STATE_SUPPORTED}"
)

# --------------------------------------------------------------------------------------------------
# 4. Configuration summary
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("CTGAN CONFIGURATION")
print("-" * 100)

print(
    f"✓ Model                 : "
    f"{CTGAN_CONFIG['model']}"
)

print(
    f"✓ Epochs                : "
    f"{CTGAN_CONFIG['epochs']}"
)

print(
    f"✓ Embedding dimension   : "
    f"{CTGAN_CONFIG['embedding_dim']}"
)

print(
    f"✓ Generator dimensions  : "
    f"{CTGAN_CONFIG['generator_dim']}"
)

print(
    f"✓ Discriminator dims.   : "
    f"{CTGAN_CONFIG['discriminator_dim']}"
)

print(
    f"✓ Batch size            : "
    f"{CTGAN_CONFIG['batch_size']}"
)

print(
    f"✓ PAC                   : "
    f"{CTGAN_CONFIG['pac']}"
)

print(
    f"✓ Batch ÷ PAC           : "
    f"{CTGAN_CONFIG['batch_size'] // CTGAN_CONFIG['pac']}"
)

print(
    f"✓ Batch % PAC           : "
    f"{CTGAN_CONFIG['batch_size'] % CTGAN_CONFIG['pac']}"
)

print(
    f"✓ Requested GPU         : "
    f"{CTGAN_CONFIG['requested_gpu']}"
)

print(
    f"✓ Differential privacy  : "
    f"{CTGAN_CONFIG['differential_privacy']}"
)

print(
    f"✓ Statistical guidance  : "
    f"{CTGAN_CONFIG['statistical_guidance']}"
)

print(
    f"✓ SPP-GAN components    : "
    f"{CTGAN_CONFIG['sppgan_components']}"
)

print(
    f"✓ Fit data policy       : "
    f"{CTGAN_CONFIG['fit_data_policy']}"
)

# --------------------------------------------------------------------------------------------------
# 5. Persist configuration
# --------------------------------------------------------------------------------------------------

CTGAN_CONFIG_PATH = (
    NB06_CONFIG_ROOT
    / "ctgan_config.json"
)

with open(
    CTGAN_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        CTGAN_CONFIG,
        f,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )

print(
    f"✓ Configuration saved   : "
    f"{CTGAN_CONFIG_PATH}"
)

print()
print("✓ SECTION 5 — CTGAN CONFIGURATION : PASS")

SECTION 5 — CONFIGURE CTGAN
✓ CTGAN random_state parameter supported : False

----------------------------------------------------------------------------------------------------
CTGAN CONFIGURATION
----------------------------------------------------------------------------------------------------
✓ Model                 : CTGANSynthesizer
✓ Epochs                : 300
✓ Embedding dimension   : 128
✓ Generator dimensions  : (256, 256)
✓ Discriminator dims.   : (256, 256)
✓ Batch size            : 500
✓ PAC                   : 10
✓ Batch ÷ PAC           : 50
✓ Batch % PAC           : 0
✓ Requested GPU         : True
✓ Differential privacy  : False
✓ Statistical guidance  : False
✓ SPP-GAN components    : False
✓ Fit data policy       : native_train_only
✓ Configuration saved   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_06/config/ctgan_config.json

✓ SECTION 5 — CTGAN CONFIGURATION : PASS


In [33]:
# ==================================================================================================
# SECTION 6 — SET SEEDS
# ==================================================================================================

print("=" * 100)
print("SECTION 6 — SET SEEDS")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Reproducibility policy
# --------------------------------------------------------------------------------------------------

MASTER_SEED = 2025
REPETITION_SEED_OFFSET = 1000

REPETITION_SEEDS = {
    "rep_01": 3026,
    "rep_02": 3027,
    "rep_03": 3028,
    "rep_04": 3029,
    "rep_05": 3030,
}

# CTGAN baseline uses the primary baseline repetition.
PRIMARY_REPETITION = "rep_01"

# Dataset-specific seeds for the primary repetition.
DATASET_SEEDS = {
    "adult_income": 3126,
    "bank_marketing": 3226,
    "diabetes_130us": 3326,
}

# --------------------------------------------------------------------------------------------------
# 2. Validate seed registry
# --------------------------------------------------------------------------------------------------

if MASTER_SEED != 2025:
    raise RuntimeError(
        f"Unexpected master seed: {MASTER_SEED}"
    )

if PRIMARY_REPETITION not in REPETITION_SEEDS:
    raise RuntimeError(
        f"Primary repetition '{PRIMARY_REPETITION}' "
        "is not present in REPETITION_SEEDS."
    )

expected_primary_seed = REPETITION_SEEDS[
    PRIMARY_REPETITION
]

# Validate the dataset seed construction used by the project policy.
expected_dataset_seeds = {
    "adult_income": expected_primary_seed + 100,
    "bank_marketing": expected_primary_seed + 200,
    "diabetes_130us": expected_primary_seed + 300,
}

if DATASET_SEEDS != expected_dataset_seeds:
    raise RuntimeError(
        "Dataset seed registry does not match the primary-repetition "
        "seed policy.\n"
        f"Expected: {expected_dataset_seeds}\n"
        f"Actual  : {DATASET_SEEDS}"
    )

if set(DATASET_SEEDS.keys()) != set(DATASET_IDS):
    raise RuntimeError(
        "Dataset seed registry does not match DATASET_IDS."
    )

if len(set(DATASET_SEEDS.values())) != len(DATASET_SEEDS):
    raise RuntimeError(
        "Dataset seeds must be unique."
    )

# --------------------------------------------------------------------------------------------------
# 3. Seed function
# --------------------------------------------------------------------------------------------------

def seed_everything(seed: int):

    if not isinstance(seed, (int, np.integer)):
        raise TypeError(
            f"Seed must be an integer; received {type(seed)}."
        )

    seed = int(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    try:
        import torch

        torch.manual_seed(seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)

        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            # Some operations may not support deterministic execution.
            pass

    except ImportError as exc:
        raise RuntimeError(
            "PyTorch is required for the CTGAN reproducibility policy."
        ) from exc

    return seed

# --------------------------------------------------------------------------------------------------
# 4. Establish initial master seed
#
# The dataset-specific seed is intentionally NOT applied for every dataset
# in this section. It will be applied immediately before each dataset's
# CTGAN initialization/training in the training section.
# --------------------------------------------------------------------------------------------------

seed_everything(MASTER_SEED)

# --------------------------------------------------------------------------------------------------
# 5. Persist seed policy
# --------------------------------------------------------------------------------------------------

SEED_POLICY = {
    "master_seed": MASTER_SEED,
    "repetition_seed_offset": REPETITION_SEED_OFFSET,
    "repetition_seeds": REPETITION_SEEDS,
    "primary_repetition": PRIMARY_REPETITION,
    "primary_repetition_seed": expected_primary_seed,
    "dataset_seeds": DATASET_SEEDS,
    "seed_application_policy": (
        "Apply dataset-specific seed immediately before CTGAN "
        "initialization/training for each dataset."
    ),
    "random_state_parameter_supported": CTGAN_CONFIG[
        "random_state_supported"
    ],
}

SEED_POLICY_PATH = (
    NB06_CONFIG_ROOT
    / "ctgan_seed_policy.json"
)

with open(
    SEED_POLICY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        SEED_POLICY,
        f,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )

# --------------------------------------------------------------------------------------------------
# 6. Report
# --------------------------------------------------------------------------------------------------

print(
    f"✓ Master seed             : {MASTER_SEED}"
)

print(
    f"✓ Primary repetition      : {PRIMARY_REPETITION}"
)

print(
    f"✓ Primary repetition seed : {expected_primary_seed}"
)

for dataset_id in DATASET_IDS:

    print(
        f"✓ {dataset_id:<20} : "
        f"{DATASET_SEEDS[dataset_id]}"
    )

print(
    f"✓ Initial RNG state       : "
    f"master seed {MASTER_SEED}"
)

print(
    f"✓ Dataset seed timing     : "
    f"before model initialization/training"
)

print(
    f"✓ Seed policy saved       : "
    f"{SEED_POLICY_PATH}"
)

print()
print("✓ SECTION 6 — SEED POLICY : PASS")

SECTION 6 — SET SEEDS
✓ Master seed             : 2025
✓ Primary repetition      : rep_01
✓ Primary repetition seed : 3026
✓ adult_income         : 3126
✓ bank_marketing       : 3226
✓ diabetes_130us       : 3326
✓ Initial RNG state       : master seed 2025
✓ Dataset seed timing     : before model initialization/training
✓ Seed policy saved       : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_06/config/ctgan_seed_policy.json

✓ SECTION 6 — SEED POLICY : PASS


In [35]:
# ==================================================================================================
# SECTION 7 — INITIALIZE CTGAN
# ==================================================================================================

print("=" * 100)
print("SECTION 7 — INITIALIZE CTGAN")
print("=" * 100)

from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

CTGAN_MODELS = {}
CTGAN_METADATA = {}
CTGAN_METADATA_VALIDATION = {}

# --------------------------------------------------------------------------------------------------
# Initialize one CTGAN model per dataset
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"Initializing : {dataset_id}")

    # ----------------------------------------------------------------------------------------------
    # 1. Dataset-specific reproducibility seed
    # ----------------------------------------------------------------------------------------------

    seed = int(
        DATASET_SEEDS[dataset_id]
    )

    seed_everything(seed)

    # ----------------------------------------------------------------------------------------------
    # 2. Retrieve validated TRAIN data
    # ----------------------------------------------------------------------------------------------

    train_df = TRAINING_DATA[dataset_id]

    if train_df.empty:
        raise RuntimeError(
            f"{dataset_id}: CTGAN training dataframe is empty."
        )

    target = TARGET_COLUMNS[dataset_id]
    identifiers = IDENTIFIER_COLUMNS[dataset_id]
    expected_columns = GENERATIVE_COLUMNS[dataset_id]

    # ----------------------------------------------------------------------------------------------
    # 3. Validate training schema before metadata detection
    # ----------------------------------------------------------------------------------------------

    if list(train_df.columns) != expected_columns:
        raise RuntimeError(
            f"{dataset_id}: training schema differs from the "
            "authoritative Notebook 02 generative schema."
        )

    if target not in train_df.columns:
        raise RuntimeError(
            f"{dataset_id}: target '{target}' is missing from CTGAN input."
        )

    if PROVENANCE_COLUMN in train_df.columns:
        raise RuntimeError(
            f"{dataset_id}: provenance column '{PROVENANCE_COLUMN}' "
            "must not be supplied to CTGAN."
        )

    identifier_leakage = [
        column
        for column in identifiers
        if column in train_df.columns
    ]

    if identifier_leakage:
        raise RuntimeError(
            f"{dataset_id}: identifier leakage detected: "
            f"{identifier_leakage}"
        )

    # ----------------------------------------------------------------------------------------------
    # 4. Detect SDV metadata from TRAIN only
    # ----------------------------------------------------------------------------------------------

    metadata = SingleTableMetadata()

    metadata.detect_from_dataframe(
        train_df
    )

    # ----------------------------------------------------------------------------------------------
    # 5. Validate detected metadata columns
    # ----------------------------------------------------------------------------------------------

    try:
        metadata_columns = list(
            metadata.to_dict()["columns"].keys()
        )
    except Exception as exc:
        raise RuntimeError(
            f"{dataset_id}: unable to extract detected SDV metadata columns."
        ) from exc

    if metadata_columns != expected_columns:
        raise RuntimeError(
            f"{dataset_id}: detected metadata schema does not match "
            "the authoritative CTGAN training schema.\n"
            f"Expected: {expected_columns}\n"
            f"Detected: {metadata_columns}"
        )

    # ----------------------------------------------------------------------------------------------
    # 6. Explicit target semantic
    #
    # The target is part of the jointly generated synthetic table.
    # It is marked categorical for consistent semantic treatment.
    # This does NOT mean CTGAN is trained as a supervised predictor.
    # ----------------------------------------------------------------------------------------------

    try:
        metadata.update_column(
            column_name=target,
            sdtype="categorical",
        )
    except Exception as exc:
        raise RuntimeError(
            f"{dataset_id}: failed to configure target '{target}' "
            "as categorical in SDV metadata."
        ) from exc

    # ----------------------------------------------------------------------------------------------
    # 7. Validate target metadata
    # ----------------------------------------------------------------------------------------------

    metadata_dict = metadata.to_dict()

    target_metadata = (
        metadata_dict
        .get("columns", {})
        .get(target)
    )

    if target_metadata is None:
        raise RuntimeError(
            f"{dataset_id}: target '{target}' is absent from SDV metadata."
        )

    if target_metadata.get("sdtype") != "categorical":
        raise RuntimeError(
            f"{dataset_id}: target '{target}' was not configured as "
            "categorical.\n"
            f"Detected sdtype: {target_metadata.get('sdtype')}"
        )

    # ----------------------------------------------------------------------------------------------
    # 8. Validate prohibited columns are absent from metadata
    # ----------------------------------------------------------------------------------------------

    prohibited_metadata_columns = [
        column
        for column in (
            [PROVENANCE_COLUMN]
            + identifiers
        )
        if column in metadata_columns
    ]

    if prohibited_metadata_columns:
        raise RuntimeError(
            f"{dataset_id}: prohibited columns present in CTGAN metadata: "
            f"{prohibited_metadata_columns}"
        )

    # ----------------------------------------------------------------------------------------------
    # 9. Initialize CTGAN
    # ----------------------------------------------------------------------------------------------

    model = CTGANSynthesizer(
        metadata=metadata,
        embedding_dim=CTGAN_CONFIG["embedding_dim"],
        generator_dim=CTGAN_CONFIG["generator_dim"],
        discriminator_dim=CTGAN_CONFIG["discriminator_dim"],
        generator_lr=CTGAN_CONFIG["generator_lr"],
        generator_decay=CTGAN_CONFIG["generator_decay"],
        discriminator_lr=CTGAN_CONFIG["discriminator_lr"],
        discriminator_decay=CTGAN_CONFIG["discriminator_decay"],
        batch_size=CTGAN_CONFIG["batch_size"],
        discriminator_steps=1,
        log_frequency=True,
        verbose=CTGAN_CONFIG["verbose"],
        epochs=CTGAN_CONFIG["epochs"],
        pac=CTGAN_CONFIG["pac"],
        enforce_min_max_values=CTGAN_CONFIG[
            "enforce_min_max_values"
        ],
        enforce_rounding=CTGAN_CONFIG[
            "enforce_rounding"
        ],
    )

    # ----------------------------------------------------------------------------------------------
    # 10. Store model and metadata
    # ----------------------------------------------------------------------------------------------

    CTGAN_MODELS[dataset_id] = model
    CTGAN_METADATA[dataset_id] = metadata

    CTGAN_METADATA_VALIDATION[dataset_id] = {
        "metadata_columns": metadata_columns,
        "schema_exact": True,
        "target": target,
        "target_sdtype": target_metadata["sdtype"],
        "provenance_present": False,
        "identifier_leakage": False,
        "seed": seed,
        "status": "PASS",
    }

    # ----------------------------------------------------------------------------------------------
    # 11. Report
    # ----------------------------------------------------------------------------------------------

    print(
        "✓ Metadata detected from TRAIN only"
    )

    print(
        "✓ Metadata schema validated"
    )

    print(
        f"✓ Target semantic       : {target} → categorical"
    )

    print(
        "✓ Provenance metadata   : EXCLUDED"
    )

    print(
        "✓ Identifier metadata   : EXCLUDED"
    )

    print(
        "✓ CTGAN initialized"
    )

    print(
        f"✓ Seed                  : {seed}"
    )

# --------------------------------------------------------------------------------------------------
# Final validation
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("CTGAN INITIALIZATION VALIDATION")
print("=" * 100)

if set(CTGAN_MODELS.keys()) != set(DATASET_IDS):
    raise RuntimeError(
        "CTGAN model registry does not contain exactly the expected datasets."
    )

if set(CTGAN_METADATA.keys()) != set(DATASET_IDS):
    raise RuntimeError(
        "CTGAN metadata registry does not contain exactly the expected datasets."
    )

if set(CTGAN_METADATA_VALIDATION.keys()) != set(DATASET_IDS):
    raise RuntimeError(
        "CTGAN metadata validation registry is incomplete."
    )

for dataset_id in DATASET_IDS:

    validation = CTGAN_METADATA_VALIDATION[dataset_id]

    if validation["status"] != "PASS":
        raise RuntimeError(
            f"{dataset_id}: CTGAN initialization validation failed."
        )

    if not validation["schema_exact"]:
        raise RuntimeError(
            f"{dataset_id}: metadata schema validation failed."
        )

    if validation["provenance_present"]:
        raise RuntimeError(
            f"{dataset_id}: provenance leakage detected."
        )

    if validation["identifier_leakage"]:
        raise RuntimeError(
            f"{dataset_id}: identifier leakage detected."
        )

    print(
        f"✓ {dataset_id:<20} | "
        f"Metadata=PASS | "
        f"Target={validation['target']} | "
        f"Seed={validation['seed']}"
    )

print()
print("✓ SECTION 7 — CTGAN INITIALIZATION : PASS")

SECTION 7 — INITIALIZE CTGAN

----------------------------------------------------------------------------------------------------
Initializing : adult_income
✓ Metadata detected from TRAIN only
✓ Metadata schema validated
✓ Target semantic       : income → categorical
✓ Provenance metadata   : EXCLUDED
✓ Identifier metadata   : EXCLUDED
✓ CTGAN initialized
✓ Seed                  : 3126

----------------------------------------------------------------------------------------------------
Initializing : bank_marketing
✓ Metadata detected from TRAIN only
✓ Metadata schema validated
✓ Target semantic       : y → categorical
✓ Provenance metadata   : EXCLUDED
✓ Identifier metadata   : EXCLUDED
✓ CTGAN initialized
✓ Seed                  : 3226

----------------------------------------------------------------------------------------------------
Initializing : diabetes_130us
✓ Metadata detected from TRAIN only
✓ Metadata schema validated
✓ Target semantic       : readmitted → categorical
✓ P

In [37]:
# ==================================================================================================
# SECTION 8 — TRAIN CTGAN
# ==================================================================================================

print()
print("=" * 100)
print("SECTION 8 — TRAIN CTGAN")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Training registries
# --------------------------------------------------------------------------------------------------

CTGAN_TRAINING_RECORDS = []
CTGAN_TRAINED_MODELS = {}
CTGAN_TRAINING_TIMES = {}
CTGAN_FAILED_DATASETS = []

TRAINING_SESSION_START = datetime.now(
    timezone.utc
)

# --------------------------------------------------------------------------------------------------
# 2. Validate required registries from previous sections
# --------------------------------------------------------------------------------------------------

if set(DATASET_IDS) != set(TRAINING_DATA.keys()):
    raise RuntimeError(
        "TRAINING_DATA registry does not match DATASET_IDS."
    )

if set(DATASET_IDS) != set(CTGAN_MODELS.keys()):
    raise RuntimeError(
        "CTGAN_MODELS registry does not match DATASET_IDS."
    )

if set(DATASET_IDS) != set(DATASET_SEEDS.keys()):
    raise RuntimeError(
        "DATASET_SEEDS registry does not match DATASET_IDS."
    )

# --------------------------------------------------------------------------------------------------
# 3. Validated SDV version
# --------------------------------------------------------------------------------------------------

SDV_VERSION = str(
    sdv.__version__
)

if SDV_VERSION != VALIDATED_SDV_VERSION:
    raise RuntimeError(
        "SDV version changed after Section 5.\n"
        f"Expected: {VALIDATED_SDV_VERSION}\n"
        f"Detected: {SDV_VERSION}"
    )

# --------------------------------------------------------------------------------------------------
# 4. Train datasets sequentially
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print()
    print("-" * 100)
    print(
        f"Training Dataset : {dataset_id}"
    )
    print("-" * 100)

    train_df = TRAINING_DATA[
        dataset_id
    ]

    synthesizer = CTGAN_MODELS[
        dataset_id
    ]

    seed = int(
        DATASET_SEEDS[
            dataset_id
        ]
    )

    # ----------------------------------------------------------------------------------------------
    # Apply dataset-specific seed immediately before training.
    # ----------------------------------------------------------------------------------------------

    seed_everything(
        seed
    )

    # ----------------------------------------------------------------------------------------------
    # Basic training-data validation
    # ----------------------------------------------------------------------------------------------

    if train_df.empty:
        raise RuntimeError(
            f"{dataset_id}: CTGAN training dataframe is empty."
        )

    if list(train_df.columns) != GENERATIVE_COLUMNS[dataset_id]:
        raise RuntimeError(
            f"{dataset_id}: CTGAN training schema does not match "
            "the authoritative Notebook 02 schema."
        )

    if TARGET_COLUMNS[dataset_id] not in train_df.columns:
        raise RuntimeError(
            f"{dataset_id}: target column is missing from CTGAN training data."
        )

    if PROVENANCE_COLUMN in train_df.columns:
        raise RuntimeError(
            f"{dataset_id}: provenance column leaked into CTGAN training data."
        )

    forbidden_identifiers = [
        column
        for column in IDENTIFIER_COLUMNS[dataset_id]
        if column in train_df.columns
    ]

    if forbidden_identifiers:
        raise RuntimeError(
            f"{dataset_id}: identifier leakage detected: "
            f"{forbidden_identifiers}"
        )

    # ----------------------------------------------------------------------------------------------
    # Dataset information
    # ----------------------------------------------------------------------------------------------

    print(
        f"Rows    : {len(train_df):,}"
    )

    print(
        f"Columns : {len(train_df.columns)}"
    )

    print(
        f"Epochs  : {CTGAN_CONFIG['epochs']}"
    )

    print(
        f"Batch   : {CTGAN_CONFIG['batch_size']}"
    )

    print(
        f"PAC     : {CTGAN_CONFIG['pac']}"
    )

    print(
        f"Seed    : {seed}"
    )

    # ----------------------------------------------------------------------------------------------
    # Constructor parameter verification
    # ----------------------------------------------------------------------------------------------

    parameters = synthesizer.get_parameters()

    parameter_checks = {
        "epochs": (
            parameters.get("epochs")
            == CTGAN_CONFIG["epochs"]
        ),

        "batch_size": (
            parameters.get("batch_size")
            == CTGAN_CONFIG["batch_size"]
        ),

        "pac": (
            parameters.get("pac")
            == CTGAN_CONFIG["pac"]
        ),

        "embedding_dim": (
            parameters.get("embedding_dim")
            == CTGAN_CONFIG["embedding_dim"]
        ),

        "generator_dim": (
            tuple(
                parameters.get(
                    "generator_dim"
                )
            )
            == tuple(
                CTGAN_CONFIG[
                    "generator_dim"
                ]
            )
        ),

        "discriminator_dim": (
            tuple(
                parameters.get(
                    "discriminator_dim"
                )
            )
            == tuple(
                CTGAN_CONFIG[
                    "discriminator_dim"
                ]
            )
        ),
    }

    failed_parameter_checks = [
        name
        for name, passed
        in parameter_checks.items()
        if not passed
    ]

    if failed_parameter_checks:
        raise RuntimeError(
            f"{dataset_id}: CTGAN parameter mismatch:\n"
            + "\n".join(
                f"  - {name}"
                for name in failed_parameter_checks
            )
        )

    print(
        "✓ CTGAN constructor parameters validated"
    )

    # ----------------------------------------------------------------------------------------------
    # Training
    # ----------------------------------------------------------------------------------------------

    train_start = time.perf_counter()

    training_status = "FAILED"
    training_error = None
    loss_records = 0
    history_path = None

    try:

        print(
            "Starting CTGAN training..."
        )

        synthesizer.fit(
            train_df
        )

        training_status = "PASS"

        print(
            "✓ CTGAN training completed."
        )

    except Exception as exc:

        training_error = (
            f"{type(exc).__name__}: {exc}"
        )

        CTGAN_FAILED_DATASETS.append(
            dataset_id
        )

        print(
            "✗ CTGAN training failed."
        )

        print(
            f"Error : {training_error}"
        )

        traceback.print_exc()

    train_end = time.perf_counter()

    training_runtime_seconds = (
        train_end
        - train_start
    )

    CTGAN_TRAINING_TIMES[
        dataset_id
    ] = training_runtime_seconds

    # ----------------------------------------------------------------------------------------------
    # Training history
    # ----------------------------------------------------------------------------------------------

    if training_status == "PASS":

        try:

            loss_df = (
                synthesizer.get_loss_values()
            )

            if loss_df is None:

                loss_df = pd.DataFrame()

            elif not isinstance(
                loss_df,
                pd.DataFrame
            ):

                loss_df = pd.DataFrame(
                    loss_df
                )

            loss_df = loss_df.copy()

            if not loss_df.empty:

                history_path = (
                    NB06_HISTORY_ROOT
                    / dataset_id
                    / "ctgan_loss_history.csv"
                )

                history_path.parent.mkdir(
                    parents=True,
                    exist_ok=True,
                )

                loss_df.to_csv(
                    history_path,
                    index=False,
                )

                loss_records = int(
                    len(loss_df)
                )

            print(
                f"✓ Loss history records : {loss_records:,}"
            )

        except Exception as exc:

            print(
                "⚠ Loss history unavailable: "
                f"{type(exc).__name__}: {exc}"
            )

    # ----------------------------------------------------------------------------------------------
    # Register successfully trained model
    # ----------------------------------------------------------------------------------------------

    if training_status == "PASS":

        CTGAN_TRAINED_MODELS[
            dataset_id
        ] = synthesizer

    # ----------------------------------------------------------------------------------------------
    # Training record
    # ----------------------------------------------------------------------------------------------

    CTGAN_TRAINING_RECORDS.append(
        {
            "notebook_id":
                NOTEBOOK_ID,

            "notebook_version":
                NOTEBOOK_VERSION,

            "dataset_id":
                dataset_id,

            "model":
                "CTGANSynthesizer",

            "model_type":
                "Conventional CTGAN",

            "sdv_version":
                SDV_VERSION,

            "seed":
                seed,

            "training_rows":
                int(len(train_df)),

            "training_columns":
                int(len(train_df.columns)),

            "epochs":
                int(
                    CTGAN_CONFIG[
                        "epochs"
                    ]
                ),

            "batch_size":
                int(
                    CTGAN_CONFIG[
                        "batch_size"
                    ]
                ),

            "pac":
                int(
                    CTGAN_CONFIG[
                        "pac"
                    ]
                ),

            "embedding_dim":
                int(
                    CTGAN_CONFIG[
                        "embedding_dim"
                    ]
                ),

            "generator_dim":
                list(
                    CTGAN_CONFIG[
                        "generator_dim"
                    ]
                ),

            "discriminator_dim":
                list(
                    CTGAN_CONFIG[
                        "discriminator_dim"
                    ]
                ),

            "generator_lr":
                float(
                    CTGAN_CONFIG[
                        "generator_lr"
                    ]
                ),

            "generator_decay":
                float(
                    CTGAN_CONFIG[
                        "generator_decay"
                    ]
                ),

            "discriminator_lr":
                float(
                    CTGAN_CONFIG[
                        "discriminator_lr"
                    ]
                ),

            "discriminator_decay":
                float(
                    CTGAN_CONFIG[
                        "discriminator_decay"
                    ]
                ),

            "training_runtime_seconds":
                float(
                    training_runtime_seconds
                ),

            "loss_records":
                int(
                    loss_records
                ),

            "fit_split":
                "train",

            "validation_used":
                False,

            "test_used":
                False,

            "differential_privacy":
                False,

            "statistical_guidance":
                False,

            "sppgan_components":
                False,

            "status":
                training_status,

            "error":
                training_error,

            "history_path":
                str(history_path)
                if history_path is not None
                else None,

            "created_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),
        }
    )

    # ----------------------------------------------------------------------------------------------
    # Report
    # ----------------------------------------------------------------------------------------------

    print(
        f"Runtime      : "
        f"{training_runtime_seconds:.2f} seconds"
    )

    print(
        f"Loss records : "
        f"{loss_records:,}"
    )

    print(
        f"Status       : "
        f"{training_status}"
    )

# --------------------------------------------------------------------------------------------------
# 5. Final training validation
# --------------------------------------------------------------------------------------------------

if CTGAN_FAILED_DATASETS:

    raise RuntimeError(
        "CTGAN training failed for:\n"
        + "\n".join(
            f"  - {dataset_id}"
            for dataset_id
            in CTGAN_FAILED_DATASETS
        )
    )

if set(
    CTGAN_TRAINED_MODELS.keys()
) != set(
    DATASET_IDS
):

    missing_models = (
        set(DATASET_IDS)
        - set(CTGAN_TRAINED_MODELS.keys())
    )

    raise RuntimeError(
        "Not all CTGAN models were successfully trained.\n"
        f"Missing: {sorted(missing_models)}"
    )

# --------------------------------------------------------------------------------------------------
# 6. Create training summary dataframe
# --------------------------------------------------------------------------------------------------

CTGAN_TRAINING_DF = pd.DataFrame(
    CTGAN_TRAINING_RECORDS
)

if len(CTGAN_TRAINING_DF) != len(DATASET_IDS):
    raise RuntimeError(
        "CTGAN training summary does not contain exactly "
        "one record per dataset."
    )

if not (
    CTGAN_TRAINING_DF["status"] == "PASS"
).all():

    raise RuntimeError(
        "One or more CTGAN training records are not marked PASS."
    )

# --------------------------------------------------------------------------------------------------
# 7. Final summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("CTGAN TRAINING SUMMARY")
print("=" * 100)

display(
    CTGAN_TRAINING_DF[
        [
            "dataset_id",
            "training_rows",
            "training_columns",
            "epochs",
            "batch_size",
            "pac",
            "seed",
            "training_runtime_seconds",
            "loss_records",
            "status",
        ]
    ]
)

print()
print("✓ All three CTGAN models trained successfully.")
print("✓ Dataset-specific seeds applied before training.")
print("✓ Sequential dataset training completed.")
print("✓ TRAIN-only fitting confirmed.")
print("✓ CTGAN parameter configuration validated.")


SECTION 8 — TRAIN CTGAN

----------------------------------------------------------------------------------------------------
Training Dataset : adult_income
----------------------------------------------------------------------------------------------------
Rows    : 34,189
Columns : 15
Epochs  : 300
Batch   : 500
PAC     : 10
Seed    : 3126
✓ CTGAN constructor parameters validated
Starting CTGAN training...
✓ CTGAN training completed.
✓ Loss history records : 300
Runtime      : 748.51 seconds
Loss records : 300
Status       : PASS

----------------------------------------------------------------------------------------------------
Training Dataset : bank_marketing
----------------------------------------------------------------------------------------------------
Rows    : 31,647
Columns : 17
Epochs  : 300
Batch   : 500
PAC     : 10
Seed    : 3226
✓ CTGAN constructor parameters validated
Starting CTGAN training...
✓ CTGAN training completed.
✓ Loss history records : 300
Runtime     

,dataset_id,training_rows,training_columns,epochs,batch_size,pac,seed,training_runtime_seconds,loss_records,status
0,adult_income,34189,15,300,500,10,3126,748.505885,300,PASS
1,bank_marketing,31647,17,300,500,10,3226,739.199827,300,PASS
2,diabetes_130us,71236,48,300,500,10,3326,4448.345951,300,PASS



✓ All three CTGAN models trained successfully.
✓ Dataset-specific seeds applied before training.
✓ Sequential dataset training completed.
✓ TRAIN-only fitting confirmed.
✓ CTGAN parameter configuration validated.


In [38]:
# ==================================================================================================
# SECTION 9 — RECORD TRAINING HISTORY
# ==================================================================================================

print("=" * 100)
print("SECTION 9 — RECORD TRAINING HISTORY")
print("=" * 100)

if len(CTGAN_TRAINING_DF) != len(DATASET_IDS):
    raise RuntimeError(
        "Training summary does not contain exactly one record per dataset."
    )

if not (CTGAN_TRAINING_DF["status"] == "PASS").all():
    raise RuntimeError("One or more CTGAN training runs failed.")

history_validation_records = []

for dataset_id in DATASET_IDS:

    history_path = (
        NB06_HISTORY_ROOT
        / dataset_id
        / "ctgan_loss_history.csv"
    )

    if not history_path.exists():
        raise FileNotFoundError(
            f"Loss history missing: {history_path}"
        )

    history = pd.read_csv(history_path)

    history_validation_records.append({
        "dataset_id": dataset_id,
        "history_exists": True,
        "loss_records": int(len(history)),
        "file_size_bytes": int(history_path.stat().st_size),
        "status": "PASS",
    })

    print(
        f"✓ {dataset_id:<20} | "
        f"loss records={len(history):,} | PASS"
    )

CTGAN_HISTORY_VALIDATION_DF = pd.DataFrame(
    history_validation_records
)

print("\n✓ SECTION 9 — TRAINING HISTORY : PASS")

SECTION 9 — RECORD TRAINING HISTORY
✓ adult_income         | loss records=300 | PASS
✓ bank_marketing       | loss records=300 | PASS
✓ diabetes_130us       | loss records=300 | PASS

✓ SECTION 9 — TRAINING HISTORY : PASS


In [39]:
# ==================================================================================================
# SECTION 10 — SAVE CHECKPOINTS
# ==================================================================================================

print("=" * 100)
print("SECTION 10 — SAVE CHECKPOINTS")
print("=" * 100)

CTGAN_CHECKPOINT_RECORDS = []

def calculate_sha256(path: Path, chunk_size: int = 1024 * 1024):

    sha256 = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            sha256.update(chunk)

    return sha256.hexdigest()


for dataset_id in DATASET_IDS:

    model = CTGAN_MODELS[dataset_id]
    seed = DATASET_SEEDS[dataset_id]

    checkpoint_dir = (
        NB06_CHECKPOINT_ROOT / dataset_id
    )
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_path = (
        checkpoint_dir / "ctgan_checkpoint.pkl"
    )

    print(f"\nSaving checkpoint : {dataset_id}")

    # SDV native persistence.
    model.save(filepath=str(checkpoint_path))

    if not checkpoint_path.exists():
        raise RuntimeError(
            f"Checkpoint was not created: {checkpoint_path}"
        )

    size_bytes = checkpoint_path.stat().st_size

    if size_bytes <= 0:
        raise RuntimeError(
            f"Checkpoint is empty: {checkpoint_path}"
        )

    sha256 = calculate_sha256(checkpoint_path)

    record = {
        "dataset_id": dataset_id,
        "seed": seed,
        "checkpoint_path": str(checkpoint_path),
        "checkpoint_size_bytes": int(size_bytes),
        "sha256": sha256,
        "artifact_status": "NEWLY_SAVED",
        "status": "PASS",
    }

    CTGAN_CHECKPOINT_RECORDS.append(record)

    print(f"✓ Size   : {size_bytes:,} bytes")
    print(f"✓ SHA256 : {sha256}")
    print("✓ Status : PASS")

CTGAN_CHECKPOINT_DF = pd.DataFrame(
    CTGAN_CHECKPOINT_RECORDS
)

CTGAN_CHECKPOINT_REGISTRY = (
    NB06_CHECKPOINT_ROOT / "checkpoint_registry.csv"
)

CTGAN_CHECKPOINT_DF.to_csv(
    CTGAN_CHECKPOINT_REGISTRY,
    index=False
)

print(f"\n✓ Registry saved : {CTGAN_CHECKPOINT_REGISTRY}")
print("\n✓ SECTION 10 — CHECKPOINTS : PASS")

SECTION 10 — SAVE CHECKPOINTS

Saving checkpoint : adult_income
✓ Size   : 4,063,796 bytes
✓ SHA256 : 5e9d4c87d12916833e7a49ca2771e21ea7b0d68afb3ea3c352d40d0de0b050d6
✓ Status : PASS

Saving checkpoint : bank_marketing
✓ Size   : 3,894,016 bytes
✓ SHA256 : b1932f4f396ce33f694c6cf403b982f47ae78915f4ee10fa94256a0dfc1616fe
✓ Status : PASS

Saving checkpoint : diabetes_130us
✓ Size   : 59,859,079 bytes
✓ SHA256 : 447d40f70e85b92e50e77c125c7ea7a2b3b11ce2ab4bce3c9ab8514c769eeb49
✓ Status : PASS

✓ Registry saved : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_06/checkpoints/checkpoint_registry.csv

✓ SECTION 10 — CHECKPOINTS : PASS


In [43]:
# ==================================================================================================
# SECTION 11 — GENERATE SYNTHETIC DATA
# ==================================================================================================

print("=" * 100)
print("SECTION 11 — GENERATE SYNTHETIC DATA")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. RESET GENERATION CONTAINERS
# --------------------------------------------------------------------------------------------------

CTGAN_SYNTHETIC_DATA = {}
CTGAN_GENERATION_RECORDS = []


# --------------------------------------------------------------------------------------------------
# 2. AUTHORITATIVE DATASET SEMANTICS
# --------------------------------------------------------------------------------------------------

CTGAN_TARGET_COLUMNS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

CTGAN_IDENTIFIER_COLUMNS = {
    "adult_income": [],
    "bank_marketing": [],
    "diabetes_130us": [
        "encounter_id",
        "patient_nbr",
    ],
}

CTGAN_PROVENANCE_COLUMN = "__original_row_id__"


# --------------------------------------------------------------------------------------------------
# 3. VALIDATE REQUIRED REGISTRIES
# --------------------------------------------------------------------------------------------------

required_registries = {
    "DATASET_IDS": DATASET_IDS,
    "CTGAN_MODELS": CTGAN_MODELS,
    "TRAINING_DATA": TRAINING_DATA,
    "DATASET_SEEDS": DATASET_SEEDS,
}

for registry_name, registry in required_registries.items():

    if registry is None:
        raise RuntimeError(
            f"{registry_name} is not defined or is None."
        )


for dataset_id in DATASET_IDS:

    if dataset_id not in CTGAN_MODELS:
        raise KeyError(
            f"CTGAN model missing for dataset: {dataset_id}"
        )

    if dataset_id not in TRAINING_DATA:
        raise KeyError(
            f"Training data missing for dataset: {dataset_id}"
        )

    if dataset_id not in DATASET_SEEDS:
        raise KeyError(
            f"Dataset seed missing for dataset: {dataset_id}"
        )

    if dataset_id not in CTGAN_TARGET_COLUMNS:
        raise KeyError(
            f"Target column not registered for dataset: {dataset_id}"
        )

    if dataset_id not in CTGAN_IDENTIFIER_COLUMNS:
        raise KeyError(
            f"Identifier policy not registered for dataset: {dataset_id}"
        )


# --------------------------------------------------------------------------------------------------
# 4. GENERATE AND VALIDATE SYNTHETIC DATA
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"Generating synthetic data : {dataset_id}")

    model = CTGAN_MODELS[dataset_id]
    train_df = TRAINING_DATA[dataset_id]

    seed = int(DATASET_SEEDS[dataset_id])

    target_column = CTGAN_TARGET_COLUMNS[dataset_id]
    identifier_columns = CTGAN_IDENTIFIER_COLUMNS[dataset_id]


    # ----------------------------------------------------------------------------------------------
    # 4.1 Validate training dataframe
    # ----------------------------------------------------------------------------------------------

    if not isinstance(train_df, pd.DataFrame):
        raise TypeError(
            f"Training data for {dataset_id} is not a pandas DataFrame."
        )

    if train_df.empty:
        raise RuntimeError(
            f"Training data is empty for {dataset_id}."
        )

    if train_df.columns.duplicated().any():

        duplicate_columns = (
            train_df.columns[
                train_df.columns.duplicated()
            ].tolist()
        )

        raise RuntimeError(
            f"Duplicate training columns detected for "
            f"{dataset_id}: {duplicate_columns}"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.2 TRAINING_DATA is the authoritative generative schema
    # ----------------------------------------------------------------------------------------------

    expected_columns = list(train_df.columns)


    # ----------------------------------------------------------------------------------------------
    # 4.3 Validate target policy
    # ----------------------------------------------------------------------------------------------

    if target_column not in expected_columns:
        raise RuntimeError(
            f"Target column '{target_column}' is missing from "
            f"validated training data for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 4.4 Validate provenance exclusion
    # ----------------------------------------------------------------------------------------------

    if CTGAN_PROVENANCE_COLUMN in expected_columns:
        raise RuntimeError(
            f"Provenance column '{CTGAN_PROVENANCE_COLUMN}' is present "
            f"in CTGAN training data for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 4.5 Validate identifier exclusion
    # ----------------------------------------------------------------------------------------------

    leaked_training_identifiers = [
        column
        for column in identifier_columns
        if column in expected_columns
    ]

    if leaked_training_identifiers:

        raise RuntimeError(
            f"Identifier leakage detected in CTGAN training data "
            f"for {dataset_id}: {leaked_training_identifiers}"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.6 Determine synthetic sample size
    # ----------------------------------------------------------------------------------------------

    sample_size = int(len(train_df))

    if sample_size <= 0:
        raise RuntimeError(
            f"Invalid synthetic sample size for {dataset_id}: "
            f"{sample_size}"
        )


    print(f"✓ Training rows  : {sample_size:,}")
    print(f"✓ Expected cols  : {len(expected_columns)}")
    print(f"✓ Target         : {target_column}")
    print(f"✓ Seed           : {seed}")


    # ----------------------------------------------------------------------------------------------
    # 4.7 Apply dataset-specific seed immediately before sampling
    # ----------------------------------------------------------------------------------------------

    seed_everything(seed)


    # ----------------------------------------------------------------------------------------------
    # 4.8 Generate synthetic data
    # ----------------------------------------------------------------------------------------------

    start_time = time.perf_counter()

    synthetic_df = model.sample(
        num_rows=sample_size
    )

    generation_runtime = (
        time.perf_counter() - start_time
    )


    # ----------------------------------------------------------------------------------------------
    # 4.9 Validate returned object
    # ----------------------------------------------------------------------------------------------

    if synthetic_df is None:
        raise RuntimeError(
            f"CTGAN returned None for synthetic data: {dataset_id}"
        )

    if not isinstance(synthetic_df, pd.DataFrame):
        raise TypeError(
            f"CTGAN output for {dataset_id} is not a pandas DataFrame."
        )

    synthetic_df = synthetic_df.copy()


    # ----------------------------------------------------------------------------------------------
    # 4.10 Validate non-empty output
    # ----------------------------------------------------------------------------------------------

    if synthetic_df.empty:
        raise RuntimeError(
            f"Generated synthetic dataframe is empty for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 4.11 Validate duplicate columns
    # ----------------------------------------------------------------------------------------------

    if synthetic_df.columns.duplicated().any():

        duplicate_columns = (
            synthetic_df.columns[
                synthetic_df.columns.duplicated()
            ].tolist()
        )

        raise RuntimeError(
            f"Duplicate synthetic columns detected for "
            f"{dataset_id}: {duplicate_columns}"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.12 Validate exact row count
    # ----------------------------------------------------------------------------------------------

    synthetic_rows = int(len(synthetic_df))

    if synthetic_rows != sample_size:
        raise RuntimeError(
            f"Synthetic row-count mismatch for {dataset_id}: "
            f"expected={sample_size}, "
            f"actual={synthetic_rows}"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.13 Validate exact synthetic schema
    # ----------------------------------------------------------------------------------------------

    actual_columns = list(synthetic_df.columns)

    if actual_columns != expected_columns:

        missing_columns = [
            column
            for column in expected_columns
            if column not in actual_columns
        ]

        unexpected_columns = [
            column
            for column in actual_columns
            if column not in expected_columns
        ]

        raise RuntimeError(
            f"Synthetic schema mismatch for {dataset_id}. "
            f"Missing={missing_columns}; "
            f"Unexpected={unexpected_columns}; "
            f"Expected order={expected_columns}; "
            f"Actual order={actual_columns}"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.14 Validate target presence
    # ----------------------------------------------------------------------------------------------

    if target_column not in synthetic_df.columns:
        raise RuntimeError(
            f"Synthetic target '{target_column}' missing "
            f"for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 4.15 Validate provenance exclusion
    # ----------------------------------------------------------------------------------------------

    if CTGAN_PROVENANCE_COLUMN in synthetic_df.columns:
        raise RuntimeError(
            f"Provenance leakage detected in synthetic data "
            f"for {dataset_id}: "
            f"'{CTGAN_PROVENANCE_COLUMN}'"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.16 Validate identifier exclusion
    # ----------------------------------------------------------------------------------------------

    leaked_synthetic_identifiers = [
        column
        for column in identifier_columns
        if column in synthetic_df.columns
    ]

    if leaked_synthetic_identifiers:

        raise RuntimeError(
            f"Identifier leakage detected in synthetic data "
            f"for {dataset_id}: "
            f"{leaked_synthetic_identifiers}"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.17 Validate synthetic column count
    # ----------------------------------------------------------------------------------------------

    synthetic_columns = int(synthetic_df.shape[1])

    if synthetic_columns != len(expected_columns):

        raise RuntimeError(
            f"Synthetic column-count mismatch for {dataset_id}: "
            f"expected={len(expected_columns)}, "
            f"actual={synthetic_columns}"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.18 Store only validated synthetic data
    # ----------------------------------------------------------------------------------------------

    CTGAN_SYNTHETIC_DATA[dataset_id] = synthetic_df


    # ----------------------------------------------------------------------------------------------
    # 4.19 Record generation metadata
    # ----------------------------------------------------------------------------------------------

    CTGAN_GENERATION_RECORDS.append({
        "dataset_id": dataset_id,
        "seed": seed,
        "training_rows": sample_size,
        "synthetic_rows": synthetic_rows,
        "training_columns": len(expected_columns),
        "synthetic_columns": synthetic_columns,
        "target_column": target_column,
        "generation_runtime_seconds": float(
            generation_runtime
        ),
        "schema_validation": "PASS",
        "row_count_validation": "PASS",
        "target_validation": "PASS",
        "provenance_exclusion": "PASS",
        "identifier_exclusion": "PASS",
        "status": "PASS",
    })


    # ----------------------------------------------------------------------------------------------
    # 4.20 Dataset-level output
    # ----------------------------------------------------------------------------------------------

    print(f"✓ Synthetic rows : {synthetic_rows:,}")
    print(f"✓ Columns        : {synthetic_columns}")
    print("✓ Schema         : PASS")
    print(f"✓ Target         : {target_column} | PASS")
    print("✓ Provenance     : EXCLUDED | PASS")
    print("✓ Identifiers    : EXCLUDED | PASS")
    print(f"✓ Runtime        : {generation_runtime:.3f} seconds")
    print("✓ Status         : PASS")


# --------------------------------------------------------------------------------------------------
# 5. CREATE GENERATION SUMMARY
# --------------------------------------------------------------------------------------------------

CTGAN_GENERATION_DF = pd.DataFrame(
    CTGAN_GENERATION_RECORDS
)


# --------------------------------------------------------------------------------------------------
# 6. FINAL SUMMARY VALIDATION
# --------------------------------------------------------------------------------------------------

if len(CTGAN_GENERATION_DF) != len(DATASET_IDS):

    raise RuntimeError(
        "Synthetic generation summary does not contain exactly "
        "one record per dataset."
    )


if set(CTGAN_GENERATION_DF["dataset_id"]) != set(DATASET_IDS):

    raise RuntimeError(
        "Synthetic generation summary contains an unexpected "
        "dataset registry."
    )


if not (
    CTGAN_GENERATION_DF["status"] == "PASS"
).all():

    raise RuntimeError(
        "One or more synthetic generation runs did not pass validation."
    )


if not (
    CTGAN_GENERATION_DF["schema_validation"] == "PASS"
).all():

    raise RuntimeError(
        "One or more synthetic datasets failed schema validation."
    )


if not (
    CTGAN_GENERATION_DF["row_count_validation"] == "PASS"
).all():

    raise RuntimeError(
        "One or more synthetic datasets failed row-count validation."
    )


if not (
    CTGAN_GENERATION_DF["target_validation"] == "PASS"
).all():

    raise RuntimeError(
        "One or more synthetic datasets failed target validation."
    )


if not (
    CTGAN_GENERATION_DF["provenance_exclusion"] == "PASS"
).all():

    raise RuntimeError(
        "One or more synthetic datasets failed provenance-exclusion validation."
    )


if not (
    CTGAN_GENERATION_DF["identifier_exclusion"] == "PASS"
).all():

    raise RuntimeError(
        "One or more synthetic datasets failed identifier-exclusion validation."
    )


# --------------------------------------------------------------------------------------------------
# 7. FINAL IN-MEMORY VALIDATION
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    synthetic_df = CTGAN_SYNTHETIC_DATA[dataset_id]
    training_df = TRAINING_DATA[dataset_id]

    if len(synthetic_df) != len(training_df):

        raise RuntimeError(
            f"Final synthetic row-count validation failed: "
            f"{dataset_id}"
        )

    if list(synthetic_df.columns) != list(training_df.columns):

        raise RuntimeError(
            f"Final synthetic schema validation failed: "
            f"{dataset_id}"
        )


# --------------------------------------------------------------------------------------------------
# 8. DISPLAY FINAL SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("CTGAN SYNTHETIC GENERATION SUMMARY")
print("=" * 100)

display(
    CTGAN_GENERATION_DF[
        [
            "dataset_id",
            "training_rows",
            "synthetic_rows",
            "training_columns",
            "synthetic_columns",
            "target_column",
            "generation_runtime_seconds",
            "schema_validation",
            "row_count_validation",
            "target_validation",
            "provenance_exclusion",
            "identifier_exclusion",
            "status",
        ]
    ]
)


# --------------------------------------------------------------------------------------------------
# 9. FINAL SECTION STATUS
# --------------------------------------------------------------------------------------------------

print("\n✓ All three CTGAN synthetic datasets generated successfully.")
print("✓ Synthetic row counts match corresponding training row counts.")
print("✓ Exact generative schemas validated.")
print("✓ Target columns validated.")
print("✓ Provenance columns excluded.")
print("✓ Explicit identifier columns excluded.")
print("✓ Generation runtimes recorded.")
print("✓ SECTION 11 — SYNTHETIC GENERATION : PASS")

SECTION 11 — GENERATE SYNTHETIC DATA

----------------------------------------------------------------------------------------------------
Generating synthetic data : adult_income
✓ Training rows  : 34,189
✓ Expected cols  : 15
✓ Target         : income
✓ Seed           : 3126
✓ Synthetic rows : 34,189
✓ Columns        : 15
✓ Schema         : PASS
✓ Target         : income | PASS
✓ Provenance     : EXCLUDED | PASS
✓ Identifiers    : EXCLUDED | PASS
✓ Runtime        : 1.944 seconds
✓ Status         : PASS

----------------------------------------------------------------------------------------------------
Generating synthetic data : bank_marketing
✓ Training rows  : 31,647
✓ Expected cols  : 17
✓ Target         : y
✓ Seed           : 3226
✓ Synthetic rows : 31,647
✓ Columns        : 17
✓ Schema         : PASS
✓ Target         : y | PASS
✓ Provenance     : EXCLUDED | PASS
✓ Identifiers    : EXCLUDED | PASS
✓ Runtime        : 1.002 seconds
✓ Status         : PASS

------------------------

,dataset_id,training_rows,synthetic_rows,training_columns,synthetic_columns,target_column,generation_runtime_seconds,schema_validation,row_count_validation,target_validation,provenance_exclusion,identifier_exclusion,status
0,adult_income,34189,34189,15,15,income,1.944127,PASS,PASS,PASS,PASS,PASS,PASS
1,bank_marketing,31647,31647,17,17,y,1.002322,PASS,PASS,PASS,PASS,PASS,PASS
2,diabetes_130us,71236,71236,48,48,readmitted,6.438770,PASS,PASS,PASS,PASS,PASS,PASS



✓ All three CTGAN synthetic datasets generated successfully.
✓ Synthetic row counts match corresponding training row counts.
✓ Exact generative schemas validated.
✓ Target columns validated.
✓ Provenance columns excluded.
✓ Explicit identifier columns excluded.
✓ Generation runtimes recorded.
✓ SECTION 11 — SYNTHETIC GENERATION : PASS


In [45]:
# ==================================================================================================
# SECTION 12 — VALIDATE SYNTHETIC DATA
# ==================================================================================================

print("=" * 100)
print("SECTION 12 — VALIDATE SYNTHETIC DATA")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. RESET VALIDATION CONTAINER
# --------------------------------------------------------------------------------------------------

CTGAN_SYNTHETIC_VALIDATION_RECORDS = []


# --------------------------------------------------------------------------------------------------
# 2. AUTHORITATIVE DATASET SEMANTICS
# --------------------------------------------------------------------------------------------------

CTGAN_TARGET_COLUMNS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

CTGAN_IDENTIFIER_COLUMNS = {
    "adult_income": [],
    "bank_marketing": [],
    "diabetes_130us": [
        "encounter_id",
        "patient_nbr",
    ],
}

CTGAN_PROVENANCE_COLUMN = "__original_row_id__"


# --------------------------------------------------------------------------------------------------
# 3. VALIDATE REQUIRED REGISTRIES
# --------------------------------------------------------------------------------------------------

required_registries = {
    "DATASET_IDS": DATASET_IDS,
    "TRAINING_DATA": TRAINING_DATA,
    "CTGAN_SYNTHETIC_DATA": CTGAN_SYNTHETIC_DATA,
}

for registry_name, registry in required_registries.items():

    if registry is None:
        raise RuntimeError(
            f"{registry_name} is not defined or is None."
        )


for dataset_id in DATASET_IDS:

    if dataset_id not in TRAINING_DATA:
        raise KeyError(
            f"Training data missing for dataset: {dataset_id}"
        )

    if dataset_id not in CTGAN_SYNTHETIC_DATA:
        raise KeyError(
            f"Synthetic data missing for dataset: {dataset_id}"
        )

    if dataset_id not in CTGAN_TARGET_COLUMNS:
        raise KeyError(
            f"Target column not registered for dataset: {dataset_id}"
        )

    if dataset_id not in CTGAN_IDENTIFIER_COLUMNS:
        raise KeyError(
            f"Identifier policy not registered for dataset: {dataset_id}"
        )


# --------------------------------------------------------------------------------------------------
# 4. VALIDATE EACH SYNTHETIC DATASET
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"Validating synthetic data : {dataset_id}")

    train_df = TRAINING_DATA[dataset_id]
    synthetic_df = CTGAN_SYNTHETIC_DATA[dataset_id]

    target = CTGAN_TARGET_COLUMNS[dataset_id]
    identifiers = CTGAN_IDENTIFIER_COLUMNS[dataset_id]

    # TRAINING_DATA was validated in Sections 3–4 and therefore serves as
    # the authoritative generative schema for this baseline.
    expected_columns = list(train_df.columns)


    # ----------------------------------------------------------------------------------------------
    # 4.1 Validate dataframe types
    # ----------------------------------------------------------------------------------------------

    if not isinstance(train_df, pd.DataFrame):
        raise TypeError(
            f"Training data for {dataset_id} is not a pandas DataFrame."
        )

    if not isinstance(synthetic_df, pd.DataFrame):
        raise TypeError(
            f"Synthetic data for {dataset_id} is not a pandas DataFrame."
        )


    # ----------------------------------------------------------------------------------------------
    # 4.2 Validate non-empty datasets
    # ----------------------------------------------------------------------------------------------

    training_nonempty = not train_df.empty
    synthetic_nonempty = not synthetic_df.empty


    # ----------------------------------------------------------------------------------------------
    # 4.3 Validate duplicate columns
    # ----------------------------------------------------------------------------------------------

    training_duplicate_columns = int(
        train_df.columns.duplicated().sum()
    )

    synthetic_duplicate_columns = int(
        synthetic_df.columns.duplicated().sum()
    )

    duplicate_columns_ok = (
        training_duplicate_columns == 0
        and synthetic_duplicate_columns == 0
    )


    # ----------------------------------------------------------------------------------------------
    # 4.4 Validate exact schema
    # ----------------------------------------------------------------------------------------------

    schema_exact = (
        list(synthetic_df.columns) == expected_columns
    )


    # ----------------------------------------------------------------------------------------------
    # 4.5 Identify missing and unexpected columns
    # ----------------------------------------------------------------------------------------------

    missing_columns = [
        column
        for column in expected_columns
        if column not in synthetic_df.columns
    ]

    unexpected_columns = [
        column
        for column in synthetic_df.columns
        if column not in expected_columns
    ]


    # ----------------------------------------------------------------------------------------------
    # 4.6 Validate target presence
    # ----------------------------------------------------------------------------------------------

    target_present = (
        target in synthetic_df.columns
    )


    # ----------------------------------------------------------------------------------------------
    # 4.7 Validate identifier exclusion
    # ----------------------------------------------------------------------------------------------

    identifier_leakage_columns = [
        column
        for column in identifiers
        if column in synthetic_df.columns
    ]

    identifier_leakage = (
        len(identifier_leakage_columns) > 0
    )


    # ----------------------------------------------------------------------------------------------
    # 4.8 Validate provenance exclusion
    # ----------------------------------------------------------------------------------------------

    provenance_present = (
        CTGAN_PROVENANCE_COLUMN in synthetic_df.columns
    )


    # ----------------------------------------------------------------------------------------------
    # 4.9 Validate sample-size equality
    # ----------------------------------------------------------------------------------------------

    training_rows = int(len(train_df))
    synthetic_rows = int(len(synthetic_df))

    sample_size_equal = (
        synthetic_rows == training_rows
    )


    # ----------------------------------------------------------------------------------------------
    # 4.10 Validate column-count equality
    # ----------------------------------------------------------------------------------------------

    training_columns = int(train_df.shape[1])
    synthetic_columns = int(synthetic_df.shape[1])

    column_count_equal = (
        synthetic_columns == training_columns
    )


    # ----------------------------------------------------------------------------------------------
    # 4.11 Determine final dataset validation status
    # ----------------------------------------------------------------------------------------------

    dataset_status = "PASS"

    validation_conditions = [
        training_nonempty,
        synthetic_nonempty,
        duplicate_columns_ok,
        schema_exact,
        target_present,
        not identifier_leakage,
        not provenance_present,
        sample_size_equal,
        column_count_equal,
    ]

    if not all(validation_conditions):
        dataset_status = "FAIL"


    # ----------------------------------------------------------------------------------------------
    # 4.12 Create validation record
    # ----------------------------------------------------------------------------------------------

    record = {
        "dataset_id": dataset_id,
        "training_rows": training_rows,
        "synthetic_rows": synthetic_rows,
        "training_columns": training_columns,
        "synthetic_columns": synthetic_columns,
        "schema_exact": bool(schema_exact),
        "missing_columns": (
            ", ".join(missing_columns)
            if missing_columns
            else ""
        ),
        "unexpected_columns": (
            ", ".join(unexpected_columns)
            if unexpected_columns
            else ""
        ),
        "target_column": target,
        "target_present": bool(target_present),
        "duplicate_columns": synthetic_duplicate_columns,
        "identifier_leakage": bool(identifier_leakage),
        "identifier_leakage_columns": (
            ", ".join(identifier_leakage_columns)
            if identifier_leakage_columns
            else ""
        ),
        "provenance_present": bool(provenance_present),
        "sample_size_equal": bool(sample_size_equal),
        "column_count_equal": bool(column_count_equal),
        "status": dataset_status,
    }

    CTGAN_SYNTHETIC_VALIDATION_RECORDS.append(record)


    # ----------------------------------------------------------------------------------------------
    # 4.13 Dataset-level reporting
    # ----------------------------------------------------------------------------------------------

    print(
        f"✓ Schema       : "
        f"{'PASS' if schema_exact else 'FAIL'}"
    )

    print(
        f"✓ Size         : "
        f"{'PASS' if sample_size_equal else 'FAIL'}"
    )

    print(
        f"✓ Target       : "
        f"{'PASS' if target_present else 'FAIL'}"
    )

    print(
        f"✓ Duplicates   : "
        f"{'PASS' if duplicate_columns_ok else 'FAIL'}"
    )

    print(
        f"✓ Identifiers  : "
        f"{'PASS' if not identifier_leakage else 'FAIL'}"
    )

    print(
        f"✓ Provenance   : "
        f"{'PASS' if not provenance_present else 'FAIL'}"
    )

    print(
        f"✓ Status       : {dataset_status}"
    )


# --------------------------------------------------------------------------------------------------
# 5. CREATE VALIDATION DATAFRAME
# --------------------------------------------------------------------------------------------------

CTGAN_SYNTHETIC_VALIDATION_DF = pd.DataFrame(
    CTGAN_SYNTHETIC_VALIDATION_RECORDS
)


# --------------------------------------------------------------------------------------------------
# 6. VALIDATE SUMMARY COMPLETENESS
# --------------------------------------------------------------------------------------------------

if len(CTGAN_SYNTHETIC_VALIDATION_DF) != len(DATASET_IDS):

    raise RuntimeError(
        "Synthetic validation summary does not contain exactly "
        "one record per dataset."
    )


if set(
    CTGAN_SYNTHETIC_VALIDATION_DF["dataset_id"]
) != set(DATASET_IDS):

    raise RuntimeError(
        "Synthetic validation summary contains an unexpected "
        "dataset registry."
    )


# --------------------------------------------------------------------------------------------------
# 7. FINAL HARD VALIDATION
# --------------------------------------------------------------------------------------------------

if not (
    CTGAN_SYNTHETIC_VALIDATION_DF["status"] == "PASS"
).all():

    failed_datasets = (
        CTGAN_SYNTHETIC_VALIDATION_DF.loc[
            CTGAN_SYNTHETIC_VALIDATION_DF["status"] != "PASS",
            "dataset_id",
        ].tolist()
    )

    raise RuntimeError(
        "Synthetic data validation failed for: "
        f"{failed_datasets}"
    )


# --------------------------------------------------------------------------------------------------
# 8. FINAL VALIDATION ASSERTIONS
# --------------------------------------------------------------------------------------------------

if not (
    CTGAN_SYNTHETIC_VALIDATION_DF["schema_exact"]
).all():

    raise RuntimeError(
        "One or more synthetic datasets failed exact schema validation."
    )


if not (
    CTGAN_SYNTHETIC_VALIDATION_DF["sample_size_equal"]
).all():

    raise RuntimeError(
        "One or more synthetic datasets failed sample-size validation."
    )


if not (
    CTGAN_SYNTHETIC_VALIDATION_DF["target_present"]
).all():

    raise RuntimeError(
        "One or more synthetic datasets failed target-presence validation."
    )


if not (
    CTGAN_SYNTHETIC_VALIDATION_DF["duplicate_columns"] == 0
).all():

    raise RuntimeError(
        "One or more synthetic datasets contain duplicate columns."
    )


if (
    CTGAN_SYNTHETIC_VALIDATION_DF["identifier_leakage"]
).any():

    raise RuntimeError(
        "Identifier leakage detected in one or more synthetic datasets."
    )


if (
    CTGAN_SYNTHETIC_VALIDATION_DF["provenance_present"]
).any():

    raise RuntimeError(
        "Provenance leakage detected in one or more synthetic datasets."
    )


# --------------------------------------------------------------------------------------------------
# 9. DISPLAY FINAL VALIDATION SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("CTGAN SYNTHETIC DATA VALIDATION SUMMARY")
print("=" * 100)

display(
    CTGAN_SYNTHETIC_VALIDATION_DF[
        [
            "dataset_id",
            "training_rows",
            "synthetic_rows",
            "training_columns",
            "synthetic_columns",
            "schema_exact",
            "target_column",
            "target_present",
            "duplicate_columns",
            "identifier_leakage",
            "provenance_present",
            "sample_size_equal",
            "column_count_equal",
            "status",
        ]
    ]
)


# --------------------------------------------------------------------------------------------------
# 10. FINAL SECTION STATUS
# --------------------------------------------------------------------------------------------------

print("\n✓ All three synthetic datasets passed validation.")
print("✓ Exact schema validation passed.")
print("✓ Synthetic row-count validation passed.")
print("✓ Target-presence validation passed.")
print("✓ Duplicate-column validation passed.")
print("✓ Identifier-exclusion validation passed.")
print("✓ Provenance-exclusion validation passed.")
print("✓ SECTION 12 — SYNTHETIC VALIDATION : PASS")

SECTION 12 — VALIDATE SYNTHETIC DATA

----------------------------------------------------------------------------------------------------
Validating synthetic data : adult_income
✓ Schema       : PASS
✓ Size         : PASS
✓ Target       : PASS
✓ Duplicates   : PASS
✓ Identifiers  : PASS
✓ Provenance   : PASS
✓ Status       : PASS

----------------------------------------------------------------------------------------------------
Validating synthetic data : bank_marketing
✓ Schema       : PASS
✓ Size         : PASS
✓ Target       : PASS
✓ Duplicates   : PASS
✓ Identifiers  : PASS
✓ Provenance   : PASS
✓ Status       : PASS

----------------------------------------------------------------------------------------------------
Validating synthetic data : diabetes_130us
✓ Schema       : PASS
✓ Size         : PASS
✓ Target       : PASS
✓ Duplicates   : PASS
✓ Identifiers  : PASS
✓ Provenance   : PASS
✓ Status       : PASS

CTGAN SYNTHETIC DATA VALIDATION SUMMARY


,dataset_id,training_rows,synthetic_rows,training_columns,synthetic_columns,schema_exact,target_column,target_present,duplicate_columns,identifier_leakage,provenance_present,sample_size_equal,column_count_equal,status
0,adult_income,34189,34189,15,15,True,income,True,0,False,False,True,True,PASS
1,bank_marketing,31647,31647,17,17,True,y,True,0,False,False,True,True,PASS
2,diabetes_130us,71236,71236,48,48,True,readmitted,True,0,False,False,True,True,PASS



✓ All three synthetic datasets passed validation.
✓ Exact schema validation passed.
✓ Synthetic row-count validation passed.
✓ Target-presence validation passed.
✓ Duplicate-column validation passed.
✓ Identifier-exclusion validation passed.
✓ Provenance-exclusion validation passed.
✓ SECTION 12 — SYNTHETIC VALIDATION : PASS


In [47]:
# ==================================================================================================
# SECTION 13 — RECORD RUNTIME
# ==================================================================================================

print("=" * 100)
print("SECTION 13 — RECORD RUNTIME")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. RESET RUNTIME CONTAINER
# --------------------------------------------------------------------------------------------------

CTGAN_RUNTIME_RECORDS = []


# --------------------------------------------------------------------------------------------------
# 2. VALIDATE REQUIRED INPUT REGISTRIES
# --------------------------------------------------------------------------------------------------

required_registries = {
    "DATASET_IDS": DATASET_IDS,
    "DATASET_SEEDS": DATASET_SEEDS,
    "CTGAN_TRAINING_DF": CTGAN_TRAINING_DF,
    "CTGAN_GENERATION_DF": CTGAN_GENERATION_DF,
    "CTGAN_SYNTHETIC_VALIDATION_DF": CTGAN_SYNTHETIC_VALIDATION_DF,
}

for registry_name, registry in required_registries.items():

    if registry is None:
        raise RuntimeError(
            f"{registry_name} is not defined or is None."
        )


# --------------------------------------------------------------------------------------------------
# 3. VALIDATE REQUIRED COLUMNS
# --------------------------------------------------------------------------------------------------

required_training_columns = {
    "dataset_id",
    "training_runtime_seconds",
    "training_rows",
    "training_columns",
    "status",
}

required_generation_columns = {
    "dataset_id",
    "generation_runtime_seconds",
    "training_rows",
    "synthetic_rows",
    "synthetic_columns",
    "status",
}

required_validation_columns = {
    "dataset_id",
    "training_rows",
    "synthetic_rows",
    "training_columns",
    "synthetic_columns",
    "schema_exact",
    "sample_size_equal",
    "status",
}

missing_training_columns = (
    required_training_columns
    - set(CTGAN_TRAINING_DF.columns)
)

missing_generation_columns = (
    required_generation_columns
    - set(CTGAN_GENERATION_DF.columns)
)

missing_validation_columns = (
    required_validation_columns
    - set(CTGAN_SYNTHETIC_VALIDATION_DF.columns)
)

if missing_training_columns:
    raise RuntimeError(
        "CTGAN_TRAINING_DF is missing required columns: "
        f"{sorted(missing_training_columns)}"
    )

if missing_generation_columns:
    raise RuntimeError(
        "CTGAN_GENERATION_DF is missing required columns: "
        f"{sorted(missing_generation_columns)}"
    )

if missing_validation_columns:
    raise RuntimeError(
        "CTGAN_SYNTHETIC_VALIDATION_DF is missing required columns: "
        f"{sorted(missing_validation_columns)}"
    )


# --------------------------------------------------------------------------------------------------
# 4. VALIDATE EXACTLY ONE RECORD PER DATASET
# --------------------------------------------------------------------------------------------------

for dataframe_name, dataframe in {
    "CTGAN_TRAINING_DF": CTGAN_TRAINING_DF,
    "CTGAN_GENERATION_DF": CTGAN_GENERATION_DF,
    "CTGAN_SYNTHETIC_VALIDATION_DF": CTGAN_SYNTHETIC_VALIDATION_DF,
}.items():

    if dataframe["dataset_id"].duplicated().any():

        duplicate_datasets = (
            dataframe.loc[
                dataframe["dataset_id"].duplicated(keep=False),
                "dataset_id",
            ]
            .drop_duplicates()
            .tolist()
        )

        raise RuntimeError(
            f"{dataframe_name} contains duplicate dataset records: "
            f"{duplicate_datasets}"
        )

    if set(dataframe["dataset_id"]) != set(DATASET_IDS):

        raise RuntimeError(
            f"{dataframe_name} does not contain exactly the expected "
            f"dataset registry."
        )

    if len(dataframe) != len(DATASET_IDS):

        raise RuntimeError(
            f"{dataframe_name} must contain exactly "
            f"{len(DATASET_IDS)} records; found {len(dataframe)}."
        )


# --------------------------------------------------------------------------------------------------
# 5. VALIDATE UPSTREAM STATUS
# --------------------------------------------------------------------------------------------------

if not (
    CTGAN_TRAINING_DF["status"] == "PASS"
).all():

    failed_training = (
        CTGAN_TRAINING_DF.loc[
            CTGAN_TRAINING_DF["status"] != "PASS",
            "dataset_id",
        ]
        .tolist()
    )

    raise RuntimeError(
        "CTGAN training did not pass for: "
        f"{failed_training}"
    )


if not (
    CTGAN_GENERATION_DF["status"] == "PASS"
).all():

    failed_generation = (
        CTGAN_GENERATION_DF.loc[
            CTGAN_GENERATION_DF["status"] != "PASS",
            "dataset_id",
        ]
        .tolist()
    )

    raise RuntimeError(
        "CTGAN generation did not pass for: "
        f"{failed_generation}"
    )


if not (
    CTGAN_SYNTHETIC_VALIDATION_DF["status"] == "PASS"
).all():

    failed_validation = (
        CTGAN_SYNTHETIC_VALIDATION_DF.loc[
            CTGAN_SYNTHETIC_VALIDATION_DF["status"] != "PASS",
            "dataset_id",
        ]
        .tolist()
    )

    raise RuntimeError(
        "Synthetic-data validation did not pass for: "
        f"{failed_validation}"
    )


# --------------------------------------------------------------------------------------------------
# 6. RECONCILE RUNTIME AND DATASET METADATA
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"Recording runtime : {dataset_id}")


    # ----------------------------------------------------------------------------------------------
    # 6.1 Retrieve exactly one upstream record
    # ----------------------------------------------------------------------------------------------

    training_matches = CTGAN_TRAINING_DF.loc[
        CTGAN_TRAINING_DF["dataset_id"] == dataset_id
    ]

    generation_matches = CTGAN_GENERATION_DF.loc[
        CTGAN_GENERATION_DF["dataset_id"] == dataset_id
    ]

    validation_matches = CTGAN_SYNTHETIC_VALIDATION_DF.loc[
        CTGAN_SYNTHETIC_VALIDATION_DF["dataset_id"] == dataset_id
    ]


    if len(training_matches) != 1:
        raise RuntimeError(
            f"Expected exactly one training record for "
            f"{dataset_id}; found {len(training_matches)}."
        )

    if len(generation_matches) != 1:
        raise RuntimeError(
            f"Expected exactly one generation record for "
            f"{dataset_id}; found {len(generation_matches)}."
        )

    if len(validation_matches) != 1:
        raise RuntimeError(
            f"Expected exactly one validation record for "
            f"{dataset_id}; found {len(validation_matches)}."
        )


    training_record = training_matches.iloc[0]
    generation_record = generation_matches.iloc[0]
    validation_record = validation_matches.iloc[0]


    # ----------------------------------------------------------------------------------------------
    # 6.2 Validate numeric runtime values
    # ----------------------------------------------------------------------------------------------

    training_runtime = float(
        training_record["training_runtime_seconds"]
    )

    generation_runtime = float(
        generation_record["generation_runtime_seconds"]
    )


    if not np.isfinite(training_runtime):
        raise RuntimeError(
            f"Non-finite training runtime for {dataset_id}: "
            f"{training_runtime}"
        )

    if not np.isfinite(generation_runtime):
        raise RuntimeError(
            f"Non-finite generation runtime for {dataset_id}: "
            f"{generation_runtime}"
        )

    if training_runtime < 0:
        raise RuntimeError(
            f"Negative training runtime for {dataset_id}: "
            f"{training_runtime}"
        )

    if generation_runtime < 0:
        raise RuntimeError(
            f"Negative generation runtime for {dataset_id}: "
            f"{generation_runtime}"
        )


    # ----------------------------------------------------------------------------------------------
    # 6.3 Reconcile training rows
    # ----------------------------------------------------------------------------------------------

    training_rows_training = int(
        training_record["training_rows"]
    )

    training_rows_generation = int(
        generation_record["training_rows"]
    )

    training_rows_validation = int(
        validation_record["training_rows"]
    )


    if not (
        training_rows_training
        == training_rows_generation
        == training_rows_validation
    ):

        raise RuntimeError(
            f"Training-row mismatch for {dataset_id}: "
            f"training={training_rows_training}, "
            f"generation={training_rows_generation}, "
            f"validation={training_rows_validation}"
        )


    # ----------------------------------------------------------------------------------------------
    # 6.4 Reconcile training columns
    # ----------------------------------------------------------------------------------------------

    training_columns_training = int(
        training_record["training_columns"]
    )

    training_columns_validation = int(
        validation_record["training_columns"]
    )


    if training_columns_training != training_columns_validation:

        raise RuntimeError(
            f"Training-column mismatch for {dataset_id}: "
            f"training={training_columns_training}, "
            f"validation={training_columns_validation}"
        )


    # ----------------------------------------------------------------------------------------------
    # 6.5 Reconcile synthetic rows
    # ----------------------------------------------------------------------------------------------

    synthetic_rows_generation = int(
        generation_record["synthetic_rows"]
    )

    synthetic_rows_validation = int(
        validation_record["synthetic_rows"]
    )


    if synthetic_rows_generation != synthetic_rows_validation:

        raise RuntimeError(
            f"Synthetic-row mismatch for {dataset_id}: "
            f"generation={synthetic_rows_generation}, "
            f"validation={synthetic_rows_validation}"
        )


    # ----------------------------------------------------------------------------------------------
    # 6.6 Reconcile synthetic columns
    # ----------------------------------------------------------------------------------------------

    synthetic_columns_generation = int(
        generation_record["synthetic_columns"]
    )

    synthetic_columns_validation = int(
        validation_record["synthetic_columns"]
    )


    if synthetic_columns_generation != synthetic_columns_validation:

        raise RuntimeError(
            f"Synthetic-column mismatch for {dataset_id}: "
            f"generation={synthetic_columns_generation}, "
            f"validation={synthetic_columns_validation}"
        )


    # ----------------------------------------------------------------------------------------------
    # 6.7 Validate sample-size equality
    # ----------------------------------------------------------------------------------------------

    if not bool(validation_record["sample_size_equal"]):

        raise RuntimeError(
            f"Sample-size validation is not PASS for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.8 Validate exact schema
    # ----------------------------------------------------------------------------------------------

    if not bool(validation_record["schema_exact"]):

        raise RuntimeError(
            f"Schema validation is not PASS for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.9 Calculate total runtime
    # ----------------------------------------------------------------------------------------------

    total_runtime = (
        training_runtime
        + generation_runtime
    )


    if not np.isfinite(total_runtime):
        raise RuntimeError(
            f"Non-finite total runtime for {dataset_id}: "
            f"{total_runtime}"
        )


    if total_runtime < training_runtime:
        raise RuntimeError(
            f"Total runtime is inconsistent with training runtime "
            f"for {dataset_id}."
        )

    if total_runtime < generation_runtime:
        raise RuntimeError(
            f"Total runtime is inconsistent with generation runtime "
            f"for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.10 Reconcile seed
    # ----------------------------------------------------------------------------------------------

    expected_seed = int(
        DATASET_SEEDS[dataset_id]
    )


    if "seed" in CTGAN_TRAINING_DF.columns:

        recorded_training_seed = int(
            training_record["seed"]
        )

        if recorded_training_seed != expected_seed:

            raise RuntimeError(
                f"Training seed mismatch for {dataset_id}: "
                f"expected={expected_seed}, "
                f"recorded={recorded_training_seed}"
            )


    if "seed" in CTGAN_GENERATION_DF.columns:

        recorded_generation_seed = int(
            generation_record["seed"]
        )

        if recorded_generation_seed != expected_seed:

            raise RuntimeError(
                f"Generation seed mismatch for {dataset_id}: "
                f"expected={expected_seed}, "
                f"recorded={recorded_generation_seed}"
            )


    # ----------------------------------------------------------------------------------------------
    # 6.11 Record reconciled runtime
    # ----------------------------------------------------------------------------------------------

    CTGAN_RUNTIME_RECORDS.append({
        "dataset_id": dataset_id,
        "seed": expected_seed,
        "training_runtime_seconds": training_runtime,
        "generation_runtime_seconds": generation_runtime,
        "total_runtime_seconds": total_runtime,
        "training_rows": training_rows_training,
        "training_columns": training_columns_training,
        "synthetic_rows": synthetic_rows_generation,
        "synthetic_columns": synthetic_columns_generation,
        "status": "PASS",
    })


    # ----------------------------------------------------------------------------------------------
    # 6.12 Dataset-level reporting
    # ----------------------------------------------------------------------------------------------

    print(
        f"✓ Training runtime   : "
        f"{training_runtime:.6f} seconds"
    )

    print(
        f"✓ Generation runtime : "
        f"{generation_runtime:.6f} seconds"
    )

    print(
        f"✓ Total runtime      : "
        f"{total_runtime:.6f} seconds"
    )

    print(
        f"✓ Rows reconciled    : "
        f"{training_rows_training:,} → "
        f"{synthetic_rows_generation:,}"
    )

    print(
        f"✓ Columns reconciled : "
        f"{training_columns_training} → "
        f"{synthetic_columns_generation}"
    )

    print("✓ Upstream records    : PASS")
    print("✓ Runtime values      : PASS")
    print("✓ Metadata reconcile  : PASS")
    print("✓ Status              : PASS")


# --------------------------------------------------------------------------------------------------
# 7. CREATE RUNTIME DATAFRAME
# --------------------------------------------------------------------------------------------------

CTGAN_RUNTIME_DF = pd.DataFrame(
    CTGAN_RUNTIME_RECORDS
)


# --------------------------------------------------------------------------------------------------
# 8. VALIDATE RUNTIME SUMMARY
# --------------------------------------------------------------------------------------------------

if len(CTGAN_RUNTIME_DF) != len(DATASET_IDS):

    raise RuntimeError(
        "Runtime summary does not contain exactly one "
        "record per dataset."
    )


if set(
    CTGAN_RUNTIME_DF["dataset_id"]
) != set(DATASET_IDS):

    raise RuntimeError(
        "Runtime summary contains an unexpected dataset registry."
    )


if CTGAN_RUNTIME_DF["dataset_id"].duplicated().any():

    raise RuntimeError(
        "Runtime summary contains duplicate dataset records."
    )


if not (
    CTGAN_RUNTIME_DF["status"] == "PASS"
).all():

    raise RuntimeError(
        "One or more runtime records failed validation."
    )


# --------------------------------------------------------------------------------------------------
# 9. VALIDATE RUNTIME NUMERICAL INTEGRITY
# --------------------------------------------------------------------------------------------------

for _, row in CTGAN_RUNTIME_DF.iterrows():

    training_runtime = float(
        row["training_runtime_seconds"]
    )

    generation_runtime = float(
        row["generation_runtime_seconds"]
    )

    total_runtime = float(
        row["total_runtime_seconds"]
    )

    if not np.isfinite(training_runtime):
        raise RuntimeError(
            f"Non-finite training runtime in final registry: "
            f"{row['dataset_id']}"
        )

    if not np.isfinite(generation_runtime):
        raise RuntimeError(
            f"Non-finite generation runtime in final registry: "
            f"{row['dataset_id']}"
        )

    if not np.isfinite(total_runtime):
        raise RuntimeError(
            f"Non-finite total runtime in final registry: "
            f"{row['dataset_id']}"
        )

    calculated_total = (
        training_runtime
        + generation_runtime
    )

    if not np.isclose(
        total_runtime,
        calculated_total,
        rtol=0.0,
        atol=1e-9,
    ):

        raise RuntimeError(
            f"Total runtime arithmetic mismatch for "
            f"{row['dataset_id']}: "
            f"recorded={total_runtime}, "
            f"calculated={calculated_total}"
        )


# --------------------------------------------------------------------------------------------------
# 10. SAVE RUNTIME REGISTRY
# --------------------------------------------------------------------------------------------------

CTGAN_RUNTIME_PATH = (
    NB06_HISTORY_ROOT / "ctgan_runtime_summary.csv"
)

CTGAN_RUNTIME_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

CTGAN_RUNTIME_DF.to_csv(
    CTGAN_RUNTIME_PATH,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 11. VERIFY SAVED RUNTIME REGISTRY
# --------------------------------------------------------------------------------------------------

if not CTGAN_RUNTIME_PATH.exists():

    raise RuntimeError(
        f"Runtime registry was not created: "
        f"{CTGAN_RUNTIME_PATH}"
    )


if CTGAN_RUNTIME_PATH.stat().st_size <= 0:

    raise RuntimeError(
        f"Runtime registry is empty: "
        f"{CTGAN_RUNTIME_PATH}"
    )


# Reload the saved registry and verify the persisted artifact.

CTGAN_RUNTIME_RELOAD_DF = pd.read_csv(
    CTGAN_RUNTIME_PATH
)


if len(CTGAN_RUNTIME_RELOAD_DF) != len(DATASET_IDS):

    raise RuntimeError(
        "Reloaded runtime registry does not contain exactly "
        "one record per dataset."
    )


if set(
    CTGAN_RUNTIME_RELOAD_DF["dataset_id"]
) != set(DATASET_IDS):

    raise RuntimeError(
        "Reloaded runtime registry contains an unexpected "
        "dataset registry."
    )


if not (
    CTGAN_RUNTIME_RELOAD_DF["status"] == "PASS"
).all():

    raise RuntimeError(
        "Reloaded runtime registry contains non-PASS records."
    )


# --------------------------------------------------------------------------------------------------
# 12. DISPLAY FINAL RUNTIME SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("CTGAN RUNTIME SUMMARY")
print("=" * 100)

print(
    CTGAN_RUNTIME_DF.to_string(
        index=False
    )
)

print(
    f"\n✓ Runtime registry : {CTGAN_RUNTIME_PATH}"
)

print(
    f"✓ Registry size    : "
    f"{CTGAN_RUNTIME_PATH.stat().st_size:,} bytes"
)


# --------------------------------------------------------------------------------------------------
# 13. FINAL SECTION STATUS
# --------------------------------------------------------------------------------------------------

print("\n✓ Exactly one runtime record confirmed for each dataset.")
print("✓ Training runtimes reconciled with Section 8.")
print("✓ Generation runtimes reconciled with Section 11.")
print("✓ Synthetic row counts reconciled with Section 12.")
print("✓ Synthetic column counts reconciled with Section 12.")
print("✓ Dataset-specific seeds reconciled.")
print("✓ Total runtimes independently validated.")
print("✓ Persisted runtime registry successfully reloaded.")
print("✓ SECTION 13 — RUNTIME RECORDING : PASS")

SECTION 13 — RECORD RUNTIME

----------------------------------------------------------------------------------------------------
Recording runtime : adult_income
✓ Training runtime   : 748.505885 seconds
✓ Generation runtime : 1.944127 seconds
✓ Total runtime      : 750.450013 seconds
✓ Rows reconciled    : 34,189 → 34,189
✓ Columns reconciled : 15 → 15
✓ Upstream records    : PASS
✓ Runtime values      : PASS
✓ Metadata reconcile  : PASS
✓ Status              : PASS

----------------------------------------------------------------------------------------------------
Recording runtime : bank_marketing
✓ Training runtime   : 739.199827 seconds
✓ Generation runtime : 1.002322 seconds
✓ Total runtime      : 740.202149 seconds
✓ Rows reconciled    : 31,647 → 31,647
✓ Columns reconciled : 17 → 17
✓ Upstream records    : PASS
✓ Runtime values      : PASS
✓ Metadata reconcile  : PASS
✓ Status              : PASS

-------------------------------------------------------------------------------

In [49]:
# ==================================================================================================
# SECTION 14 — SAVE MODEL
# ==================================================================================================

print("=" * 100)
print("SECTION 14 — SAVE MODEL")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. RESET MODEL RECORDS
# --------------------------------------------------------------------------------------------------

CTGAN_MODEL_RECORDS = []


# --------------------------------------------------------------------------------------------------
# 2. VALIDATE REQUIRED REGISTRIES
# --------------------------------------------------------------------------------------------------

required_registries = {
    "DATASET_IDS": DATASET_IDS,
    "CTGAN_MODELS": CTGAN_MODELS,
    "DATASET_SEEDS": DATASET_SEEDS,
}

for registry_name, registry in required_registries.items():

    if registry is None:
        raise RuntimeError(
            f"{registry_name} is not defined or is None."
        )


for dataset_id in DATASET_IDS:

    if dataset_id not in CTGAN_MODELS:
        raise KeyError(
            f"CTGAN model missing for dataset: {dataset_id}"
        )

    if dataset_id not in DATASET_SEEDS:
        raise KeyError(
            f"Dataset seed missing for dataset: {dataset_id}"
        )


# --------------------------------------------------------------------------------------------------
# 3. VALIDATE CTGAN CONFIGURATION REGISTRY
# --------------------------------------------------------------------------------------------------

required_ctgan_parameters = {
    "epochs": 300,
    "batch_size": 500,
    "pac": 10,
    "embedding_dim": 128,
    "generator_dim": (256, 256),
    "discriminator_dim": (256, 256),
}


# --------------------------------------------------------------------------------------------------
# 4. SAVE, RELOAD, AND VALIDATE EACH MODEL
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"Saving model : {dataset_id}")

    model = CTGAN_MODELS[dataset_id]

    if model is None:
        raise RuntimeError(
            f"CTGAN model is None for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 4.1 Resolve model directory and path
    # ----------------------------------------------------------------------------------------------

    model_dir = (
        NB06_MODEL_ROOT / dataset_id
    )

    model_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    model_path = (
        model_dir / "ctgan_model.pkl"
    )


    # ----------------------------------------------------------------------------------------------
    # 4.2 Save trained CTGAN model
    # ----------------------------------------------------------------------------------------------

    model.save(
        filepath=str(model_path)
    )


    # ----------------------------------------------------------------------------------------------
    # 4.3 Validate saved file
    # ----------------------------------------------------------------------------------------------

    if not model_path.exists():

        raise RuntimeError(
            f"Model file was not created: {model_path}"
        )


    size_bytes = model_path.stat().st_size

    if size_bytes <= 0:

        raise RuntimeError(
            f"Model file is empty: {model_path}"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.4 Calculate SHA-256
    # ----------------------------------------------------------------------------------------------

    sha256 = calculate_sha256(
        model_path
    )


    if not sha256 or len(sha256) != 64:

        raise RuntimeError(
            f"Invalid SHA-256 generated for {dataset_id}."
        )


    print(
        f"✓ Model size : {size_bytes:,} bytes"
    )

    print(
        f"✓ SHA256     : {sha256}"
    )


    # ----------------------------------------------------------------------------------------------
    # 4.5 Reload persisted model
    # ----------------------------------------------------------------------------------------------

    print("✓ Reloading persisted model...")

    try:

        reloaded_model = model.__class__.load(
            filepath=str(model_path)
        )

    except Exception as exc:

        raise RuntimeError(
            f"Failed to reload persisted CTGAN model for "
            f"{dataset_id}: {exc}"
        ) from exc


    if reloaded_model is None:

        raise RuntimeError(
            f"Reloaded model is None for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 4.6 Validate reloaded model type
    # ----------------------------------------------------------------------------------------------

    if not isinstance(
        reloaded_model,
        model.__class__
    ):

        raise RuntimeError(
            f"Reloaded model type mismatch for {dataset_id}: "
            f"expected={model.__class__.__name__}, "
            f"actual={type(reloaded_model).__name__}"
        )


    print(
        f"✓ Reloaded model type : "
        f"{type(reloaded_model).__name__}"
    )


    # ----------------------------------------------------------------------------------------------
    # 4.7 Validate persisted model parameters
    # ----------------------------------------------------------------------------------------------

    original_parameters = model.get_parameters()
    reloaded_parameters = reloaded_model.get_parameters()


    parameter_validation = {}

    for parameter_name, expected_value in required_ctgan_parameters.items():

        if parameter_name not in original_parameters:

            raise RuntimeError(
                f"Parameter '{parameter_name}' missing from "
                f"original CTGAN model for {dataset_id}."
            )

        if parameter_name not in reloaded_parameters:

            raise RuntimeError(
                f"Parameter '{parameter_name}' missing from "
                f"reloaded CTGAN model for {dataset_id}."
            )


        original_value = original_parameters[
            parameter_name
        ]

        reloaded_value = reloaded_parameters[
            parameter_name
        ]


        if parameter_name in {
            "generator_dim",
            "discriminator_dim",
        }:

            original_value = tuple(
                original_value
            )

            reloaded_value = tuple(
                reloaded_value
            )

            expected_value = tuple(
                expected_value
            )


        parameter_validation[
            parameter_name
        ] = bool(
            original_value == expected_value
            and reloaded_value == expected_value
            and original_value == reloaded_value
        )


        if not parameter_validation[
            parameter_name
        ]:

            raise RuntimeError(
                f"CTGAN parameter validation failed for "
                f"{dataset_id}: {parameter_name} | "
                f"expected={expected_value}, "
                f"original={original_value}, "
                f"reloaded={reloaded_value}"
            )


    print("✓ CTGAN parameters : PASS")


    # ----------------------------------------------------------------------------------------------
    # 4.8 Validate model functional sampling after reload
    # ----------------------------------------------------------------------------------------------

    train_df = TRAINING_DATA[dataset_id]

    expected_columns = list(
        train_df.columns
    )

    validation_sample_size = min(
        10,
        len(train_df)
    )


    if validation_sample_size <= 0:

        raise RuntimeError(
            f"Invalid functional validation sample size for "
            f"{dataset_id}."
        )


    validation_seed = int(
        DATASET_SEEDS[dataset_id]
    )

    seed_everything(
        validation_seed
    )


    print(
        f"✓ Functional reload test : "
        f"sampling {validation_sample_size} rows..."
    )


    try:

        validation_sample = reloaded_model.sample(
            num_rows=validation_sample_size
        )

    except Exception as exc:

        raise RuntimeError(
            f"Reloaded CTGAN model failed functional sampling "
            f"for {dataset_id}: {exc}"
        ) from exc


    if validation_sample is None:

        raise RuntimeError(
            f"Reloaded model returned None during functional "
            f"validation for {dataset_id}."
        )


    if not isinstance(
        validation_sample,
        pd.DataFrame
    ):

        raise TypeError(
            f"Reloaded model returned an unexpected object type "
            f"for {dataset_id}: "
            f"{type(validation_sample).__name__}"
        )


    if validation_sample.empty:

        raise RuntimeError(
            f"Reloaded model returned an empty validation sample "
            f"for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 4.9 Validate functional sample row count
    # ----------------------------------------------------------------------------------------------

    if len(validation_sample) != validation_sample_size:

        raise RuntimeError(
            f"Reloaded model sample-size mismatch for {dataset_id}: "
            f"expected={validation_sample_size}, "
            f"actual={len(validation_sample)}"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.10 Validate functional sample schema
    # ----------------------------------------------------------------------------------------------

    if list(validation_sample.columns) != expected_columns:

        missing_columns = [
            column
            for column in expected_columns
            if column not in validation_sample.columns
        ]

        unexpected_columns = [
            column
            for column in validation_sample.columns
            if column not in expected_columns
        ]

        raise RuntimeError(
            f"Reloaded model schema mismatch for {dataset_id}: "
            f"missing={missing_columns}; "
            f"unexpected={unexpected_columns}"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.11 Validate target after reload
    # ----------------------------------------------------------------------------------------------

    target_columns = {
        "adult_income": "income",
        "bank_marketing": "y",
        "diabetes_130us": "readmitted",
    }

    target_column = target_columns[dataset_id]


    if target_column not in validation_sample.columns:

        raise RuntimeError(
            f"Target '{target_column}' missing from reloaded "
            f"model validation sample for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 4.12 Validate provenance exclusion after reload
    # ----------------------------------------------------------------------------------------------

    provenance_column = "__original_row_id__"


    if provenance_column in validation_sample.columns:

        raise RuntimeError(
            f"Provenance column '{provenance_column}' unexpectedly "
            f"present after model reload for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 4.13 Validate explicit identifier exclusion after reload
    # ----------------------------------------------------------------------------------------------

    identifier_columns = {
        "adult_income": [],
        "bank_marketing": [],
        "diabetes_130us": [
            "encounter_id",
            "patient_nbr",
        ],
    }


    leaked_identifiers = [
        column
        for column in identifier_columns[dataset_id]
        if column in validation_sample.columns
    ]


    if leaked_identifiers:

        raise RuntimeError(
            f"Identifier leakage after model reload for "
            f"{dataset_id}: {leaked_identifiers}"
        )


    print(
        f"✓ Reload functional sample : PASS"
    )

    print(
        f"✓ Reload sample schema     : PASS"
    )

    print(
        f"✓ Reload target            : {target_column} | PASS"
    )

    print(
        "✓ Reload provenance        : EXCLUDED | PASS"
    )

    print(
        "✓ Reload identifiers       : EXCLUDED | PASS"
    )


    # ----------------------------------------------------------------------------------------------
    # 4.14 Record model artifact
    # ----------------------------------------------------------------------------------------------

    CTGAN_MODEL_RECORDS.append({

        "dataset_id":
            dataset_id,

        "model":
            "CTGANSynthesizer",

        "seed":
            int(DATASET_SEEDS[dataset_id]),

        "model_path":
            str(model_path),

        "model_size_bytes":
            int(size_bytes),

        "sha256":
            sha256,

        "reloaded_model_type":
            type(reloaded_model).__name__,

        "parameter_validation":
            "PASS",

        "reload_validation":
            "PASS",

        "functional_sampling_validation":
            "PASS",

        "schema_validation":
            "PASS",

        "target_validation":
            "PASS",

        "provenance_exclusion":
            "PASS",

        "identifier_exclusion":
            "PASS",

        "artifact_status":
            "NEWLY_SAVED",

        "status":
            "PASS",
    })


    print("✓ Model persistence status : PASS")


# --------------------------------------------------------------------------------------------------
# 5. CREATE MODEL DATAFRAME
# --------------------------------------------------------------------------------------------------

CTGAN_MODEL_DF = pd.DataFrame(
    CTGAN_MODEL_RECORDS
)


# --------------------------------------------------------------------------------------------------
# 6. VALIDATE MODEL REGISTRY
# --------------------------------------------------------------------------------------------------

if len(CTGAN_MODEL_DF) != len(DATASET_IDS):

    raise RuntimeError(
        "CTGAN model registry does not contain exactly "
        "one record per dataset."
    )


if set(
    CTGAN_MODEL_DF["dataset_id"]
) != set(DATASET_IDS):

    raise RuntimeError(
        "CTGAN model registry contains an unexpected "
        "dataset registry."
    )


if CTGAN_MODEL_DF["dataset_id"].duplicated().any():

    raise RuntimeError(
        "CTGAN model registry contains duplicate dataset records."
    )


if not (
    CTGAN_MODEL_DF["status"] == "PASS"
).all():

    raise RuntimeError(
        "One or more CTGAN model persistence records failed."
    )


if not (
    CTGAN_MODEL_DF["reload_validation"] == "PASS"
).all():

    raise RuntimeError(
        "One or more persisted CTGAN models failed reload validation."
    )


if not (
    CTGAN_MODEL_DF["functional_sampling_validation"] == "PASS"
).all():

    raise RuntimeError(
        "One or more persisted CTGAN models failed functional "
        "sampling validation."
    )


# --------------------------------------------------------------------------------------------------
# 7. SAVE MODEL REGISTRY
# --------------------------------------------------------------------------------------------------

CTGAN_MODEL_REGISTRY = (
    NB06_MODEL_ROOT / "ctgan_model_registry.csv"
)

CTGAN_MODEL_REGISTRY.parent.mkdir(
    parents=True,
    exist_ok=True
)

CTGAN_MODEL_DF.to_csv(
    CTGAN_MODEL_REGISTRY,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 8. VERIFY MODEL REGISTRY PERSISTENCE
# --------------------------------------------------------------------------------------------------

if not CTGAN_MODEL_REGISTRY.exists():

    raise RuntimeError(
        f"Model registry was not created: "
        f"{CTGAN_MODEL_REGISTRY}"
    )


if CTGAN_MODEL_REGISTRY.stat().st_size <= 0:

    raise RuntimeError(
        f"Model registry is empty: "
        f"{CTGAN_MODEL_REGISTRY}"
    )


CTGAN_MODEL_RELOAD_DF = pd.read_csv(
    CTGAN_MODEL_REGISTRY
)


if len(CTGAN_MODEL_RELOAD_DF) != len(DATASET_IDS):

    raise RuntimeError(
        "Reloaded model registry does not contain exactly "
        "one record per dataset."
    )


if set(
    CTGAN_MODEL_RELOAD_DF["dataset_id"]
) != set(DATASET_IDS):

    raise RuntimeError(
        "Reloaded model registry contains an unexpected "
        "dataset registry."
    )


if not (
    CTGAN_MODEL_RELOAD_DF["status"] == "PASS"
).all():

    raise RuntimeError(
        "Reloaded model registry contains non-PASS records."
    )


# --------------------------------------------------------------------------------------------------
# 9. DISPLAY FINAL MODEL REGISTRY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("CTGAN MODEL PERSISTENCE SUMMARY")
print("=" * 100)

print(
    CTGAN_MODEL_DF[
        [
            "dataset_id",
            "model",
            "seed",
            "model_size_bytes",
            "sha256",
            "reloaded_model_type",
            "parameter_validation",
            "reload_validation",
            "functional_sampling_validation",
            "schema_validation",
            "target_validation",
            "provenance_exclusion",
            "identifier_exclusion",
            "artifact_status",
            "status",
        ]
    ].to_string(index=False)
)


print(
    f"\n✓ Registry : {CTGAN_MODEL_REGISTRY}"
)

print(
    f"✓ Registry size : "
    f"{CTGAN_MODEL_REGISTRY.stat().st_size:,} bytes"
)


# --------------------------------------------------------------------------------------------------
# 10. FINAL SECTION STATUS
# --------------------------------------------------------------------------------------------------

print("\n✓ All three CTGAN models saved successfully.")
print("✓ Saved model files verified as non-empty.")
print("✓ SHA-256 integrity hashes recorded.")
print("✓ Persisted models successfully reloaded.")
print("✓ Reloaded model types validated.")
print("✓ CTGAN parameters validated after persistence.")
print("✓ Reloaded models passed functional sampling tests.")
print("✓ Reloaded model schemas validated.")
print("✓ Target columns validated after reload.")
print("✓ Provenance columns excluded.")
print("✓ Explicit identifiers excluded.")
print("✓ Model registry persisted and successfully reloaded.")
print("✓ SECTION 14 — MODEL PERSISTENCE : PASS")

SECTION 14 — SAVE MODEL

----------------------------------------------------------------------------------------------------
Saving model : adult_income
✓ Model size : 4,073,568 bytes
✓ SHA256     : 8529d9b98460f6ddf1844b125667252b6a3db2b9c446c64c65a2620441de922c
✓ Reloading persisted model...
✓ Reloaded model type : CTGANSynthesizer
✓ CTGAN parameters : PASS
✓ Functional reload test : sampling 10 rows...
✓ Reload functional sample : PASS
✓ Reload sample schema     : PASS
✓ Reload target            : income | PASS
✓ Reload provenance        : EXCLUDED | PASS
✓ Reload identifiers       : EXCLUDED | PASS
✓ Model persistence status : PASS

----------------------------------------------------------------------------------------------------
Saving model : bank_marketing
✓ Model size : 3,904,049 bytes
✓ SHA256     : b9a2d9e9bfd7a87deb7012814ad308fdd3bc7fc91738ce52c9cdc45c8e064ced
✓ Reloading persisted model...
✓ Reloaded model type : CTGANSynthesizer
✓ CTGAN parameters : PASS
✓ Functional r

In [51]:
# ==================================================================================================
# SECTION 15 — SAVE SYNTHETIC DATA
# ==================================================================================================

print("=" * 100)
print("SECTION 15 — SAVE SYNTHETIC DATA")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. RESET SYNTHETIC ARTIFACT RECORDS
# --------------------------------------------------------------------------------------------------

CTGAN_SYNTHETIC_ARTIFACT_RECORDS = []


# --------------------------------------------------------------------------------------------------
# 2. VALIDATE REQUIRED REGISTRIES
# --------------------------------------------------------------------------------------------------

required_registries = {
    "DATASET_IDS": DATASET_IDS,
    "CTGAN_SYNTHETIC_DATA": CTGAN_SYNTHETIC_DATA,
}

for registry_name, registry in required_registries.items():

    if registry is None:
        raise RuntimeError(
            f"{registry_name} is not defined or is None."
        )


for dataset_id in DATASET_IDS:

    if dataset_id not in CTGAN_SYNTHETIC_DATA:
        raise KeyError(
            f"Synthetic data missing for dataset: {dataset_id}"
        )


# --------------------------------------------------------------------------------------------------
# 3. AUTHORITATIVE DATASET SEMANTICS
# --------------------------------------------------------------------------------------------------

CTGAN_TARGET_COLUMNS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

CTGAN_IDENTIFIER_COLUMNS = {
    "adult_income": [],
    "bank_marketing": [],
    "diabetes_130us": [
        "encounter_id",
        "patient_nbr",
    ],
}

CTGAN_PROVENANCE_COLUMN = "__original_row_id__"


# --------------------------------------------------------------------------------------------------
# 4. SAVE AND VALIDATE SYNTHETIC DATA
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"Saving synthetic data : {dataset_id}")

    synthetic_df = CTGAN_SYNTHETIC_DATA[dataset_id]


    # ----------------------------------------------------------------------------------------------
    # 4.1 Validate dataframe
    # ----------------------------------------------------------------------------------------------

    if not isinstance(
        synthetic_df,
        pd.DataFrame
    ):
        raise TypeError(
            f"Synthetic data for {dataset_id} is not a pandas DataFrame."
        )

    if synthetic_df.empty:
        raise RuntimeError(
            f"Synthetic data is empty for {dataset_id}."
        )

    if synthetic_df.columns.duplicated().any():

        duplicate_columns = (
            synthetic_df.columns[
                synthetic_df.columns.duplicated()
            ].tolist()
        )

        raise RuntimeError(
            f"Duplicate synthetic columns detected for "
            f"{dataset_id}: {duplicate_columns}"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.2 Resolve authoritative schema from validated training data
    # ----------------------------------------------------------------------------------------------

    if dataset_id not in TRAINING_DATA:
        raise KeyError(
            f"Training data missing for schema validation: {dataset_id}"
        )

    training_df = TRAINING_DATA[dataset_id]

    expected_columns = list(
        training_df.columns
    )

    target_column = CTGAN_TARGET_COLUMNS[dataset_id]
    identifier_columns = CTGAN_IDENTIFIER_COLUMNS[dataset_id]


    # ----------------------------------------------------------------------------------------------
    # 4.3 Validate source synthetic dataframe schema
    # ----------------------------------------------------------------------------------------------

    if list(synthetic_df.columns) != expected_columns:

        missing_columns = [
            column
            for column in expected_columns
            if column not in synthetic_df.columns
        ]

        unexpected_columns = [
            column
            for column in synthetic_df.columns
            if column not in expected_columns
        ]

        raise RuntimeError(
            f"Synthetic schema mismatch before persistence for "
            f"{dataset_id}: "
            f"missing={missing_columns}; "
            f"unexpected={unexpected_columns}"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.4 Validate target
    # ----------------------------------------------------------------------------------------------

    if target_column not in synthetic_df.columns:

        raise RuntimeError(
            f"Target '{target_column}' missing from synthetic data "
            f"for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 4.5 Validate provenance exclusion
    # ----------------------------------------------------------------------------------------------

    if CTGAN_PROVENANCE_COLUMN in synthetic_df.columns:

        raise RuntimeError(
            f"Provenance column '{CTGAN_PROVENANCE_COLUMN}' found "
            f"in synthetic data for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 4.6 Validate identifier exclusion
    # ----------------------------------------------------------------------------------------------

    leaked_identifiers = [
        column
        for column in identifier_columns
        if column in synthetic_df.columns
    ]

    if leaked_identifiers:

        raise RuntimeError(
            f"Identifier leakage detected in synthetic data for "
            f"{dataset_id}: {leaked_identifiers}"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.7 Validate expected row count
    # ----------------------------------------------------------------------------------------------

    expected_rows = int(
        len(training_df)
    )

    source_rows = int(
        len(synthetic_df)
    )

    if source_rows != expected_rows:

        raise RuntimeError(
            f"Synthetic row-count mismatch before persistence "
            f"for {dataset_id}: "
            f"expected={expected_rows}; "
            f"actual={source_rows}"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.8 Calculate source dataframe content hash
    # ----------------------------------------------------------------------------------------------

    source_content_hash = hashlib.sha256()

    source_content_hash.update(
        pd.util.hash_pandas_object(
            synthetic_df,
            index=True
        ).values.tobytes()
    )

    source_content_hash = (
        source_content_hash.hexdigest()
    )


    # ----------------------------------------------------------------------------------------------
    # 4.9 Resolve output directory
    # ----------------------------------------------------------------------------------------------

    dataset_dir = (
        NB06_SYNTHETIC_ROOT / dataset_id
    )

    dataset_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    # ----------------------------------------------------------------------------------------------
    # 4.10 Resolve synthetic artifact path
    # ----------------------------------------------------------------------------------------------

    synthetic_path = (
        dataset_dir / "ctgan_synthetic.csv"
    )


    # ----------------------------------------------------------------------------------------------
    # 4.11 Save synthetic dataframe
    # ----------------------------------------------------------------------------------------------

    synthetic_df.to_csv(
        synthetic_path,
        index=False
    )


    # ----------------------------------------------------------------------------------------------
    # 4.12 Validate saved file
    # ----------------------------------------------------------------------------------------------

    if not synthetic_path.exists():

        raise RuntimeError(
            f"Synthetic file was not created: "
            f"{synthetic_path}"
        )


    file_size = (
        synthetic_path.stat().st_size
    )


    if file_size <= 0:

        raise RuntimeError(
            f"Synthetic file is empty: "
            f"{synthetic_path}"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.13 Calculate persisted file SHA-256
    # ----------------------------------------------------------------------------------------------

    sha256 = calculate_sha256(
        synthetic_path
    )


    if not sha256 or len(sha256) != 64:

        raise RuntimeError(
            f"Invalid SHA-256 generated for "
            f"{dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 4.14 Reload persisted synthetic dataframe
    # ----------------------------------------------------------------------------------------------

    print("✓ Reloading persisted synthetic data...")

    reloaded_df = pd.read_csv(
        synthetic_path
    )


    # ----------------------------------------------------------------------------------------------
    # 4.15 Validate reload type and non-empty state
    # ----------------------------------------------------------------------------------------------

    if not isinstance(
        reloaded_df,
        pd.DataFrame
    ):
        raise TypeError(
            f"Reloaded synthetic artifact for {dataset_id} "
            f"is not a pandas DataFrame."
        )


    if reloaded_df.empty:

        raise RuntimeError(
            f"Reloaded synthetic artifact is empty for "
            f"{dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 4.16 Validate reloaded duplicate columns
    # ----------------------------------------------------------------------------------------------

    if reloaded_df.columns.duplicated().any():

        duplicate_columns = (
            reloaded_df.columns[
                reloaded_df.columns.duplicated()
            ].tolist()
        )

        raise RuntimeError(
            f"Duplicate columns found after persistence for "
            f"{dataset_id}: {duplicate_columns}"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.17 Validate reloaded row count
    # ----------------------------------------------------------------------------------------------

    reloaded_rows = int(
        len(reloaded_df)
    )

    if reloaded_rows != source_rows:

        raise RuntimeError(
            f"Persisted row-count mismatch for {dataset_id}: "
            f"source={source_rows}; "
            f"reloaded={reloaded_rows}"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.18 Validate reloaded schema
    # ----------------------------------------------------------------------------------------------

    if list(reloaded_df.columns) != expected_columns:

        missing_columns = [
            column
            for column in expected_columns
            if column not in reloaded_df.columns
        ]

        unexpected_columns = [
            column
            for column in reloaded_df.columns
            if column not in expected_columns
        ]

        raise RuntimeError(
            f"Persisted schema mismatch for {dataset_id}: "
            f"missing={missing_columns}; "
            f"unexpected={unexpected_columns}"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.19 Validate reloaded target
    # ----------------------------------------------------------------------------------------------

    if target_column not in reloaded_df.columns:

        raise RuntimeError(
            f"Target '{target_column}' missing after persistence "
            f"for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 4.20 Validate reloaded provenance exclusion
    # ----------------------------------------------------------------------------------------------

    if CTGAN_PROVENANCE_COLUMN in reloaded_df.columns:

        raise RuntimeError(
            f"Provenance column '{CTGAN_PROVENANCE_COLUMN}' "
            f"present after persistence for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 4.21 Validate reloaded identifier exclusion
    # ----------------------------------------------------------------------------------------------

    reloaded_identifier_leakage = [
        column
        for column in identifier_columns
        if column in reloaded_df.columns
    ]

    if reloaded_identifier_leakage:

        raise RuntimeError(
            f"Identifier leakage detected after persistence for "
            f"{dataset_id}: {reloaded_identifier_leakage}"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.22 Calculate reloaded dataframe content hash
    # ----------------------------------------------------------------------------------------------

    reloaded_content_hash = hashlib.sha256()

    reloaded_content_hash.update(
        pd.util.hash_pandas_object(
            reloaded_df,
            index=True
        ).values.tobytes()
    )

    reloaded_content_hash = (
        reloaded_content_hash.hexdigest()
    )


    # ----------------------------------------------------------------------------------------------
    # 4.23 Validate content integrity
    # ----------------------------------------------------------------------------------------------

    content_integrity_ok = (
        source_content_hash
        == reloaded_content_hash
    )


    if not content_integrity_ok:

        raise RuntimeError(
            f"Persisted content integrity validation failed "
            f"for {dataset_id}: "
            f"source_hash={source_content_hash}; "
            f"reloaded_hash={reloaded_content_hash}"
        )


    # ----------------------------------------------------------------------------------------------
    # 4.24 Record persistence artifact
    # ----------------------------------------------------------------------------------------------

    CTGAN_SYNTHETIC_ARTIFACT_RECORDS.append({

        "dataset_id":
            dataset_id,

        "synthetic_path":
            str(synthetic_path),

        "rows":
            reloaded_rows,

        "columns":
            int(reloaded_df.shape[1]),

        "target_column":
            target_column,

        "file_size_bytes":
            int(file_size),

        "sha256":
            sha256,

        "source_content_hash":
            source_content_hash,

        "reloaded_content_hash":
            reloaded_content_hash,

        "schema_verified":
            True,

        "row_count_verified":
            True,

        "target_verified":
            True,

        "provenance_exclusion_verified":
            True,

        "identifier_exclusion_verified":
            True,

        "content_verified":
            True,

        "persistence_verified":
            True,

        "status":
            "PASS",
    })


    # ----------------------------------------------------------------------------------------------
    # 4.25 Dataset-level reporting
    # ----------------------------------------------------------------------------------------------

    print(
        f"✓ Rows              : {reloaded_rows:,}"
    )

    print(
        f"✓ Columns           : {reloaded_df.shape[1]}"
    )

    print(
        f"✓ Target            : {target_column} | PASS"
    )

    print(
        "✓ Schema            : PASS"
    )

    print(
        "✓ Row count         : PASS"
    )

    print(
        "✓ Provenance        : EXCLUDED | PASS"
    )

    print(
        "✓ Identifiers       : EXCLUDED | PASS"
    )

    print(
        "✓ Content integrity : PASS"
    )

    print(
        f"✓ Size              : {file_size:,} bytes"
    )

    print(
        f"✓ SHA256            : {sha256}"
    )

    print(
        "✓ Persistence       : PASS"
    )

    del reloaded_df


# --------------------------------------------------------------------------------------------------
# 5. CREATE SYNTHETIC ARTIFACT DATAFRAME
# --------------------------------------------------------------------------------------------------

CTGAN_SYNTHETIC_ARTIFACT_DF = pd.DataFrame(
    CTGAN_SYNTHETIC_ARTIFACT_RECORDS
)


# --------------------------------------------------------------------------------------------------
# 6. VALIDATE ARTIFACT REGISTRY
# --------------------------------------------------------------------------------------------------

if len(
    CTGAN_SYNTHETIC_ARTIFACT_DF
) != len(DATASET_IDS):

    raise RuntimeError(
        "Synthetic artifact registry does not contain exactly "
        "one record per dataset."
    )


if set(
    CTGAN_SYNTHETIC_ARTIFACT_DF["dataset_id"]
) != set(DATASET_IDS):

    raise RuntimeError(
        "Synthetic artifact registry contains an unexpected "
        "dataset registry."
    )


if CTGAN_SYNTHETIC_ARTIFACT_DF[
    "dataset_id"
].duplicated().any():

    raise RuntimeError(
        "Synthetic artifact registry contains duplicate "
        "dataset records."
    )


if not (
    CTGAN_SYNTHETIC_ARTIFACT_DF["status"] == "PASS"
).all():

    raise RuntimeError(
        "One or more synthetic artifact persistence records failed."
    )


# --------------------------------------------------------------------------------------------------
# 7. VALIDATE ALL PERSISTENCE FLAGS
# --------------------------------------------------------------------------------------------------

persistence_flags = [
    "schema_verified",
    "row_count_verified",
    "target_verified",
    "provenance_exclusion_verified",
    "identifier_exclusion_verified",
    "content_verified",
    "persistence_verified",
]

for flag in persistence_flags:

    if not (
        CTGAN_SYNTHETIC_ARTIFACT_DF[flag]
    ).all():

        raise RuntimeError(
            f"Synthetic artifact validation failed for flag: {flag}"
        )


# --------------------------------------------------------------------------------------------------
# 8. SAVE SYNTHETIC ARTIFACT REGISTRY
# --------------------------------------------------------------------------------------------------

CTGAN_SYNTHETIC_REGISTRY = (
    NB06_SYNTHETIC_ROOT
    / "ctgan_synthetic_artifact_registry.csv"
)

CTGAN_SYNTHETIC_REGISTRY.parent.mkdir(
    parents=True,
    exist_ok=True
)

CTGAN_SYNTHETIC_ARTIFACT_DF.to_csv(
    CTGAN_SYNTHETIC_REGISTRY,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 9. VERIFY REGISTRY PERSISTENCE
# --------------------------------------------------------------------------------------------------

if not CTGAN_SYNTHETIC_REGISTRY.exists():

    raise RuntimeError(
        f"Synthetic artifact registry was not created: "
        f"{CTGAN_SYNTHETIC_REGISTRY}"
    )


if CTGAN_SYNTHETIC_REGISTRY.stat().st_size <= 0:

    raise RuntimeError(
        f"Synthetic artifact registry is empty: "
        f"{CTGAN_SYNTHETIC_REGISTRY}"
    )


CTGAN_SYNTHETIC_REGISTRY_RELOAD_DF = pd.read_csv(
    CTGAN_SYNTHETIC_REGISTRY
)


if len(
    CTGAN_SYNTHETIC_REGISTRY_RELOAD_DF
) != len(DATASET_IDS):

    raise RuntimeError(
        "Reloaded synthetic artifact registry does not contain "
        "exactly one record per dataset."
    )


if set(
    CTGAN_SYNTHETIC_REGISTRY_RELOAD_DF["dataset_id"]
) != set(DATASET_IDS):

    raise RuntimeError(
        "Reloaded synthetic artifact registry contains an "
        "unexpected dataset registry."
    )


if not (
    CTGAN_SYNTHETIC_REGISTRY_RELOAD_DF["status"] == "PASS"
).all():

    raise RuntimeError(
        "Reloaded synthetic artifact registry contains "
        "non-PASS records."
    )


# --------------------------------------------------------------------------------------------------
# 10. FINAL SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("CTGAN SYNTHETIC ARTIFACT PERSISTENCE SUMMARY")
print("=" * 100)

print(
    CTGAN_SYNTHETIC_ARTIFACT_DF[
        [
            "dataset_id",
            "rows",
            "columns",
            "target_column",
            "file_size_bytes",
            "sha256",
            "schema_verified",
            "row_count_verified",
            "target_verified",
            "provenance_exclusion_verified",
            "identifier_exclusion_verified",
            "content_verified",
            "persistence_verified",
            "status",
        ]
    ].to_string(index=False)
)


print(
    f"\n✓ Registry : {CTGAN_SYNTHETIC_REGISTRY}"
)

print(
    f"✓ Registry size : "
    f"{CTGAN_SYNTHETIC_REGISTRY.stat().st_size:,} bytes"
)


# --------------------------------------------------------------------------------------------------
# 11. FINAL SECTION STATUS
# --------------------------------------------------------------------------------------------------

print("\n✓ All three synthetic datasets saved successfully.")
print("✓ Synthetic files verified as non-empty.")
print("✓ Exact row counts verified.")
print("✓ Exact schemas verified.")
print("✓ Target columns verified.")
print("✓ Provenance columns excluded.")
print("✓ Explicit identifiers excluded.")
print("✓ Persisted content integrity verified.")
print("✓ SHA-256 file hashes recorded.")
print("✓ Synthetic artifact registry persisted.")
print("✓ Persisted registry successfully reloaded.")
print("✓ SECTION 15 — SYNTHETIC PERSISTENCE : PASS")

SECTION 15 — SAVE SYNTHETIC DATA

----------------------------------------------------------------------------------------------------
Saving synthetic data : adult_income
✓ Reloading persisted synthetic data...
✓ Rows              : 34,189
✓ Columns           : 15
✓ Target            : income | PASS
✓ Schema            : PASS
✓ Row count         : PASS
✓ Provenance        : EXCLUDED | PASS
✓ Identifiers       : EXCLUDED | PASS
✓ Content integrity : PASS
✓ Size              : 3,684,842 bytes
✓ SHA256            : 88e87a50b692ed843562ee5117c670cc12ee0d60169adff4a282546705904cd0
✓ Persistence       : PASS

----------------------------------------------------------------------------------------------------
Saving synthetic data : bank_marketing
✓ Reloading persisted synthetic data...
✓ Rows              : 31,647
✓ Columns           : 17
✓ Target            : y | PASS
✓ Schema            : PASS
✓ Row count         : PASS
✓ Provenance        : EXCLUDED | PASS
✓ Identifiers       : EXCLUDED 

In [53]:
# ==================================================================================================
# SECTION 16 — SAVE METADATA / MANIFEST
# ==================================================================================================

print("=" * 100)
print("SECTION 16 — SAVE METADATA / MANIFEST")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. RESET SECTION 16 REGISTRIES
# --------------------------------------------------------------------------------------------------

CTGAN_METADATA_RECORDS = []
CTGAN_MANIFEST_RECORDS = []


# --------------------------------------------------------------------------------------------------
# 2. AUTHORITATIVE DATASET SEMANTICS
# --------------------------------------------------------------------------------------------------

CTGAN_TARGET_COLUMNS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

CTGAN_IDENTIFIER_COLUMNS = {
    "adult_income": [],
    "bank_marketing": [],
    "diabetes_130us": [
        "encounter_id",
        "patient_nbr",
    ],
}

CTGAN_PROVENANCE_COLUMN = "__original_row_id__"


# --------------------------------------------------------------------------------------------------
# 3. VALIDATE REQUIRED UPSTREAM REGISTRIES
# --------------------------------------------------------------------------------------------------

required_objects = {
    "DATASET_IDS": DATASET_IDS,
    "TRAINING_DATA": TRAINING_DATA,
    "CTGAN_METADATA": CTGAN_METADATA,
    "DATASET_SEEDS": DATASET_SEEDS,
    "CTGAN_MODEL_DF": CTGAN_MODEL_DF,
    "CTGAN_CHECKPOINT_DF": CTGAN_CHECKPOINT_DF,
    "CTGAN_SYNTHETIC_ARTIFACT_DF": CTGAN_SYNTHETIC_ARTIFACT_DF,
    "CTGAN_TRAINING_DF": CTGAN_TRAINING_DF,
    "CTGAN_GENERATION_DF": CTGAN_GENERATION_DF,
    "CTGAN_CONFIG": CTGAN_CONFIG,
}

for object_name, object_value in required_objects.items():

    if object_value is None:

        raise RuntimeError(
            f"Required Section 16 dependency is missing: "
            f"{object_name}"
        )


# --------------------------------------------------------------------------------------------------
# 4. VALIDATE DATASET REGISTRY
# --------------------------------------------------------------------------------------------------

if len(DATASET_IDS) != 3:

    raise RuntimeError(
        f"Expected exactly 3 datasets; found {len(DATASET_IDS)}."
    )


if set(DATASET_IDS) != set(CTGAN_TARGET_COLUMNS):

    raise RuntimeError(
        "DATASET_IDS does not match the authoritative CTGAN target registry."
    )


if set(DATASET_IDS) != set(CTGAN_IDENTIFIER_COLUMNS):

    raise RuntimeError(
        "DATASET_IDS does not match the authoritative CTGAN identifier registry."
    )


# --------------------------------------------------------------------------------------------------
# 5. HELPER — REQUIRE EXACTLY ONE UPSTREAM RECORD
# --------------------------------------------------------------------------------------------------

def get_unique_pass_record(
    dataframe,
    dataset_id,
    dataframe_name
):

    if not isinstance(dataframe, pd.DataFrame):

        raise TypeError(
            f"{dataframe_name} is not a pandas DataFrame."
        )

    if "dataset_id" not in dataframe.columns:

        raise RuntimeError(
            f"{dataframe_name} does not contain dataset_id."
        )

    matching = dataframe[
        dataframe["dataset_id"] == dataset_id
    ]

    if len(matching) != 1:

        raise RuntimeError(
            f"{dataframe_name} must contain exactly one record "
            f"for {dataset_id}; found {len(matching)}."
        )

    record = matching.iloc[0]

    if "status" in dataframe.columns:

        if str(record["status"]) != "PASS":

            raise RuntimeError(
                f"{dataframe_name} record for {dataset_id} "
                f"is not PASS."
            )

    return record


# --------------------------------------------------------------------------------------------------
# 6. PROCESS EACH DATASET
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"Processing metadata / manifest : {dataset_id}")


    # ----------------------------------------------------------------------------------------------
    # 6.1 Retrieve authoritative training data
    # ----------------------------------------------------------------------------------------------

    train_df = TRAINING_DATA[dataset_id]

    if not isinstance(train_df, pd.DataFrame):

        raise TypeError(
            f"Training data for {dataset_id} is not a DataFrame."
        )

    if train_df.empty:

        raise RuntimeError(
            f"Training data is empty for {dataset_id}."
        )


    expected_columns = list(
        train_df.columns
    )

    target_column = CTGAN_TARGET_COLUMNS[
        dataset_id
    ]

    identifier_columns = CTGAN_IDENTIFIER_COLUMNS[
        dataset_id
    ]


    # ----------------------------------------------------------------------------------------------
    # 6.2 Retrieve authoritative seed
    # ----------------------------------------------------------------------------------------------

    if dataset_id not in DATASET_SEEDS:

        raise KeyError(
            f"Dataset seed missing for {dataset_id}."
        )

    seed = int(
        DATASET_SEEDS[dataset_id]
    )


    # ----------------------------------------------------------------------------------------------
    # 6.3 Retrieve CTGAN metadata
    # ----------------------------------------------------------------------------------------------

    if dataset_id not in CTGAN_METADATA:

        raise KeyError(
            f"CTGAN metadata missing for {dataset_id}."
        )

    metadata = CTGAN_METADATA[
        dataset_id
    ]


    if metadata is None:

        raise RuntimeError(
            f"CTGAN metadata is None for {dataset_id}."
        )


    if not hasattr(
        metadata,
        "to_dict"
    ):

        raise TypeError(
            f"CTGAN metadata for {dataset_id} does not provide to_dict()."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.4 Convert metadata to machine-readable dictionary
    # ----------------------------------------------------------------------------------------------

    metadata_dict = metadata.to_dict()


    if not isinstance(
        metadata_dict,
        dict
    ):

        raise TypeError(
            f"CTGAN metadata.to_dict() did not return a dictionary "
            f"for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.5 Validate metadata dataset columns
    # ----------------------------------------------------------------------------------------------

    metadata_columns = []

    if isinstance(
        metadata_dict.get("columns"),
        dict
    ):

        metadata_columns = list(
            metadata_dict["columns"].keys()
        )

    elif isinstance(
        metadata_dict.get("columns"),
        list
    ):

        metadata_columns = [
            item["name"]
            if isinstance(item, dict) and "name" in item
            else str(item)
            for item in metadata_dict["columns"]
        ]


    if not metadata_columns:

        raise RuntimeError(
            f"Unable to resolve metadata columns for {dataset_id}."
        )


    if metadata_columns != expected_columns:

        raise RuntimeError(
            f"Metadata schema mismatch for {dataset_id}: "
            f"expected={expected_columns}; "
            f"metadata={metadata_columns}"
        )


    # ----------------------------------------------------------------------------------------------
    # 6.6 Validate target metadata
    # ----------------------------------------------------------------------------------------------

    if target_column not in metadata_columns:

        raise RuntimeError(
            f"Target '{target_column}' is missing from CTGAN metadata "
            f"for {dataset_id}."
        )


    target_metadata = (
        metadata_dict["columns"].get(target_column)
        if isinstance(metadata_dict.get("columns"), dict)
        else None
    )


    if isinstance(target_metadata, dict):

        target_sdtype = target_metadata.get(
            "sdtype"
        )

        if target_sdtype != "categorical":

            raise RuntimeError(
                f"Target metadata sdtype for {dataset_id} "
                f"must be categorical; found {target_sdtype}."
            )


    # ----------------------------------------------------------------------------------------------
    # 6.7 Validate provenance exclusion
    # ----------------------------------------------------------------------------------------------

    if CTGAN_PROVENANCE_COLUMN in metadata_columns:

        raise RuntimeError(
            f"Provenance column '{CTGAN_PROVENANCE_COLUMN}' "
            f"found in CTGAN metadata for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.8 Validate identifier exclusion
    # ----------------------------------------------------------------------------------------------

    metadata_identifier_leakage = [
        column
        for column in identifier_columns
        if column in metadata_columns
    ]

    if metadata_identifier_leakage:

        raise RuntimeError(
            f"Identifier columns found in CTGAN metadata for "
            f"{dataset_id}: {metadata_identifier_leakage}"
        )


    # ----------------------------------------------------------------------------------------------
    # 6.9 Retrieve validated upstream artifact records
    # ----------------------------------------------------------------------------------------------

    model_row = get_unique_pass_record(
        CTGAN_MODEL_DF,
        dataset_id,
        "CTGAN_MODEL_DF"
    )

    checkpoint_row = get_unique_pass_record(
        CTGAN_CHECKPOINT_DF,
        dataset_id,
        "CTGAN_CHECKPOINT_DF"
    )

    synthetic_row = get_unique_pass_record(
        CTGAN_SYNTHETIC_ARTIFACT_DF,
        dataset_id,
        "CTGAN_SYNTHETIC_ARTIFACT_DF"
    )

    history_row = get_unique_pass_record(
        CTGAN_TRAINING_DF,
        dataset_id,
        "CTGAN_TRAINING_DF"
    )

    generation_row = get_unique_pass_record(
        CTGAN_GENERATION_DF,
        dataset_id,
        "CTGAN_GENERATION_DF"
    )


    # ----------------------------------------------------------------------------------------------
    # 6.10 Validate upstream seed consistency
    # ----------------------------------------------------------------------------------------------

    upstream_seed_values = {
        "model": int(model_row["seed"]),
        "checkpoint": int(checkpoint_row["seed"]),
        "synthetic": (
            int(synthetic_row["seed"])
            if "seed" in synthetic_row.index
            and pd.notna(synthetic_row["seed"])
            else seed
        ),
    }


    for source_name, source_seed in upstream_seed_values.items():

        if source_seed != seed:

            raise RuntimeError(
                f"Seed mismatch for {dataset_id}: "
                f"authoritative={seed}; "
                f"{source_name}={source_seed}"
            )


    # ----------------------------------------------------------------------------------------------
    # 6.11 Validate training dimensions
    # ----------------------------------------------------------------------------------------------

    training_rows = int(
        len(train_df)
    )

    training_columns = int(
        train_df.shape[1]
    )

    if training_rows <= 0:

        raise RuntimeError(
            f"Invalid training row count for {dataset_id}."
        )

    if training_columns != len(expected_columns):

        raise RuntimeError(
            f"Training column-count mismatch for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.12 Validate synthetic dimensions
    # ----------------------------------------------------------------------------------------------

    synthetic_rows = int(
        synthetic_row["rows"]
    )

    synthetic_columns = int(
        synthetic_row["columns"]
    )


    if synthetic_rows != training_rows:

        raise RuntimeError(
            f"Training/synthetic row mismatch for {dataset_id}: "
            f"training={training_rows}; "
            f"synthetic={synthetic_rows}"
        )


    if synthetic_columns != training_columns:

        raise RuntimeError(
            f"Training/synthetic column mismatch for {dataset_id}: "
            f"training={training_columns}; "
            f"synthetic={synthetic_columns}"
        )


    # ----------------------------------------------------------------------------------------------
    # 6.13 Validate synthetic artifact path
    # ----------------------------------------------------------------------------------------------

    synthetic_path = Path(
        str(synthetic_row["synthetic_path"])
    )

    if not synthetic_path.exists():

        raise RuntimeError(
            f"Synthetic artifact missing for {dataset_id}: "
            f"{synthetic_path}"
        )


    if synthetic_path.stat().st_size <= 0:

        raise RuntimeError(
            f"Synthetic artifact is empty for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.14 Validate model artifact
    # ----------------------------------------------------------------------------------------------

    model_path = Path(
        str(model_row["model_path"])
    )

    if not model_path.exists():

        raise RuntimeError(
            f"CTGAN model artifact missing for {dataset_id}: "
            f"{model_path}"
        )


    if model_path.stat().st_size <= 0:

        raise RuntimeError(
            f"CTGAN model artifact is empty for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.15 Validate checkpoint artifact
    # ----------------------------------------------------------------------------------------------

    checkpoint_path = Path(
        str(checkpoint_row["checkpoint_path"])
    )

    if not checkpoint_path.exists():

        raise RuntimeError(
            f"CTGAN checkpoint missing for {dataset_id}: "
            f"{checkpoint_path}"
        )


    if checkpoint_path.stat().st_size <= 0:

        raise RuntimeError(
            f"CTGAN checkpoint is empty for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.16 Validate runtime information
    # ----------------------------------------------------------------------------------------------

    training_runtime = float(
        history_row["training_runtime_seconds"]
    )

    generation_runtime = float(
        generation_row["generation_runtime_seconds"]
    )


    if (
        not np.isfinite(training_runtime)
        or training_runtime < 0
    ):

        raise RuntimeError(
            f"Invalid training runtime for {dataset_id}: "
            f"{training_runtime}"
        )


    if (
        not np.isfinite(generation_runtime)
        or generation_runtime < 0
    ):

        raise RuntimeError(
            f"Invalid generation runtime for {dataset_id}: "
            f"{generation_runtime}"
        )


    # ----------------------------------------------------------------------------------------------
    # 6.17 Create metadata JSON record
    # ----------------------------------------------------------------------------------------------

    metadata_dir = (
        NB06_METADATA_ROOT / dataset_id
    )

    metadata_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    metadata_path = (
        metadata_dir / "ctgan_metadata.json"
    )


    metadata_record = {

        "dataset_id":
            dataset_id,

        "model":
            "CTGANSynthesizer",

        "seed":
            seed,

        "target_column":
            target_column,

        "identifier_columns":
            identifier_columns,

        "provenance_column":
            CTGAN_PROVENANCE_COLUMN,

        "generative_columns":
            expected_columns,

        "fit_split":
            "train",

        "training_rows":
            training_rows,

        "training_columns":
            training_columns,

        "validation_used":
            False,

        "test_used":
            False,

        "target_retained":
            True,

        "target_used_as_predictor":
            False,

        "provenance_excluded":
            True,

        "identifiers_excluded":
            True,

        "differential_privacy":
            False,

        "statistical_guidance":
            False,

        "sppgan_components":
            False,

        "metadata":
            metadata_dict,
    }


    # ----------------------------------------------------------------------------------------------
    # 6.18 Persist metadata JSON
    # ----------------------------------------------------------------------------------------------

    with open(
        metadata_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            metadata_record,
            f,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )


    if not metadata_path.exists():

        raise RuntimeError(
            f"Metadata file was not created for {dataset_id}."
        )


    if metadata_path.stat().st_size <= 0:

        raise RuntimeError(
            f"Metadata file is empty for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.19 Calculate metadata SHA-256
    # ----------------------------------------------------------------------------------------------

    metadata_sha256 = calculate_sha256(
        metadata_path
    )


    if (
        not metadata_sha256
        or len(metadata_sha256) != 64
    ):

        raise RuntimeError(
            f"Invalid metadata SHA-256 for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.20 Reload metadata JSON
    # ----------------------------------------------------------------------------------------------

    with open(
        metadata_path,
        "r",
        encoding="utf-8"
    ) as f:

        reloaded_metadata_record = json.load(f)


    if not isinstance(
        reloaded_metadata_record,
        dict
    ):

        raise RuntimeError(
            f"Reloaded metadata is not a dictionary for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.21 Validate reloaded metadata JSON
    # ----------------------------------------------------------------------------------------------

    if reloaded_metadata_record["dataset_id"] != dataset_id:

        raise RuntimeError(
            f"Metadata dataset_id mismatch for {dataset_id}."
        )


    if reloaded_metadata_record["model"] != "CTGANSynthesizer":

        raise RuntimeError(
            f"Metadata model mismatch for {dataset_id}."
        )


    if int(
        reloaded_metadata_record["seed"]
    ) != seed:

        raise RuntimeError(
            f"Metadata seed mismatch for {dataset_id}."
        )


    if (
        reloaded_metadata_record["target_column"]
        != target_column
    ):

        raise RuntimeError(
            f"Metadata target mismatch for {dataset_id}."
        )


    if (
        reloaded_metadata_record["generative_columns"]
        != expected_columns
    ):

        raise RuntimeError(
            f"Metadata generative schema mismatch for {dataset_id}."
        )


    if (
        reloaded_metadata_record["training_rows"]
        != training_rows
    ):

        raise RuntimeError(
            f"Metadata training row mismatch for {dataset_id}."
        )


    if (
        reloaded_metadata_record["training_columns"]
        != training_columns
    ):

        raise RuntimeError(
            f"Metadata training column mismatch for {dataset_id}."
        )


    if (
        reloaded_metadata_record["provenance_excluded"]
        is not True
    ):

        raise RuntimeError(
            f"Metadata provenance policy failed for {dataset_id}."
        )


    if (
        reloaded_metadata_record["identifiers_excluded"]
        is not True
    ):

        raise RuntimeError(
            f"Metadata identifier policy failed for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.22 Record metadata artifact
    # ----------------------------------------------------------------------------------------------

    CTGAN_METADATA_RECORDS.append({

        "dataset_id":
            dataset_id,

        "metadata_path":
            str(metadata_path),

        "metadata_size_bytes":
            int(metadata_path.stat().st_size),

        "sha256":
            metadata_sha256,

        "metadata_columns":
            int(len(expected_columns)),

        "training_rows":
            training_rows,

        "training_columns":
            training_columns,

        "target_column":
            target_column,

        "seed":
            seed,

        "validation_used":
            False,

        "test_used":
            False,

        "differential_privacy":
            False,

        "statistical_guidance":
            False,

        "sppgan_components":
            False,

        "reload_verified":
            True,

        "schema_verified":
            True,

        "status":
            "PASS",
    })


    # ----------------------------------------------------------------------------------------------
    # 6.23 Create manifest record
    # ----------------------------------------------------------------------------------------------

    manifest_record = {

        "dataset_id":
            dataset_id,

        "model":
            "CTGANSynthesizer",

        "seed":
            seed,

        "fit_split":
            "train",

        "training_rows":
            training_rows,

        "training_columns":
            training_columns,

        "synthetic_rows":
            synthetic_rows,

        "synthetic_columns":
            synthetic_columns,

        "target_column":
            target_column,

        "identifier_columns":
            json.dumps(
                identifier_columns,
                ensure_ascii=False
            ),

        "provenance_column":
            CTGAN_PROVENANCE_COLUMN,

        "validation_used":
            False,

        "test_used":
            False,

        "target_retained":
            True,

        "target_used_as_predictor":
            False,

        "provenance_excluded":
            True,

        "identifiers_excluded":
            True,

        "differential_privacy":
            False,

        "statistical_guidance":
            False,

        "sppgan_components":
            False,

        "model_path":
            str(model_path),

        "model_sha256":
            str(model_row["sha256"]),

        "checkpoint_path":
            str(checkpoint_path),

        "checkpoint_sha256":
            str(checkpoint_row["sha256"]),

        "synthetic_path":
            str(synthetic_path),

        "synthetic_sha256":
            str(synthetic_row["sha256"]),

        "metadata_path":
            str(metadata_path),

        "metadata_sha256":
            metadata_sha256,

        "training_runtime_seconds":
            training_runtime,

        "generation_runtime_seconds":
            generation_runtime,

        "status":
            "PASS",
    }


    CTGAN_MANIFEST_RECORDS.append(
        manifest_record
    )


    # ----------------------------------------------------------------------------------------------
    # 6.24 Dataset-level status
    # ----------------------------------------------------------------------------------------------

    print(
        f"✓ Training rows       : {training_rows:,}"
    )

    print(
        f"✓ Training columns    : {training_columns}"
    )

    print(
        f"✓ Synthetic rows      : {synthetic_rows:,}"
    )

    print(
        f"✓ Synthetic columns   : {synthetic_columns}"
    )

    print(
        f"✓ Target              : {target_column} | PASS"
    )

    print(
        "✓ Metadata schema     : PASS"
    )

    print(
        "✓ Provenance policy   : PASS"
    )

    print(
        "✓ Identifier policy   : PASS"
    )

    print(
        "✓ Seed consistency    : PASS"
    )

    print(
        "✓ Model artifact      : PASS"
    )

    print(
        "✓ Checkpoint artifact : PASS"
    )

    print(
        "✓ Synthetic artifact  : PASS"
    )

    print(
        "✓ Metadata JSON reload: PASS"
    )


# --------------------------------------------------------------------------------------------------
# 7. CREATE METADATA REGISTRY DATAFRAME
# --------------------------------------------------------------------------------------------------

CTGAN_METADATA_DF = pd.DataFrame(
    CTGAN_METADATA_RECORDS
)


# --------------------------------------------------------------------------------------------------
# 8. CREATE MANIFEST DATAFRAME
# --------------------------------------------------------------------------------------------------

CTGAN_MANIFEST_DF = pd.DataFrame(
    CTGAN_MANIFEST_RECORDS
)


# --------------------------------------------------------------------------------------------------
# 9. VALIDATE METADATA REGISTRY
# --------------------------------------------------------------------------------------------------

if len(
    CTGAN_METADATA_DF
) != len(DATASET_IDS):

    raise RuntimeError(
        "Metadata registry must contain exactly one record per dataset."
    )


if set(
    CTGAN_METADATA_DF["dataset_id"]
) != set(DATASET_IDS):

    raise RuntimeError(
        "Metadata registry contains an unexpected dataset set."
    )


if CTGAN_METADATA_DF[
    "dataset_id"
].duplicated().any():

    raise RuntimeError(
        "Metadata registry contains duplicate dataset records."
    )


if not (
    CTGAN_METADATA_DF["status"] == "PASS"
).all():

    raise RuntimeError(
        "One or more metadata registry records failed."
    )


if not (
    CTGAN_METADATA_DF["reload_verified"]
).all():

    raise RuntimeError(
        "Metadata reload validation failed."
    )


# --------------------------------------------------------------------------------------------------
# 10. VALIDATE MANIFEST DATAFRAME
# --------------------------------------------------------------------------------------------------

if len(
    CTGAN_MANIFEST_DF
) != len(DATASET_IDS):

    raise RuntimeError(
        "Manifest must contain exactly one record per dataset."
    )


if set(
    CTGAN_MANIFEST_DF["dataset_id"]
) != set(DATASET_IDS):

    raise RuntimeError(
        "Manifest contains an unexpected dataset set."
    )


if CTGAN_MANIFEST_DF[
    "dataset_id"
].duplicated().any():

    raise RuntimeError(
        "Manifest contains duplicate dataset records."
    )


if not (
    CTGAN_MANIFEST_DF["status"] == "PASS"
).all():

    raise RuntimeError(
        "One or more manifest records failed."
    )


# --------------------------------------------------------------------------------------------------
# 11. CROSS-VALIDATE MANIFEST DIMENSIONS
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    manifest_row = get_unique_pass_record(
        CTGAN_MANIFEST_DF,
        dataset_id,
        "CTGAN_MANIFEST_DF"
    )

    training_rows = int(
        len(TRAINING_DATA[dataset_id])
    )

    training_columns = int(
        TRAINING_DATA[dataset_id].shape[1]
    )

    if int(
        manifest_row["training_rows"]
    ) != training_rows:

        raise RuntimeError(
            f"Manifest training row mismatch for {dataset_id}."
        )


    if int(
        manifest_row["training_columns"]
    ) != training_columns:

        raise RuntimeError(
            f"Manifest training column mismatch for {dataset_id}."
        )


    if int(
        manifest_row["synthetic_rows"]
    ) != training_rows:

        raise RuntimeError(
            f"Manifest synthetic row mismatch for {dataset_id}."
        )


    if int(
        manifest_row["synthetic_columns"]
    ) != training_columns:

        raise RuntimeError(
            f"Manifest synthetic column mismatch for {dataset_id}."
        )


# --------------------------------------------------------------------------------------------------
# 12. CREATE OUTPUT DIRECTORIES
# --------------------------------------------------------------------------------------------------

NB06_METADATA_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

NB06_MANIFEST_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

NB06_CONFIG_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# --------------------------------------------------------------------------------------------------
# 13. SAVE METADATA REGISTRY
# --------------------------------------------------------------------------------------------------

CTGAN_METADATA_REGISTRY = (
    NB06_METADATA_ROOT
    / "ctgan_metadata_registry.csv"
)

CTGAN_METADATA_DF.to_csv(
    CTGAN_METADATA_REGISTRY,
    index=False
)


if not CTGAN_METADATA_REGISTRY.exists():

    raise RuntimeError(
        "Metadata registry was not created."
    )


if CTGAN_METADATA_REGISTRY.stat().st_size <= 0:

    raise RuntimeError(
        "Metadata registry is empty."
    )


# --------------------------------------------------------------------------------------------------
# 14. SAVE CTGAN MANIFEST
# --------------------------------------------------------------------------------------------------

CTGAN_MANIFEST_PATH = (
    NB06_MANIFEST_ROOT
    / "ctgan_manifest.csv"
)

CTGAN_MANIFEST_DF.to_csv(
    CTGAN_MANIFEST_PATH,
    index=False
)


if not CTGAN_MANIFEST_PATH.exists():

    raise RuntimeError(
        "CTGAN manifest was not created."
    )


if CTGAN_MANIFEST_PATH.stat().st_size <= 0:

    raise RuntimeError(
        "CTGAN manifest is empty."
    )


# --------------------------------------------------------------------------------------------------
# 15. SAVE MACHINE-READABLE EXPERIMENT CONFIGURATION
# --------------------------------------------------------------------------------------------------

CTGAN_EXPERIMENT_MANIFEST_PATH = (
    NB06_CONFIG_ROOT
    / "ctgan_experiment_manifest.json"
)


CTGAN_EXPERIMENT_MANIFEST = {

    "notebook_id":
        NOTEBOOK_ID,

    "notebook_name":
        NOTEBOOK_NAME,

    "notebook_version":
        NOTEBOOK_VERSION,

    "project_root":
        str(PROJECT_ROOT),

    "datasets":
        list(DATASET_IDS),

    "dataset_seeds":
        {
            dataset_id: int(DATASET_SEEDS[dataset_id])
            for dataset_id in DATASET_IDS
        },

    "model":
        "CTGANSynthesizer",

    "model_config":
        CTGAN_CONFIG,

    "fit_split":
        "train",

    "validation_used":
        False,

    "test_used":
        False,

    "differential_privacy":
        False,

    "statistical_guidance":
        False,

    "sppgan_components":
        False,

    "target_columns":
        CTGAN_TARGET_COLUMNS,

    "identifier_columns":
        CTGAN_IDENTIFIER_COLUMNS,

    "provenance_column":
        CTGAN_PROVENANCE_COLUMN,

    "metadata_registry":
        str(CTGAN_METADATA_REGISTRY),

    "manifest_path":
        str(CTGAN_MANIFEST_PATH),

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


with open(
    CTGAN_EXPERIMENT_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CTGAN_EXPERIMENT_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )


if not CTGAN_EXPERIMENT_MANIFEST_PATH.exists():

    raise RuntimeError(
        "Machine-readable CTGAN experiment manifest was not created."
    )


if CTGAN_EXPERIMENT_MANIFEST_PATH.stat().st_size <= 0:

    raise RuntimeError(
        "Machine-readable CTGAN experiment manifest is empty."
    )


# --------------------------------------------------------------------------------------------------
# 16. CALCULATE CONFIGURATION MANIFEST SHA-256
# --------------------------------------------------------------------------------------------------

CTGAN_EXPERIMENT_MANIFEST_SHA256 = (
    calculate_sha256(
        CTGAN_EXPERIMENT_MANIFEST_PATH
    )
)


if (
    not CTGAN_EXPERIMENT_MANIFEST_SHA256
    or len(CTGAN_EXPERIMENT_MANIFEST_SHA256) != 64
):

    raise RuntimeError(
        "Invalid SHA-256 for machine-readable experiment manifest."
    )


# --------------------------------------------------------------------------------------------------
# 17. RELOAD METADATA REGISTRY
# --------------------------------------------------------------------------------------------------

CTGAN_METADATA_REGISTRY_RELOAD_DF = pd.read_csv(
    CTGAN_METADATA_REGISTRY
)


if len(
    CTGAN_METADATA_REGISTRY_RELOAD_DF
) != len(DATASET_IDS):

    raise RuntimeError(
        "Reloaded metadata registry has an invalid number of records."
    )


if set(
    CTGAN_METADATA_REGISTRY_RELOAD_DF["dataset_id"]
) != set(DATASET_IDS):

    raise RuntimeError(
        "Reloaded metadata registry contains an unexpected dataset set."
    )


if CTGAN_METADATA_REGISTRY_RELOAD_DF[
    "dataset_id"
].duplicated().any():

    raise RuntimeError(
        "Reloaded metadata registry contains duplicate datasets."
    )


if not (
    CTGAN_METADATA_REGISTRY_RELOAD_DF["status"] == "PASS"
).all():

    raise RuntimeError(
        "Reloaded metadata registry contains non-PASS records."
    )


# --------------------------------------------------------------------------------------------------
# 18. RELOAD CTGAN MANIFEST
# --------------------------------------------------------------------------------------------------

CTGAN_MANIFEST_RELOAD_DF = pd.read_csv(
    CTGAN_MANIFEST_PATH
)


if len(
    CTGAN_MANIFEST_RELOAD_DF
) != len(DATASET_IDS):

    raise RuntimeError(
        "Reloaded CTGAN manifest has an invalid number of records."
    )


if set(
    CTGAN_MANIFEST_RELOAD_DF["dataset_id"]
) != set(DATASET_IDS):

    raise RuntimeError(
        "Reloaded CTGAN manifest contains an unexpected dataset set."
    )


if CTGAN_MANIFEST_RELOAD_DF[
    "dataset_id"
].duplicated().any():

    raise RuntimeError(
        "Reloaded CTGAN manifest contains duplicate datasets."
    )


if not (
    CTGAN_MANIFEST_RELOAD_DF["status"] == "PASS"
).all():

    raise RuntimeError(
        "Reloaded CTGAN manifest contains non-PASS records."
    )


# --------------------------------------------------------------------------------------------------
# 19. RELOAD MACHINE-READABLE CONFIGURATION
# --------------------------------------------------------------------------------------------------

with open(
    CTGAN_EXPERIMENT_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as f:

    CTGAN_EXPERIMENT_MANIFEST_RELOAD = json.load(f)


if not isinstance(
    CTGAN_EXPERIMENT_MANIFEST_RELOAD,
    dict
):

    raise RuntimeError(
        "Reloaded experiment manifest is not a dictionary."
    )


# --------------------------------------------------------------------------------------------------
# 20. VALIDATE MACHINE-READABLE CONFIGURATION
# --------------------------------------------------------------------------------------------------

if (
    CTGAN_EXPERIMENT_MANIFEST_RELOAD["notebook_id"]
    != NOTEBOOK_ID
):

    raise RuntimeError(
        "Experiment manifest notebook_id mismatch."
    )


if (
    CTGAN_EXPERIMENT_MANIFEST_RELOAD["notebook_name"]
    != NOTEBOOK_NAME
):

    raise RuntimeError(
        "Experiment manifest notebook_name mismatch."
    )


if (
    CTGAN_EXPERIMENT_MANIFEST_RELOAD["notebook_version"]
    != NOTEBOOK_VERSION
):

    raise RuntimeError(
        "Experiment manifest notebook_version mismatch."
    )


if (
    CTGAN_EXPERIMENT_MANIFEST_RELOAD["model"]
    != "CTGANSynthesizer"
):

    raise RuntimeError(
        "Experiment manifest model mismatch."
    )


if (
    CTGAN_EXPERIMENT_MANIFEST_RELOAD["datasets"]
    != list(DATASET_IDS)
):

    raise RuntimeError(
        "Experiment manifest dataset registry mismatch."
    )


if (
    CTGAN_EXPERIMENT_MANIFEST_RELOAD["target_columns"]
    != CTGAN_TARGET_COLUMNS
):

    raise RuntimeError(
        "Experiment manifest target registry mismatch."
    )


if (
    CTGAN_EXPERIMENT_MANIFEST_RELOAD["identifier_columns"]
    != CTGAN_IDENTIFIER_COLUMNS
):

    raise RuntimeError(
        "Experiment manifest identifier registry mismatch."
    )


if (
    CTGAN_EXPERIMENT_MANIFEST_RELOAD["provenance_column"]
    != CTGAN_PROVENANCE_COLUMN
):

    raise RuntimeError(
        "Experiment manifest provenance registry mismatch."
    )


if (
    CTGAN_EXPERIMENT_MANIFEST_RELOAD["validation_used"]
    is not False
):

    raise RuntimeError(
        "Experiment manifest validation policy mismatch."
    )


if (
    CTGAN_EXPERIMENT_MANIFEST_RELOAD["test_used"]
    is not False
):

    raise RuntimeError(
        "Experiment manifest test policy mismatch."
    )


if (
    CTGAN_EXPERIMENT_MANIFEST_RELOAD["differential_privacy"]
    is not False
):

    raise RuntimeError(
        "Experiment manifest DP policy mismatch."
    )


if (
    CTGAN_EXPERIMENT_MANIFEST_RELOAD["statistical_guidance"]
    is not False
):

    raise RuntimeError(
        "Experiment manifest statistical-guidance policy mismatch."
    )


if (
    CTGAN_EXPERIMENT_MANIFEST_RELOAD["sppgan_components"]
    is not False
):

    raise RuntimeError(
        "Experiment manifest SPP-GAN policy mismatch."
    )


# --------------------------------------------------------------------------------------------------
# 21. FINAL ARTIFACT SHA-256 SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 16 ARTIFACT SUMMARY")
print("=" * 100)

for dataset_id in DATASET_IDS:

    metadata_row = get_unique_pass_record(
        CTGAN_METADATA_DF,
        dataset_id,
        "CTGAN_METADATA_DF"
    )

    manifest_row = get_unique_pass_record(
        CTGAN_MANIFEST_DF,
        dataset_id,
        "CTGAN_MANIFEST_DF"
    )

    print(
        f"\n{dataset_id}"
    )

    print(
        f"  ✓ Metadata SHA256  : {metadata_row['sha256']}"
    )

    print(
        f"  ✓ Model SHA256     : {manifest_row['model_sha256']}"
    )

    print(
        f"  ✓ Checkpoint SHA256: {manifest_row['checkpoint_sha256']}"
    )

    print(
        f"  ✓ Synthetic SHA256 : {manifest_row['synthetic_sha256']}"
    )

    print(
        f"  ✓ Training rows    : "
        f"{int(manifest_row['training_rows']):,}"
    )

    print(
        f"  ✓ Synthetic rows   : "
        f"{int(manifest_row['synthetic_rows']):,}"
    )

    print(
        f"  ✓ Training columns : "
        f"{int(manifest_row['training_columns'])}"
    )

    print(
        f"  ✓ Synthetic columns: "
        f"{int(manifest_row['synthetic_columns'])}"
    )

    print(
        "  ✓ Cross-artifact consistency : PASS"
    )


print(
    f"\n✓ Metadata registry : "
    f"{CTGAN_METADATA_REGISTRY}"
)

print(
    f"✓ Manifest          : "
    f"{CTGAN_MANIFEST_PATH}"
)

print(
    f"✓ Experiment config : "
    f"{CTGAN_EXPERIMENT_MANIFEST_PATH}"
)

print(
    f"✓ Experiment config SHA256 : "
    f"{CTGAN_EXPERIMENT_MANIFEST_SHA256}"
)


# --------------------------------------------------------------------------------------------------
# 22. FINAL SECTION STATUS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)

print(
    "✓ Metadata JSON files created for all datasets."
)

print(
    "✓ Metadata schemas validated against authoritative training schemas."
)

print(
    "✓ Target metadata validated."
)

print(
    "✓ Provenance exclusion validated."
)

print(
    "✓ Identifier exclusion validated."
)

print(
    "✓ Dataset-specific seeds validated."
)

print(
    "✓ Model artifacts cross-validated."
)

print(
    "✓ Checkpoint artifacts cross-validated."
)

print(
    "✓ Synthetic artifacts cross-validated."
)

print(
    "✓ Training/synthetic dimensions cross-validated."
)

print(
    "✓ Metadata JSON files reloaded successfully."
)

print(
    "✓ Metadata registry persisted and reloaded successfully."
)

print(
    "✓ CTGAN manifest persisted and reloaded successfully."
)

print(
    "✓ Machine-readable experiment configuration persisted."
)

print(
    "✓ Machine-readable experiment configuration reloaded."
)

print(
    "✓ Configuration policy consistency validated."
)

print(
    "✓ SECTION 16 — METADATA / MANIFEST : PASS"
)

print("=" * 100)

SECTION 16 — SAVE METADATA / MANIFEST

----------------------------------------------------------------------------------------------------
Processing metadata / manifest : adult_income
✓ Training rows       : 34,189
✓ Training columns    : 15
✓ Synthetic rows      : 34,189
✓ Synthetic columns   : 15
✓ Target              : income | PASS
✓ Metadata schema     : PASS
✓ Provenance policy   : PASS
✓ Identifier policy   : PASS
✓ Seed consistency    : PASS
✓ Model artifact      : PASS
✓ Checkpoint artifact : PASS
✓ Synthetic artifact  : PASS
✓ Metadata JSON reload: PASS

----------------------------------------------------------------------------------------------------
Processing metadata / manifest : bank_marketing
✓ Training rows       : 31,647
✓ Training columns    : 17
✓ Synthetic rows      : 31,647
✓ Synthetic columns   : 17
✓ Target              : y | PASS
✓ Metadata schema     : PASS
✓ Provenance policy   : PASS
✓ Identifier policy   : PASS
✓ Seed consistency    : PASS
✓ Model artif

In [55]:
# ==================================================================================================
# SECTION 17 — VERIFY ARTIFACTS
# ==================================================================================================

print("=" * 100)
print("SECTION 17 — VERIFY ARTIFACTS")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. RESET VERIFICATION RECORDS
# --------------------------------------------------------------------------------------------------

CTGAN_ARTIFACT_VERIFICATION_RECORDS = []


# --------------------------------------------------------------------------------------------------
# 2. AUTHORITATIVE DATASET SEMANTICS
# --------------------------------------------------------------------------------------------------

CTGAN_TARGET_COLUMNS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

CTGAN_IDENTIFIER_COLUMNS = {
    "adult_income": [],
    "bank_marketing": [],
    "diabetes_130us": [
        "encounter_id",
        "patient_nbr",
    ],
}

CTGAN_PROVENANCE_COLUMN = "__original_row_id__"


# --------------------------------------------------------------------------------------------------
# 3. VALIDATE REQUIRED SECTION 16 OBJECTS
# --------------------------------------------------------------------------------------------------

required_objects = {
    "DATASET_IDS": DATASET_IDS,
    "CTGAN_MANIFEST_DF": CTGAN_MANIFEST_DF,
    "CTGAN_METADATA_DF": CTGAN_METADATA_DF,
    "CTGAN_MODEL_DF": CTGAN_MODEL_DF,
    "CTGAN_CHECKPOINT_DF": CTGAN_CHECKPOINT_DF,
    "CTGAN_SYNTHETIC_ARTIFACT_DF": CTGAN_SYNTHETIC_ARTIFACT_DF,
    "CTGAN_TRAINING_DF": CTGAN_TRAINING_DF,
    "CTGAN_GENERATION_DF": CTGAN_GENERATION_DF,
    "TRAINING_DATA": TRAINING_DATA,
}

for object_name, object_value in required_objects.items():

    if object_value is None:

        raise RuntimeError(
            f"Required Section 17 dependency is missing: "
            f"{object_name}"
        )


# --------------------------------------------------------------------------------------------------
# 4. HELPER — EXACTLY ONE RECORD
# --------------------------------------------------------------------------------------------------

def get_unique_record(
    dataframe,
    dataset_id,
    dataframe_name
):

    if not isinstance(
        dataframe,
        pd.DataFrame
    ):

        raise TypeError(
            f"{dataframe_name} is not a pandas DataFrame."
        )

    if "dataset_id" not in dataframe.columns:

        raise RuntimeError(
            f"{dataframe_name} does not contain dataset_id."
        )

    matching = dataframe[
        dataframe["dataset_id"] == dataset_id
    ]

    if len(matching) != 1:

        raise RuntimeError(
            f"{dataframe_name} must contain exactly one "
            f"record for {dataset_id}; found {len(matching)}."
        )

    return matching.iloc[0]


# --------------------------------------------------------------------------------------------------
# 5. HELPER — BOOLEAN NORMALIZATION
# --------------------------------------------------------------------------------------------------

def require_false(
    value,
    field_name,
    dataset_id
):

    normalized = str(value).strip().lower()

    if normalized not in {
        "false",
        "0",
    }:

        raise RuntimeError(
            f"{field_name} must be False for {dataset_id}; "
            f"found {value!r}"
        )

    return True


def require_true(
    value,
    field_name,
    dataset_id
):

    normalized = str(value).strip().lower()

    if normalized not in {
        "true",
        "1",
    }:

        raise RuntimeError(
            f"{field_name} must be True for {dataset_id}; "
            f"found {value!r}"
        )

    return True


# --------------------------------------------------------------------------------------------------
# 6. VERIFY EACH DATASET
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"Verifying : {dataset_id}")


    # ----------------------------------------------------------------------------------------------
    # 6.1 Retrieve manifest record
    # ----------------------------------------------------------------------------------------------

    manifest_row = get_unique_record(
        CTGAN_MANIFEST_DF,
        dataset_id,
        "CTGAN_MANIFEST_DF"
    )


    # ----------------------------------------------------------------------------------------------
    # 6.2 Manifest status
    # ----------------------------------------------------------------------------------------------

    if str(
        manifest_row["status"]
    ) != "PASS":

        raise RuntimeError(
            f"Manifest status is not PASS for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.3 Retrieve upstream records
    # ----------------------------------------------------------------------------------------------

    model_row = get_unique_record(
        CTGAN_MODEL_DF,
        dataset_id,
        "CTGAN_MODEL_DF"
    )

    checkpoint_row = get_unique_record(
        CTGAN_CHECKPOINT_DF,
        dataset_id,
        "CTGAN_CHECKPOINT_DF"
    )

    synthetic_row = get_unique_record(
        CTGAN_SYNTHETIC_ARTIFACT_DF,
        dataset_id,
        "CTGAN_SYNTHETIC_ARTIFACT_DF"
    )

    metadata_row = get_unique_record(
        CTGAN_METADATA_DF,
        dataset_id,
        "CTGAN_METADATA_DF"
    )

    training_row = get_unique_record(
        CTGAN_TRAINING_DF,
        dataset_id,
        "CTGAN_TRAINING_DF"
    )

    generation_row = get_unique_record(
        CTGAN_GENERATION_DF,
        dataset_id,
        "CTGAN_GENERATION_DF"
    )


    # ----------------------------------------------------------------------------------------------
    # 6.4 Resolve artifact paths
    # ----------------------------------------------------------------------------------------------

    model_path = Path(
        str(manifest_row["model_path"])
    )

    checkpoint_path = Path(
        str(manifest_row["checkpoint_path"])
    )

    synthetic_path = Path(
        str(manifest_row["synthetic_path"])
    )

    metadata_path = Path(
        str(manifest_row["metadata_path"])
    )


    # ----------------------------------------------------------------------------------------------
    # 6.5 Verify artifact existence
    # ----------------------------------------------------------------------------------------------

    model_exists = model_path.exists()
    checkpoint_exists = checkpoint_path.exists()
    synthetic_exists = synthetic_path.exists()
    metadata_exists = metadata_path.exists()


    if not all([
        model_exists,
        checkpoint_exists,
        synthetic_exists,
        metadata_exists,
    ]):

        raise RuntimeError(
            f"One or more artifacts are missing for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.6 Verify artifact file sizes
    # ----------------------------------------------------------------------------------------------

    model_nonempty = (
        model_path.stat().st_size > 0
    )

    checkpoint_nonempty = (
        checkpoint_path.stat().st_size > 0
    )

    synthetic_nonempty = (
        synthetic_path.stat().st_size > 0
    )

    metadata_nonempty = (
        metadata_path.stat().st_size > 0
    )


    if not all([
        model_nonempty,
        checkpoint_nonempty,
        synthetic_nonempty,
        metadata_nonempty,
    ]):

        raise RuntimeError(
            f"One or more artifacts are empty for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.7 SHA-256 verification
    # ----------------------------------------------------------------------------------------------

    model_sha256_actual = calculate_sha256(
        model_path
    )

    checkpoint_sha256_actual = calculate_sha256(
        checkpoint_path
    )

    synthetic_sha256_actual = calculate_sha256(
        synthetic_path
    )

    metadata_sha256_actual = calculate_sha256(
        metadata_path
    )


    model_hash_ok = (
        model_sha256_actual
        == str(manifest_row["model_sha256"])
    )

    checkpoint_hash_ok = (
        checkpoint_sha256_actual
        == str(manifest_row["checkpoint_sha256"])
    )

    synthetic_hash_ok = (
        synthetic_sha256_actual
        == str(manifest_row["synthetic_sha256"])
    )

    metadata_hash_ok = (
        metadata_sha256_actual
        == str(manifest_row["metadata_sha256"])
    )


    if not all([
        model_hash_ok,
        checkpoint_hash_ok,
        synthetic_hash_ok,
        metadata_hash_ok,
    ]):

        raise RuntimeError(
            f"SHA-256 verification failed for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.8 Cross-check hashes against Section 14/15/16 registries
    # ----------------------------------------------------------------------------------------------

    model_registry_hash_ok = (
        model_sha256_actual
        == str(model_row["sha256"])
    )

    checkpoint_registry_hash_ok = (
        checkpoint_sha256_actual
        == str(checkpoint_row["sha256"])
    )

    synthetic_registry_hash_ok = (
        synthetic_sha256_actual
        == str(synthetic_row["sha256"])
    )

    metadata_registry_hash_ok = (
        metadata_sha256_actual
        == str(metadata_row["sha256"])
    )


    if not all([
        model_registry_hash_ok,
        checkpoint_registry_hash_ok,
        synthetic_registry_hash_ok,
        metadata_registry_hash_ok,
    ]):

        raise RuntimeError(
            f"Cross-registry SHA-256 mismatch for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.9 Validate authoritative training schema
    # ----------------------------------------------------------------------------------------------

    train_df = TRAINING_DATA[
        dataset_id
    ]

    expected_columns = list(
        train_df.columns
    )

    target_column = CTGAN_TARGET_COLUMNS[
        dataset_id
    ]

    identifier_columns = CTGAN_IDENTIFIER_COLUMNS[
        dataset_id
    ]


    training_rows = int(
        len(train_df)
    )

    training_columns = int(
        train_df.shape[1]
    )


    if training_rows != int(
        manifest_row["training_rows"]
    ):

        raise RuntimeError(
            f"Training row mismatch for {dataset_id}."
        )


    if training_columns != int(
        manifest_row["training_columns"]
    ):

        raise RuntimeError(
            f"Training column mismatch for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.10 Reload synthetic data
    # ----------------------------------------------------------------------------------------------

    synthetic_df = pd.read_csv(
        synthetic_path
    )


    if synthetic_df.empty:

        raise RuntimeError(
            f"Reloaded synthetic data is empty for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.11 Synthetic schema validation
    # ----------------------------------------------------------------------------------------------

    schema_ok = (
        list(synthetic_df.columns)
        == expected_columns
    )


    if not schema_ok:

        raise RuntimeError(
            f"Synthetic schema mismatch for {dataset_id}: "
            f"expected={expected_columns}; "
            f"actual={list(synthetic_df.columns)}"
        )


    # ----------------------------------------------------------------------------------------------
    # 6.12 Synthetic row-count validation
    # ----------------------------------------------------------------------------------------------

    sample_size_ok = (
        len(synthetic_df)
        == training_rows
        == int(manifest_row["synthetic_rows"])
    )


    if not sample_size_ok:

        raise RuntimeError(
            f"Synthetic row-count mismatch for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.13 Synthetic duplicate-column validation
    # ----------------------------------------------------------------------------------------------

    duplicate_columns_ok = (
        not synthetic_df.columns.duplicated().any()
    )


    if not duplicate_columns_ok:

        raise RuntimeError(
            f"Duplicate synthetic columns detected for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.14 Target validation
    # ----------------------------------------------------------------------------------------------

    target_ok = (
        target_column
        in synthetic_df.columns
    )


    if not target_ok:

        raise RuntimeError(
            f"Target '{target_column}' missing for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.15 Provenance validation
    # ----------------------------------------------------------------------------------------------

    provenance_ok = (
        CTGAN_PROVENANCE_COLUMN
        not in synthetic_df.columns
    )


    if not provenance_ok:

        raise RuntimeError(
            f"Provenance column found in synthetic data "
            f"for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.16 Identifier validation
    # ----------------------------------------------------------------------------------------------

    leaked_identifiers = [
        column
        for column in identifier_columns
        if column in synthetic_df.columns
    ]

    identifier_ok = (
        len(leaked_identifiers) == 0
    )


    if not identifier_ok:

        raise RuntimeError(
            f"Identifier leakage for {dataset_id}: "
            f"{leaked_identifiers}"
        )


    # ----------------------------------------------------------------------------------------------
    # 6.17 Synthetic content hash validation
    # ----------------------------------------------------------------------------------------------

    synthetic_content_hasher = hashlib.sha256()

    synthetic_content_hasher.update(
        pd.util.hash_pandas_object(
            synthetic_df,
            index=True
        ).values.tobytes()
    )

    reloaded_content_hash = (
        synthetic_content_hasher.hexdigest()
    )


    if "source_content_hash" in synthetic_row.index:

        stored_content_hash = str(
            synthetic_row["source_content_hash"]
        )

        content_integrity_ok = (
            reloaded_content_hash
            == stored_content_hash
        )

    else:

        content_integrity_ok = True


    if not content_integrity_ok:

        raise RuntimeError(
            f"Synthetic content integrity mismatch "
            f"for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.18 Reload and validate metadata JSON
    # ----------------------------------------------------------------------------------------------

    with open(
        metadata_path,
        "r",
        encoding="utf-8"
    ) as f:

        metadata_payload = json.load(f)


    if not isinstance(
        metadata_payload,
        dict
    ):

        raise RuntimeError(
            f"Metadata payload is not a dictionary for {dataset_id}."
        )


    metadata_dataset_ok = (
        metadata_payload.get("dataset_id")
        == dataset_id
    )

    metadata_model_ok = (
        metadata_payload.get("model")
        == "CTGANSynthesizer"
    )

    metadata_fit_ok = (
        metadata_payload.get("fit_split")
        == "train"
    )

    metadata_validation_ok = (
        metadata_payload.get("validation_used")
        is False
    )

    metadata_test_ok = (
        metadata_payload.get("test_used")
        is False
    )

    metadata_target_ok = (
        metadata_payload.get("target_column")
        == target_column
    )

    metadata_schema_ok = (
        metadata_payload.get("generative_columns")
        == expected_columns
    )

    metadata_rows_ok = (
        int(metadata_payload.get("training_rows"))
        == training_rows
    )

    metadata_columns_ok = (
        int(metadata_payload.get("training_columns"))
        == training_columns
    )

    metadata_provenance_ok = (
        metadata_payload.get("provenance_excluded")
        is True
    )

    metadata_identifier_ok = (
        metadata_payload.get("identifiers_excluded")
        is True
    )


    metadata_ok = all([
        metadata_dataset_ok,
        metadata_model_ok,
        metadata_fit_ok,
        metadata_validation_ok,
        metadata_test_ok,
        metadata_target_ok,
        metadata_schema_ok,
        metadata_rows_ok,
        metadata_columns_ok,
        metadata_provenance_ok,
        metadata_identifier_ok,
    ])


    if not metadata_ok:

        raise RuntimeError(
            f"Metadata validation failed for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.19 Research-policy validation
    # ----------------------------------------------------------------------------------------------

    train_only_ok = (
        str(manifest_row["fit_split"])
        == "train"
    )

    validation_excluded_ok = require_false(
        manifest_row["validation_used"],
        "validation_used",
        dataset_id
    )

    test_excluded_ok = require_false(
        manifest_row["test_used"],
        "test_used",
        dataset_id
    )

    dp_off_ok = require_false(
        manifest_row["differential_privacy"],
        "differential_privacy",
        dataset_id
    )

    guidance_off_ok = require_false(
        manifest_row["statistical_guidance"],
        "statistical_guidance",
        dataset_id
    )

    sppgan_off_ok = require_false(
        manifest_row["sppgan_components"],
        "sppgan_components",
        dataset_id
    )

    target_retained_ok = require_true(
        manifest_row["target_retained"],
        "target_retained",
        dataset_id
    )

    target_predictor_off_ok = require_false(
        manifest_row["target_used_as_predictor"],
        "target_used_as_predictor",
        dataset_id
    )

    provenance_excluded_ok = require_true(
        manifest_row["provenance_excluded"],
        "provenance_excluded",
        dataset_id
    )

    identifiers_excluded_ok = require_true(
        manifest_row["identifiers_excluded"],
        "identifiers_excluded",
        dataset_id
    )


    # ----------------------------------------------------------------------------------------------
    # 6.20 Seed consistency validation
    # ----------------------------------------------------------------------------------------------

    manifest_seed = int(
        manifest_row["seed"]
    )

    model_seed = int(
        model_row["seed"]
    )

    checkpoint_seed = int(
        checkpoint_row["seed"]
    )

    metadata_seed = int(
        metadata_row["seed"]
    )


    seed_ok = (
        manifest_seed
        == int(DATASET_SEEDS[dataset_id])
        == model_seed
        == checkpoint_seed
        == metadata_seed
    )


    if not seed_ok:

        raise RuntimeError(
            f"Seed inconsistency detected for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.21 Runtime consistency validation
    # ----------------------------------------------------------------------------------------------

    training_runtime = float(
        training_row["training_runtime_seconds"]
    )

    generation_runtime = float(
        generation_row["generation_runtime_seconds"]
    )

    manifest_training_runtime = float(
        manifest_row["training_runtime_seconds"]
    )

    manifest_generation_runtime = float(
        manifest_row["generation_runtime_seconds"]
    )


    runtime_ok = (
        np.isfinite(training_runtime)
        and np.isfinite(generation_runtime)
        and training_runtime >= 0
        and generation_runtime >= 0
        and np.isclose(
            training_runtime,
            manifest_training_runtime,
            rtol=0,
            atol=1e-9,
        )
        and np.isclose(
            generation_runtime,
            manifest_generation_runtime,
            rtol=0,
            atol=1e-9,
        )
    )


    if not runtime_ok:

        raise RuntimeError(
            f"Runtime inconsistency detected for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.22 Model/checkpoint/synthetic/metadata path consistency
    # ----------------------------------------------------------------------------------------------

    path_consistency_ok = all([
        str(model_row["model_path"]) == str(model_path),
        str(checkpoint_row["checkpoint_path"]) == str(checkpoint_path),
        str(synthetic_row["synthetic_path"]) == str(synthetic_path),
        str(metadata_row["metadata_path"]) == str(metadata_path),
    ])


    if not path_consistency_ok:

        raise RuntimeError(
            f"Artifact path inconsistency detected for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.23 Final dataset-level checks
    # ----------------------------------------------------------------------------------------------

    all_checks = [

        model_exists,
        checkpoint_exists,
        synthetic_exists,
        metadata_exists,

        model_nonempty,
        checkpoint_nonempty,
        synthetic_nonempty,
        metadata_nonempty,

        model_hash_ok,
        checkpoint_hash_ok,
        synthetic_hash_ok,
        metadata_hash_ok,

        model_registry_hash_ok,
        checkpoint_registry_hash_ok,
        synthetic_registry_hash_ok,
        metadata_registry_hash_ok,

        schema_ok,
        sample_size_ok,
        duplicate_columns_ok,
        target_ok,
        provenance_ok,
        identifier_ok,
        content_integrity_ok,

        metadata_ok,

        train_only_ok,
        validation_excluded_ok,
        test_excluded_ok,

        dp_off_ok,
        guidance_off_ok,
        sppgan_off_ok,

        target_retained_ok,
        target_predictor_off_ok,

        provenance_excluded_ok,
        identifiers_excluded_ok,

        seed_ok,
        runtime_ok,
        path_consistency_ok,
    ]


    overall_status = (
        "PASS"
        if all(all_checks)
        else "FAIL"
    )


    if overall_status != "PASS":

        raise RuntimeError(
            f"Artifact verification failed for {dataset_id}."
        )


    # ----------------------------------------------------------------------------------------------
    # 6.24 Record verification result
    # ----------------------------------------------------------------------------------------------

    CTGAN_ARTIFACT_VERIFICATION_RECORDS.append({

        "dataset_id":
            dataset_id,

        "model_exists":
            model_exists,

        "checkpoint_exists":
            checkpoint_exists,

        "synthetic_exists":
            synthetic_exists,

        "metadata_exists":
            metadata_exists,

        "model_nonempty":
            model_nonempty,

        "checkpoint_nonempty":
            checkpoint_nonempty,

        "synthetic_nonempty":
            synthetic_nonempty,

        "metadata_nonempty":
            metadata_nonempty,

        "model_sha256_verified":
            model_hash_ok,

        "checkpoint_sha256_verified":
            checkpoint_hash_ok,

        "synthetic_sha256_verified":
            synthetic_hash_ok,

        "metadata_sha256_verified":
            metadata_hash_ok,

        "model_registry_hash_verified":
            model_registry_hash_ok,

        "checkpoint_registry_hash_verified":
            checkpoint_registry_hash_ok,

        "synthetic_registry_hash_verified":
            synthetic_registry_hash_ok,

        "metadata_registry_hash_verified":
            metadata_registry_hash_ok,

        "schema_verified":
            schema_ok,

        "sample_size_verified":
            sample_size_ok,

        "duplicate_columns_verified":
            duplicate_columns_ok,

        "target_verified":
            target_ok,

        "provenance_excluded":
            provenance_ok,

        "identifiers_excluded":
            identifier_ok,

        "content_integrity_verified":
            content_integrity_ok,

        "metadata_verified":
            metadata_ok,

        "train_only":
            train_only_ok,

        "validation_excluded":
            validation_excluded_ok,

        "test_excluded":
            test_excluded_ok,

        "target_retained":
            target_retained_ok,

        "target_not_used_as_predictor":
            target_predictor_off_ok,

        "differential_privacy_disabled":
            dp_off_ok,

        "statistical_guidance_disabled":
            guidance_off_ok,

        "sppgan_components_disabled":
            sppgan_off_ok,

        "seed_consistent":
            seed_ok,

        "runtime_consistent":
            runtime_ok,

        "artifact_paths_consistent":
            path_consistency_ok,

        "status":
            overall_status,
    })


    # ----------------------------------------------------------------------------------------------
    # 6.25 Dataset-level reporting
    # ----------------------------------------------------------------------------------------------

    print(
        f"✓ Model              : PASS"
    )

    print(
        f"✓ Checkpoint         : PASS"
    )

    print(
        f"✓ Synthetic          : PASS"
    )

    print(
        f"✓ Metadata           : PASS"
    )

    print(
        f"✓ SHA-256            : PASS"
    )

    print(
        f"✓ Registry hashes    : PASS"
    )

    print(
        f"✓ Schema             : PASS"
    )

    print(
        f"✓ Sample size        : PASS"
    )

    print(
        f"✓ Target             : PASS"
    )

    print(
        f"✓ Provenance         : EXCLUDED | PASS"
    )

    print(
        f"✓ Identifiers        : EXCLUDED | PASS"
    )

    print(
        f"✓ Content integrity  : PASS"
    )

    print(
        f"✓ Metadata validation: PASS"
    )

    print(
        f"✓ Train-only policy  : PASS"
    )

    print(
        f"✓ DP disabled        : PASS"
    )

    print(
        f"✓ Statistical guide  : OFF | PASS"
    )

    print(
        f"✓ SPP-GAN components : OFF | PASS"
    )

    print(
        f"✓ Seed consistency   : PASS"
    )

    print(
        f"✓ Runtime consistency: PASS"
    )

    print(
        f"✓ Path consistency   : PASS"
    )

    print(
        f"✓ Status             : {overall_status}"
    )


    del synthetic_df


# --------------------------------------------------------------------------------------------------
# 7. CREATE VERIFICATION DATAFRAME
# --------------------------------------------------------------------------------------------------

CTGAN_ARTIFACT_VERIFICATION_DF = pd.DataFrame(
    CTGAN_ARTIFACT_VERIFICATION_RECORDS
)


# --------------------------------------------------------------------------------------------------
# 8. VERIFY VERIFICATION RECORD COUNT
# --------------------------------------------------------------------------------------------------

if len(
    CTGAN_ARTIFACT_VERIFICATION_DF
) != len(DATASET_IDS):

    raise RuntimeError(
        "Artifact verification registry does not contain exactly "
        "one record per dataset."
    )


if set(
    CTGAN_ARTIFACT_VERIFICATION_DF["dataset_id"]
) != set(DATASET_IDS):

    raise RuntimeError(
        "Artifact verification registry contains an unexpected "
        "dataset set."
    )


if CTGAN_ARTIFACT_VERIFICATION_DF[
    "dataset_id"
].duplicated().any():

    raise RuntimeError(
        "Artifact verification registry contains duplicate datasets."
    )


if not (
    CTGAN_ARTIFACT_VERIFICATION_DF["status"] == "PASS"
).all():

    raise RuntimeError(
        "One or more CTGAN artifact verification records failed."
    )


# --------------------------------------------------------------------------------------------------
# 9. SAVE CSV VERIFICATION REPORT
# --------------------------------------------------------------------------------------------------

CTGAN_VERIFICATION_PATH = (
    NB06_VALIDATION_ROOT
    / "ctgan_artifact_validation.csv"
)

NB06_VALIDATION_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

CTGAN_ARTIFACT_VERIFICATION_DF.to_csv(
    CTGAN_VERIFICATION_PATH,
    index=False
)


if not CTGAN_VERIFICATION_PATH.exists():

    raise RuntimeError(
        "CTGAN artifact validation CSV was not created."
    )


if CTGAN_VERIFICATION_PATH.stat().st_size <= 0:

    raise RuntimeError(
        "CTGAN artifact validation CSV is empty."
    )


# --------------------------------------------------------------------------------------------------
# 10. RELOAD CSV VERIFICATION REPORT
# --------------------------------------------------------------------------------------------------

CTGAN_ARTIFACT_VERIFICATION_RELOAD_DF = pd.read_csv(
    CTGAN_VERIFICATION_PATH
)


if len(
    CTGAN_ARTIFACT_VERIFICATION_RELOAD_DF
) != len(DATASET_IDS):

    raise RuntimeError(
        "Reloaded artifact verification CSV has an invalid record count."
    )


if set(
    CTGAN_ARTIFACT_VERIFICATION_RELOAD_DF["dataset_id"]
) != set(DATASET_IDS):

    raise RuntimeError(
        "Reloaded artifact verification CSV contains an unexpected "
        "dataset set."
    )


if not (
    CTGAN_ARTIFACT_VERIFICATION_RELOAD_DF["status"] == "PASS"
).all():

    raise RuntimeError(
        "Reloaded artifact verification CSV contains non-PASS records."
    )


# --------------------------------------------------------------------------------------------------
# 11. CREATE JSON AUDIT REPORT
# --------------------------------------------------------------------------------------------------

CTGAN_VERIFICATION_JSON = (
    NB06_VALIDATION_ROOT
    / "ctgan_artifact_validation.json"
)

verification_payload = {

    "notebook_id":
        NOTEBOOK_ID,

    "notebook_name":
        NOTEBOOK_NAME,

    "notebook_version":
        NOTEBOOK_VERSION,

    "expected_runs":
        len(DATASET_IDS),

    "verified_runs":
        int(
            (
                CTGAN_ARTIFACT_VERIFICATION_DF["status"]
                == "PASS"
            ).sum()
        ),

    "status":
        "PASS",

    "datasets":
        list(DATASET_IDS),

    "records":
        CTGAN_ARTIFACT_VERIFICATION_DF.to_dict(
            orient="records"
        ),
}


with open(
    CTGAN_VERIFICATION_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        verification_payload,
        f,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )


# --------------------------------------------------------------------------------------------------
# 12. VERIFY JSON REPORT
# --------------------------------------------------------------------------------------------------

if not CTGAN_VERIFICATION_JSON.exists():

    raise RuntimeError(
        "CTGAN artifact validation JSON was not created."
    )


if CTGAN_VERIFICATION_JSON.stat().st_size <= 0:

    raise RuntimeError(
        "CTGAN artifact validation JSON is empty."
    )


with open(
    CTGAN_VERIFICATION_JSON,
    "r",
    encoding="utf-8"
) as f:

    verification_payload_reload = json.load(f)


if verification_payload_reload.get(
    "notebook_id"
) != NOTEBOOK_ID:

    raise RuntimeError(
        "Verification JSON notebook_id mismatch."
    )


if verification_payload_reload.get(
    "expected_runs"
) != len(DATASET_IDS):

    raise RuntimeError(
        "Verification JSON expected_runs mismatch."
    )


if verification_payload_reload.get(
    "verified_runs"
) != len(DATASET_IDS):

    raise RuntimeError(
        "Verification JSON verified_runs mismatch."
    )


if verification_payload_reload.get(
    "status"
) != "PASS":

    raise RuntimeError(
        "Verification JSON status is not PASS."
    )


if verification_payload_reload.get(
    "datasets"
) != list(DATASET_IDS):

    raise RuntimeError(
        "Verification JSON dataset registry mismatch."
    )


if len(
    verification_payload_reload.get(
        "records",
        []
    )
) != len(DATASET_IDS):

    raise RuntimeError(
        "Verification JSON record count mismatch."
    )


# --------------------------------------------------------------------------------------------------
# 13. FINAL VERIFICATION SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("CTGAN ARTIFACT VERIFICATION SUMMARY")
print("=" * 100)

print(
    CTGAN_ARTIFACT_VERIFICATION_DF[
        [
            "dataset_id",
            "model_sha256_verified",
            "checkpoint_sha256_verified",
            "synthetic_sha256_verified",
            "metadata_sha256_verified",
            "schema_verified",
            "sample_size_verified",
            "target_verified",
            "provenance_excluded",
            "identifiers_excluded",
            "content_integrity_verified",
            "metadata_verified",
            "seed_consistent",
            "runtime_consistent",
            "artifact_paths_consistent",
            "status",
        ]
    ].to_string(index=False)
)


# --------------------------------------------------------------------------------------------------
# 14. FINAL ARTIFACT REPORT FINGERPRINTS
# --------------------------------------------------------------------------------------------------

CTGAN_VERIFICATION_CSV_SHA256 = calculate_sha256(
    CTGAN_VERIFICATION_PATH
)

CTGAN_VERIFICATION_JSON_SHA256 = calculate_sha256(
    CTGAN_VERIFICATION_JSON
)


print(
    f"\n✓ CSV report SHA256  : "
    f"{CTGAN_VERIFICATION_CSV_SHA256}"
)

print(
    f"✓ JSON report SHA256 : "
    f"{CTGAN_VERIFICATION_JSON_SHA256}"
)

print(
    f"✓ CSV report         : "
    f"{CTGAN_VERIFICATION_PATH}"
)

print(
    f"✓ JSON report        : "
    f"{CTGAN_VERIFICATION_JSON}"
)


# --------------------------------------------------------------------------------------------------
# 15. FINAL SECTION STATUS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)

print(
    "✓ All CTGAN model artifacts verified."
)

print(
    "✓ All CTGAN checkpoints verified."
)

print(
    "✓ All CTGAN synthetic artifacts verified."
)

print(
    "✓ All CTGAN metadata artifacts verified."
)

print(
    "✓ SHA-256 fingerprints verified against Section 16 manifest."
)

print(
    "✓ SHA-256 fingerprints cross-validated against upstream registries."
)

print(
    "✓ Synthetic schemas verified."
)

print(
    "✓ Synthetic sample sizes verified."
)

print(
    "✓ Target retention verified."
)

print(
    "✓ Provenance exclusion verified."
)

print(
    "✓ Identifier exclusion verified."
)

print(
    "✓ Synthetic content integrity verified."
)

print(
    "✓ Metadata payloads reloaded and validated."
)

print(
    "✓ Train-only policy verified."
)

print(
    "✓ Differential privacy disabled as required for CTGAN baseline."
)

print(
    "✓ Statistical guidance disabled as required for CTGAN baseline."
)

print(
    "✓ SPP-GAN components disabled as required for CTGAN baseline."
)

print(
    "✓ Dataset-specific seed consistency verified."
)

print(
    "✓ Runtime consistency verified."
)

print(
    "✓ Artifact path consistency verified."
)

print(
    "✓ CSV verification report persisted and reloaded."
)

print(
    "✓ JSON verification report persisted and reloaded."
)

print(
    "✓ SECTION 17 — ARTIFACT VERIFICATION : PASS"
)

print("=" * 100)

SECTION 17 — VERIFY ARTIFACTS

----------------------------------------------------------------------------------------------------
Verifying : adult_income
✓ Model              : PASS
✓ Checkpoint         : PASS
✓ Synthetic          : PASS
✓ Metadata           : PASS
✓ SHA-256            : PASS
✓ Registry hashes    : PASS
✓ Schema             : PASS
✓ Sample size        : PASS
✓ Target             : PASS
✓ Provenance         : EXCLUDED | PASS
✓ Identifiers        : EXCLUDED | PASS
✓ Content integrity  : PASS
✓ Metadata validation: PASS
✓ Train-only policy  : PASS
✓ DP disabled        : PASS
✓ Statistical guide  : OFF | PASS
✓ SPP-GAN components : OFF | PASS
✓ Seed consistency   : PASS
✓ Runtime consistency: PASS
✓ Path consistency   : PASS
✓ Status             : PASS

----------------------------------------------------------------------------------------------------
Verifying : bank_marketing
✓ Model              : PASS
✓ Checkpoint         : PASS
✓ Synthetic          : PASS
✓ Metada

In [56]:
# ==================================================================================================
# SECTION 18 — COMPLETION SUMMARY
# ==================================================================================================

print("=" * 100)
print("SECTION 18 — COMPLETION SUMMARY")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Final integrity gate
# --------------------------------------------------------------------------------------------------

expected_runs = len(DATASET_IDS)

model_count = len(CTGAN_MODEL_DF)
checkpoint_count = len(CTGAN_CHECKPOINT_DF)
synthetic_count = len(CTGAN_SYNTHETIC_ARTIFACT_DF)
metadata_count = len(CTGAN_METADATA_DF)
verified_count = int(
    (
        CTGAN_ARTIFACT_VERIFICATION_DF["status"]
        == "PASS"
    ).sum()
)

all_final_checks = [
    model_count == expected_runs,
    checkpoint_count == expected_runs,
    synthetic_count == expected_runs,
    metadata_count == expected_runs,
    verified_count == expected_runs,
    (
        CTGAN_TRAINING_DF["status"]
        == "PASS"
    ).all(),
    (
        CTGAN_SYNTHETIC_VALIDATION_DF["status"]
        == "PASS"
    ).all(),
    (
        CTGAN_ARTIFACT_VERIFICATION_DF["status"]
        == "PASS"
    ).all(),
]

if not all(all_final_checks):
    raise RuntimeError(
        "Notebook 06 final completion gate failed."
    )

# --------------------------------------------------------------------------------------------------
# Completion report
# --------------------------------------------------------------------------------------------------

completion_report = {
    "notebook_id": NOTEBOOK_ID,
    "notebook_name": NOTEBOOK_NAME,
    "notebook_version": NOTEBOOK_VERSION,

    "project_root": str(PROJECT_ROOT),
    "notebook_root": str(NB06_ROOT),

    "datasets": {
        dataset_id: {
            "training_rows": int(
                TRAINING_SHAPES[dataset_id][0]
            ),
            "training_columns": int(
                TRAINING_SHAPES[dataset_id][1]
            ),
            "target_column": TARGET_COLUMNS[dataset_id],
        }
        for dataset_id in DATASET_IDS
    },

    "model": CTGAN_CONFIG,

    "experimental_integrity": {
        "train_split_only": True,
        "validation_split_excluded": True,
        "test_split_excluded": True,
        "notebook_02_preprocessing_refitted": False,
        "native_generative_schema_used": True,
        "target_retained": True,
        "target_used_as_predictor": False,
        "provenance_excluded": True,
        "identifiers_excluded": True,
        "synthetic_size_equals_training_size": True,
        "differential_privacy_applied": False,
        "statistical_guidance_applied": False,
        "sppgan_components_used": False,
        "reproducible_seed_policy": True,
    },

    "artifacts": {
        "models": model_count,
        "checkpoints": checkpoint_count,
        "synthetic_datasets": synthetic_count,
        "metadata_records": metadata_count,
        "verified_runs": verified_count,
    },

    "output_locations": {
        "models": str(NB06_MODEL_ROOT),
        "checkpoints": str(NB06_CHECKPOINT_ROOT),
        "synthetic": str(NB06_SYNTHETIC_ROOT),
        "metadata": str(NB06_METADATA_ROOT),
        "history": str(NB06_HISTORY_ROOT),
        "manifest": str(NB06_MANIFEST_ROOT),
        "validation": str(NB06_VALIDATION_ROOT),
        "config": str(NB06_CONFIG_ROOT),
    },

    "final_status": "PASS",

    "completed_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

CTGAN_COMPLETION_REPORT = (
    NB06_VALIDATION_ROOT
    / "ctgan_completion_report.json"
)

with open(
    CTGAN_COMPLETION_REPORT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        completion_report,
        f,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )

# --------------------------------------------------------------------------------------------------
# Reload completion report to verify persistence
# --------------------------------------------------------------------------------------------------

with open(
    CTGAN_COMPLETION_REPORT,
    "r",
    encoding="utf-8"
) as f:
    reloaded_completion_report = json.load(f)

if reloaded_completion_report["final_status"] != "PASS":
    raise RuntimeError(
        "Completion report reload validation failed."
    )

# --------------------------------------------------------------------------------------------------
# Final display
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 100)
print("NOTEBOOK 06 — CTGAN BASELINE")
print("=" * 100)

print(f"Notebook ID           : {NOTEBOOK_ID}")
print(f"Notebook version      : {NOTEBOOK_VERSION}")
print(f"Project root          : {PROJECT_ROOT}")
print(f"Notebook 06 output    : {NB06_ROOT}")

print("\nDATASETS")
print("-" * 100)

for dataset_id in DATASET_IDS:

    rows = TRAINING_SHAPES[dataset_id][0]

    print(
        f"✓ {dataset_id:<20} | "
        f"training rows={rows:,}"
    )

print("\nMODEL")
print("-" * 100)

print("✓ CTGANSynthesizer")
print(f"✓ Epochs                : {CTGAN_CONFIG['epochs']}")
print(f"✓ Embedding dimension   : {CTGAN_CONFIG['embedding_dim']}")
print(f"✓ Generator dimensions  : {CTGAN_CONFIG['generator_dim']}")
print(f"✓ Discriminator dims.   : {CTGAN_CONFIG['discriminator_dim']}")
print(f"✓ Batch size            : {CTGAN_CONFIG['batch_size']}")
print(f"✓ PAC                   : {CTGAN_CONFIG['pac']}")
print(f"✓ Requested GPU         : {CTGAN_CONFIG['requested_gpu']}")

print("\nEXPERIMENTAL INTEGRITY")
print("-" * 100)

integrity_checks = [
    "TRAIN split only used for fitting",
    "VALIDATION split excluded from fitting",
    "TEST split excluded from fitting",
    "Notebook 02 preprocessing not refitted",
    "Native generative schema used",
    "Target retained in synthetic data",
    "Target never used as predictor",
    "Provenance excluded",
    "Identifiers excluded",
    "Synthetic size equals training size",
    "Differential privacy not applied",
    "Statistical guidance not applied",
    "SPP-GAN components not used",
    "Reproducible seed policy applied",
]

for item in integrity_checks:
    print(f"✓ {item}")

print("\nARTIFACTS")
print("-" * 100)

print(f"✓ CTGAN models        : {model_count}")
print(f"✓ Checkpoints         : {checkpoint_count}")
print(f"✓ Synthetic datasets  : {synthetic_count}")
print(f"✓ Metadata records    : {metadata_count}")
print(f"✓ Verified runs       : {verified_count}")

print("\nOUTPUT LOCATIONS")
print("-" * 100)

print(f"Models       : {NB06_MODEL_ROOT}")
print(f"Checkpoints  : {NB06_CHECKPOINT_ROOT}")
print(f"Synthetic    : {NB06_SYNTHETIC_ROOT}")
print(f"Metadata     : {NB06_METADATA_ROOT}")
print(f"History      : {NB06_HISTORY_ROOT}")
print(f"Manifests    : {NB06_MANIFEST_ROOT}")
print(f"Validation   : {NB06_VALIDATION_ROOT}")
print(f"Config       : {NB06_CONFIG_ROOT}")

print("\n" + "-" * 100)

print(f"✓ Expected CTGAN runs   : {expected_runs}")
print(f"✓ Models                : {model_count}")
print(f"✓ Checkpoints           : {checkpoint_count}")
print(f"✓ Synthetic datasets    : {synthetic_count}")
print(f"✓ Metadata records      : {metadata_count}")
print(
    f"✓ Artifact verification : "
    f"{verified_count}/{expected_runs} PASS"
)
print(
    f"✓ Completion report     : "
    f"{CTGAN_COMPLETION_REPORT}"
)

print("\n" + "=" * 100)
print("✓ NOTEBOOK 06 — CTGAN BASELINE : PASS")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Final RAM cleanup
# --------------------------------------------------------------------------------------------------

try:
    del CTGAN_MODELS
except Exception:
    pass

try:
    del CTGAN_METADATA
except Exception:
    pass

try:
    del CTGAN_SYNTHETIC_DATA
except Exception:
    pass

try:
    del TRAINING_DATA
except Exception:
    pass

gc.collect()

print("\n✓ RAM cleanup completed.")

SECTION 18 — COMPLETION SUMMARY


NOTEBOOK 06 — CTGAN BASELINE
Notebook ID           : 06
Notebook version      : 1.0
Project root          : /content/drive/MyDrive/SPP_GAN_Research
Notebook 06 output    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_06

DATASETS
----------------------------------------------------------------------------------------------------
✓ adult_income         | training rows=34,189
✓ bank_marketing       | training rows=31,647
✓ diabetes_130us       | training rows=71,236

MODEL
----------------------------------------------------------------------------------------------------
✓ CTGANSynthesizer
✓ Epochs                : 300
✓ Embedding dimension   : 128
✓ Generator dimensions  : (256, 256)
✓ Discriminator dims.   : (256, 256)
✓ Batch size            : 500
✓ PAC                   : 10
✓ Requested GPU         : True

EXPERIMENTAL INTEGRITY
----------------------------------------------------------------------------------------------------